In [1]:
# Importing libraries for text analysis, visualization, embeddings, clustering and BigQuery access

from pathlib import Path
import sys

import pandas as pd
import numpy as np

import html
import re
import unicodedata
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF

from google.cloud import bigquery

import pickle

pd.set_option('display.max_rows', None)         # Show all rows
pd.set_option('display.max_columns', None)      # Show all columns

In [2]:
# Setting project paths and loading reusable helper functions

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
scripts_directory = project_root / "scripts"

if str(scripts_directory) not in sys.path:
    sys.path.append(str(scripts_directory))

from run_sql_scripts import load_project_configuration
from run_sql_scripts import configure_google_credentials

In [3]:
# Loading project configuration and creating BigQuery connections for Electronics NLP analysis

configuration = load_project_configuration()
configure_google_credentials(credentials_path_value = configuration["credentials_path"])

bigquery_client = bigquery.Client(project = configuration["gcp_project_id"])


def run_query(sql_text):
    return bigquery_client.query(sql_text).to_dataframe()

In [4]:
# Profiling Electronics review records for NLP readiness

modeling_ready_profile_query = f"""
SELECT
    analysis_pool_role,
    sentiment_cohort,
    verified_purchase,
    price_band,
    issue_priority_level,
    COUNT(*) AS document_count,
    SUM(sampled_review_count) AS total_sampled_reviews,
    AVG(sampled_review_count) AS average_sampled_reviews_per_document,
    MIN(sampled_review_count) AS min_sampled_reviews_per_document,
    MAX(sampled_review_count) AS max_sampled_reviews_per_document,
    AVG(aggregated_text_length) AS average_aggregated_text_length,
    MIN(aggregated_text_length) AS min_aggregated_text_length,
    MAX(aggregated_text_length) AS max_aggregated_text_length
FROM `{configuration["gcp_project_id"]}.{configuration["bigquery_core_dataset_id"]}.electronics_text_modeling_ready`
GROUP BY
    analysis_pool_role,
    sentiment_cohort,
    verified_purchase,
    price_band,
    issue_priority_level
ORDER BY
    analysis_pool_role,
    sentiment_cohort,
    verified_purchase,
    price_band,
    issue_priority_level
"""

modeling_ready_profile_dataframe = run_query(modeling_ready_profile_query)
display(modeling_ready_profile_dataframe)

,analysis_pool_role,sentiment_cohort,verified_purchase,price_band,issue_priority_level,document_count,total_sampled_reviews,average_sampled_reviews_per_document,min_sampled_reviews_per_document,max_sampled_reviews_per_document,average_aggregated_text_length,min_aggregated_text_length,max_aggregated_text_length
0,Comparison baseline,Satisfaction,True,Budget,Monitor,3703,315963,85.326222,3,250,14844.635701,300,77129
1,Comparison baseline,Satisfaction,True,Budget,Priority review,1070,163689,152.980374,2,250,28525.889720,122,114791
2,Comparison baseline,Satisfaction,True,Lower mid,Monitor,3721,342517,92.049718,1,250,18707.086805,48,125902
3,Comparison baseline,Satisfaction,True,Lower mid,Priority review,949,138171,145.596417,1,250,32456.590095,83,125585
4,Comparison baseline,Satisfaction,True,Premium,Monitor,2903,272286,93.794695,1,250,27513.939029,55,189470
5,Comparison baseline,Satisfaction,True,Premium,Priority review,1185,169992,143.453165,3,250,44856.638819,428,167141
6,Comparison baseline,Satisfaction,True,Upper mid,Monitor,3668,356748,97.259542,1,250,23517.886314,310,124291
7,Comparison baseline,Satisfaction,True,Upper mid,Priority review,1282,190817,148.843214,1,250,39131.535881,88,139066
8,Comparison baseline,Satisfaction,True,Very premium,Monitor,712,50585,71.046348,1,250,25821.790730,184,132848
9,Topic modeling target,Dissatisfaction,True,Budget,Monitor,3168,315949,99.731376,1,250,21892.470013,33,102683


In [5]:
# Creating the Electronics NLP document dataset for downstream text and issue-theme analysis

electronics_nlp_documents_query = f"""
SELECT
    document_id,
    analysis_pool_role,
    sentiment_cohort,
    verified_purchase,
    price_band,
    issue_priority_level,
    product_evidence_segment,
    parent_asin,
    product_title_clean,
    product_display_name,
    store_clean,
    store_display_name,
    product_identification_quality,
    review_count,
    target_review_count_per_sentiment,
    target_review_count_for_product,
    per_document_review_cap,
    sampled_review_count,
    sampled_review_ratio,
    average_rating,
    average_helpful_vote,
    average_review_text_length,
    aggregated_text,
    aggregated_text_length,
    is_price_band_interpretation_eligible,
    is_price_sensitive_theme_comparison_eligible,
    is_issue_priority_comparison_eligible,
    is_product_interpretation_eligible
FROM `{configuration["gcp_project_id"]}.{configuration["bigquery_core_dataset_id"]}.electronics_text_modeling_ready`
ORDER BY
    analysis_pool_role,
    sentiment_cohort,
    verified_purchase,
    price_band,
    issue_priority_level,
    document_id
"""

electronics_nlp_documents_dataframe = run_query(electronics_nlp_documents_query)
print(f"Loaded rows : {len(electronics_nlp_documents_dataframe):,}")
electronics_nlp_documents_dataframe.head()

Loaded rows : 36,487


,document_id,analysis_pool_role,sentiment_cohort,verified_purchase,price_band,issue_priority_level,product_evidence_segment,parent_asin,product_title_clean,product_display_name,store_clean,store_display_name,product_identification_quality,review_count,target_review_count_per_sentiment,target_review_count_for_product,per_document_review_cap,sampled_review_count,sampled_review_ratio,average_rating,average_helpful_vote,average_review_text_length,aggregated_text,aggregated_text_length,is_price_band_interpretation_eligible,is_price_sensitive_theme_comparison_eligible,is_issue_priority_comparison_eligible,is_product_interpretation_eligible
0,Satisfaction|true|Budget|Monitor|106171327X|ch...,Comparison baseline,Satisfaction,True,Budget,Monitor,Evidence-backed product,106171327X,SanDisk SDSDQUA-064G-A11 Professional Ultra 64...,SanDisk SDSDQUA-064G-A11 Professional Ultra 64...,SanDisk,SanDisk,High,164,315909,27,250,27,0.164634,4.932927,0.548780,96.420732,great price my last card died after a year so ...,2870,True,True,True,True
1,Satisfaction|true|Budget|Monitor|B00000J1QR|ch...,Comparison baseline,Satisfaction,True,Budget,Monitor,Evidence-backed product,B00000J1QR,"Allsop CD and DVD FastWipes, lint-free wipes f...","Allsop CD and DVD FastWipes, lint-free wipes f...",Allsop,Allsop,High,169,315909,28,250,28,0.165680,4.715976,2.349112,166.295858,great for keeping dvds or cdcs clean br br yes...,4261,True,True,True,True
2,Satisfaction|true|Budget|Monitor|B00004Z5M1|ch...,Comparison baseline,Satisfaction,True,Budget,Monitor,Evidence-backed product,B00004Z5M1,Belkin - F3U133b10 (F3U133b10) Hi-Speed USB A/...,Belkin - F3U133b10 (F3U133b10) Hi-Speed USB A/...,Belkin,Belkin,High,850,315909,137,250,137,0.161176,4.889412,0.435294,163.344706,i really appreciate the good value of this pri...,25729,True,True,True,True
3,Satisfaction|true|Budget|Monitor|B00004ZCJF|ch...,Comparison baseline,Satisfaction,True,Budget,Monitor,Evidence-backed product,B00004ZCJF,Tiffen 49UVP 49mm UV Protection Camera Lens Fi...,Tiffen 49UVP 49mm UV Protection Camera Lens Fi...,Tiffen,Tiffen,High,194,315909,32,250,32,0.164948,4.829897,0.283505,164.737113,for the price i always buy tiffen lens filters...,5032,True,True,True,True
4,Satisfaction|true|Budget|Monitor|B00004ZCJJ|ch...,Comparison baseline,Satisfaction,True,Budget,Monitor,Evidence-backed product,B00004ZCJJ,Tiffen 62UVP 62mm UV Protection Filter,Tiffen 62UVP 62mm UV Protection Filter,Tiffen,Tiffen,High,225,315909,37,250,37,0.164444,4.826667,0.102222,151.924444,no tiene problema con la fotograf a review_bou...,6772,True,True,True,True


In [6]:
electronics_nlp_documents_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36487 entries, 0 to 36486
Data columns (total 28 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   document_id                                   36487 non-null  object 
 1   analysis_pool_role                            36487 non-null  object 
 2   sentiment_cohort                              36487 non-null  object 
 3   verified_purchase                             36487 non-null  boolean
 4   price_band                                    36487 non-null  object 
 5   issue_priority_level                          36487 non-null  object 
 6   product_evidence_segment                      36487 non-null  object 
 7   parent_asin                                   36487 non-null  object 
 8   product_title_clean                           36487 non-null  object 
 9   product_display_name                          36487 non-null 

In [7]:
# Summarizing document quality, text readiness and review aggregation statistics
document_quality_summary_dataframe = pd.DataFrame({
    "metric": [
        "row_count",
        "unique_document_id_count",
        "duplicate_document_id_count",
        "null_aggregated_text_count",
        "blank_aggregated_text_count",
        "null_price_band_count",
        "null_issue_priority_level_count",
        "null_product_display_name_count",
        "null_store_display_name_count"
    ],
    "value": [
        len(electronics_nlp_documents_dataframe),
        electronics_nlp_documents_dataframe["document_id"].nunique(),
        len(electronics_nlp_documents_dataframe) - electronics_nlp_documents_dataframe["document_id"].nunique(),
        electronics_nlp_documents_dataframe["aggregated_text"].isna().sum(),
        electronics_nlp_documents_dataframe["aggregated_text"].fillna("").str.strip().eq("").sum(),
        electronics_nlp_documents_dataframe["price_band"].isna().sum(),
        electronics_nlp_documents_dataframe["issue_priority_level"].isna().sum(),
        electronics_nlp_documents_dataframe["product_display_name"].isna().sum(),
        electronics_nlp_documents_dataframe["store_display_name"].isna().sum()
    ]
})

display(document_quality_summary_dataframe)

,metric,value
0,row_count,36487
1,unique_document_id_count,36487
2,duplicate_document_id_count,0
3,null_aggregated_text_count,0
4,blank_aggregated_text_count,0
5,null_price_band_count,0
6,null_issue_priority_level_count,0
7,null_product_display_name_count,0
8,null_store_display_name_count,0


In [8]:
# Reviewing text-length distribution across Electronics NLP documents
text_length_distribution_dataframe = electronics_nlp_documents_dataframe[[
    "aggregated_text_length",
    "sampled_review_count",
    "average_rating",
    "average_helpful_vote",
    "average_review_text_length"
]].describe(percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T

display(text_length_distribution_dataframe)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
aggregated_text_length,36487.0,28973.935758,25206.134567,33.0,2661.44,4711.0,10264.0,19887.0,41545.5,79402.0,113102.12,203015.0
sampled_review_count,36487.0,109.663387,80.874216,1.0,15.0,26.0,43.0,78.0,170.0,250.0,250.0,250.0
average_rating,36487.0,3.161623,1.762326,1.020833,1.123364,1.188235,1.309278,4.703448,4.844503,4.918746,4.946734,5.0
average_helpful_vote,36487.0,1.450585,2.272433,0.0,0.095931,0.189189,0.464103,0.873555,1.655364,4.432372,9.103622,110.148936
average_review_text_length,36487.0,255.282469,116.476092,77.340426,107.455052,128.449066,176.137389,226.361111,301.652297,480.449834,669.619363,1458.325581


In [9]:
# Reviewing document distribution across satisfaction, price and priority segments

segment_distribution_dataframe = (
    electronics_nlp_documents_dataframe
    .groupby(
        ["analysis_pool_role", "sentiment_cohort", "verified_purchase", "price_band", "issue_priority_level"],
        dropna = False
    )
    .agg(
        document_count = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum"),
        average_aggregated_text_length = ("aggregated_text_length", "mean")
    )
    .reset_index()
    .sort_values(
        by = ["analysis_pool_role", "sentiment_cohort", "verified_purchase", "price_band", "issue_priority_level"]
    )
)

display(segment_distribution_dataframe)

,analysis_pool_role,sentiment_cohort,verified_purchase,price_band,issue_priority_level,document_count,total_sampled_reviews,average_aggregated_text_length
0,Comparison baseline,Satisfaction,True,Budget,Monitor,3703,315963,14844.635701
1,Comparison baseline,Satisfaction,True,Budget,Priority review,1070,163689,28525.88972
2,Comparison baseline,Satisfaction,True,Lower mid,Monitor,3721,342517,18707.086805
3,Comparison baseline,Satisfaction,True,Lower mid,Priority review,949,138171,32456.590095
4,Comparison baseline,Satisfaction,True,Premium,Monitor,2903,272286,27513.939029
5,Comparison baseline,Satisfaction,True,Premium,Priority review,1185,169992,44856.638819
6,Comparison baseline,Satisfaction,True,Upper mid,Monitor,3668,356748,23517.886314
7,Comparison baseline,Satisfaction,True,Upper mid,Priority review,1282,190817,39131.535881
8,Comparison baseline,Satisfaction,True,Very premium,Monitor,712,50585,25821.79073
9,Topic modeling target,Dissatisfaction,True,Budget,Monitor,3168,315949,21892.470013


In [10]:
# Cleaning and standardizing aggregated Electronics review text for NLP analysis
electronics_nlp_documents_dataframe["aggregated_text_clean"] = (
    electronics_nlp_documents_dataframe["aggregated_text"]
    .fillna("")
    .str.replace(r"\s+", " ", regex = True)
    .str.strip()
)

electronics_nlp_documents_dataframe["is_usable_text_document"] = (
    electronics_nlp_documents_dataframe["aggregated_text_clean"].ne("")
    & electronics_nlp_documents_dataframe["aggregated_text_length"].fillna(0).ge(100)
    & electronics_nlp_documents_dataframe["sampled_review_count"].fillna(0).ge(25)
)

post_load_quality_summary_dataframe = pd.DataFrame({
    "metric": [
        "usable_text_document_count",
        "excluded_text_document_count",
        "usable_topic_modeling_target_count",
        "usable_comparison_baseline_count"
    ],
    "value": [
        electronics_nlp_documents_dataframe["is_usable_text_document"].sum(),
        (~electronics_nlp_documents_dataframe["is_usable_text_document"]).sum(),
        (
            electronics_nlp_documents_dataframe["is_usable_text_document"]
            & electronics_nlp_documents_dataframe["analysis_pool_role"].eq("Topic modeling target")
        ).sum(),
        (
            electronics_nlp_documents_dataframe["is_usable_text_document"]
            & electronics_nlp_documents_dataframe["analysis_pool_role"].eq("Comparison baseline")
        ).sum()
    ]
})

display(post_load_quality_summary_dataframe)

,metric,value
0,usable_text_document_count,35213
1,excluded_text_document_count,1274
2,usable_topic_modeling_target_count,16853
3,usable_comparison_baseline_count,18360


In [11]:
# Previewing cleaned Electronics review documents prepared for NLP analysis

usable_documents_preview_dataframe = electronics_nlp_documents_dataframe.loc[
    electronics_nlp_documents_dataframe["is_usable_text_document"],
    [
        "document_id",
        "analysis_pool_role",
        "sentiment_cohort",
        "price_band",
        "issue_priority_level",
        "product_display_name",
        "sampled_review_count",
        "aggregated_text_length",
        "aggregated_text_clean"
    ]
].head(10)

display(usable_documents_preview_dataframe)

,document_id,analysis_pool_role,sentiment_cohort,price_band,issue_priority_level,product_display_name,sampled_review_count,aggregated_text_length,aggregated_text_clean
0,Satisfaction|true|Budget|Monitor|106171327X|ch...,Comparison baseline,Satisfaction,Budget,Monitor,SanDisk SDSDQUA-064G-A11 Professional Ultra 64...,27,2870,great price my last card died after a year so ...
1,Satisfaction|true|Budget|Monitor|B00000J1QR|ch...,Comparison baseline,Satisfaction,Budget,Monitor,"Allsop CD and DVD FastWipes, lint-free wipes f...",28,4261,great for keeping dvds or cdcs clean br br yes...
2,Satisfaction|true|Budget|Monitor|B00004Z5M1|ch...,Comparison baseline,Satisfaction,Budget,Monitor,Belkin - F3U133b10 (F3U133b10) Hi-Speed USB A/...,137,25729,i really appreciate the good value of this pri...
3,Satisfaction|true|Budget|Monitor|B00004ZCJF|ch...,Comparison baseline,Satisfaction,Budget,Monitor,Tiffen 49UVP 49mm UV Protection Camera Lens Fi...,32,5032,for the price i always buy tiffen lens filters...
4,Satisfaction|true|Budget|Monitor|B00004ZCJJ|ch...,Comparison baseline,Satisfaction,Budget,Monitor,Tiffen 62UVP 62mm UV Protection Filter,37,6772,no tiene problema con la fotograf a review_bou...
5,Satisfaction|true|Budget|Monitor|B00005MDZD|ch...,Comparison baseline,Satisfaction,Budget,Monitor,Gam3Gear Component AV Audio Video Cable for PS...,62,18761,this cable fixed my black and white problem wi...
6,Satisfaction|true|Budget|Monitor|B00005T3DX|ch...,Comparison baseline,Satisfaction,Budget,Monitor,"Duracell CR2 3V Lithium Battery, 1 Count Pack,...",27,3576,the batteries fit the security connections i n...
7,Satisfaction|true|Budget|Monitor|B00005T3EY|ch...,Comparison baseline,Satisfaction,Budget,Monitor,RCA Matching Transformer -VH54R,32,6265,old school transformer needed it for a vintage...
8,Satisfaction|true|Budget|Monitor|B00005T3GH|ch...,Comparison baseline,Satisfaction,Budget,Monitor,RCA AH216 Stereo Headphone Adapter Plug,29,5622,the package came in rca brand packaging seems ...
9,Satisfaction|true|Budget|Monitor|B0000668YX|ch...,Comparison baseline,Satisfaction,Budget,Monitor,Belkin Single Outlet MasterCube Wall-Mount Sur...,27,7864,our phones got fried in a nearby lightning str...


In [12]:
# Filtering usable Electronics NLP documents with sufficient text quality
usable_electronics_nlp_documents_dataframe = (
    electronics_nlp_documents_dataframe.loc[
        electronics_nlp_documents_dataframe["is_usable_text_document"]
    ].copy()
)

print(f"Usable documents retained for NLP preparation : {len(usable_electronics_nlp_documents_dataframe):,}")

Usable documents retained for NLP preparation : 35,213


In [13]:
# Reviewing text-cleaning artifacts and remaining formatting patterns
text_artifact_summary_dataframe = pd.DataFrame({
    "metric": [
        "document_count",
        "documents_with_html_break_tokens",
        "documents_with_encoded_html_entities",
        "documents_with_non_ascii_characters",
        "documents_with_repeated_whitespace",
        "documents_with_digit_sequences",
        "documents_with_very_long_text_over_100000_chars"
    ],
    "value": [
        len(usable_electronics_nlp_documents_dataframe),
        usable_electronics_nlp_documents_dataframe["aggregated_text_clean"].str.contains(r"\bbr\b", regex = True, na = False).sum(),
        usable_electronics_nlp_documents_dataframe["aggregated_text_clean"].str.contains(r"&[a-zA-Z]+;", regex = True, na = False).sum(),
        usable_electronics_nlp_documents_dataframe["aggregated_text_clean"].str.contains(r"[^\x00-\x7F]", regex = True, na = False).sum(),
        usable_electronics_nlp_documents_dataframe["aggregated_text_clean"].str.contains(r"\s{2,}", regex = True, na = False).sum(),
        usable_electronics_nlp_documents_dataframe["aggregated_text_clean"].str.contains(r"\d{2,}", regex = True, na = False).sum(),
        usable_electronics_nlp_documents_dataframe["aggregated_text_length"].gt(100000).sum()
    ]
})

display(text_artifact_summary_dataframe)

,metric,value
0,document_count,35213
1,documents_with_html_break_tokens,34671
2,documents_with_encoded_html_entities,0
3,documents_with_non_ascii_characters,0
4,documents_with_repeated_whitespace,0
5,documents_with_digit_sequences,34847
6,documents_with_very_long_text_over_100000_chars,695


In [14]:
# Defining grouping dimensions for Electronics NLP segmentation analysis
group_columns = [
    "analysis_pool_role",
    "price_band",
    "issue_priority_level"
]

diagnostic_sample_size_per_segment = 400
random_number_generator = np.random.default_rng(42)

diagnostic_sample_dataframe = (
    usable_electronics_nlp_documents_dataframe
    .assign(_random_order = random_number_generator.random(len(usable_electronics_nlp_documents_dataframe)))
    .sort_values(group_columns + ["_random_order"])
    .groupby(group_columns, dropna = False, group_keys = False)
    .head(diagnostic_sample_size_per_segment)
    .drop(columns = ["_random_order"])
    .reset_index(drop = True)
)

print(f"Diagnostic sample size : {len(diagnostic_sample_dataframe):,}")

diagnostic_segment_summary_dataframe = (
    diagnostic_sample_dataframe
    .groupby(group_columns, dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum"),
        average_aggregated_text_length = ("aggregated_text_length", "mean")
    )
    .reset_index()
    .sort_values(group_columns)
)

display(diagnostic_segment_summary_dataframe)

Diagnostic sample size : 7,200


,analysis_pool_role,price_band,issue_priority_level,document_count,total_sampled_reviews,average_aggregated_text_length
0,Comparison baseline,Budget,Monitor,400,36957,16056.8025
1,Comparison baseline,Budget,Priority review,400,62964,29284.56
2,Comparison baseline,Lower mid,Monitor,400,37060,19070.8875
3,Comparison baseline,Lower mid,Priority review,400,58542,32358.74
4,Comparison baseline,Premium,Monitor,400,39900,29048.9675
5,Comparison baseline,Premium,Priority review,400,58513,45702.2425
6,Comparison baseline,Upper mid,Monitor,400,42222,25456.2075
7,Comparison baseline,Upper mid,Priority review,400,59644,39842.3675
8,Comparison baseline,Very premium,Monitor,400,28451,25841.9075
9,Topic modeling target,Budget,Monitor,400,42753,23364.41


In [15]:
# Defining review boundary markers, text-cleaning tokens and protected technical terms

review_boundary_token = "review_boundary"

short_technical_tokens = {
    "tv", "pc", "vr", "hd", "sd", "ac", "dc",
    "usb", "ssd", "wifi", "hdmi", "ps5", "ps4",
    "oled", "qled", "lcd", "led", "dslr", "gps"
}

domain_stopwords = {
    "product", "products", "item", "items", "review", "reviews",
    "videoid",
    "month", "months", "day", "days",
    "did", "does", "got", "just", "thing", "things",
    "make", "makes", "made", "using", "used", "use",
    "new", "little", "nice", "way", "know", "want",
    "look", "looks", "looking"
}

month_stopwords = {
    "january", "february", "march", "april", "june", "july",
    "august", "september", "october", "november", "december",
    "jan", "feb", "mar", "apr", "jun", "jul", "aug", "sep", "sept",
    "oct", "nov", "dec"
}

low_value_noise_tokens = {
    "br", "quot", "nbsp", "amp"
}

artifact_stopwords = {
    "deadline", "pos", "wast", "waist", "straw", "avail",
    "honor", "smh", "wtf"
}

generic_sentiment_tokens = {
    "good", "great", "better", "best", "perfect", "recommend",
    "recommended", "happy", "easy", "love", "loved", "liked",
    "fine", "pretty", "absolutely"
}

generic_context_tokens = {
    "like", "time", "people", "say", "actually", "think", "lot",
    "small", "big", "old", "right", "long", "set", "stars",
    "able", "getting", "having", "times", "going", "needed",
    "different", "second", "far", "takes", "taking", "bit", "fact",
    "need", "device", "money", "away",
    "week", "weeks", "year", "years", "hour", "hours",
    "doing", "tell", "gets"
}

residual_generic_tokens = {
    "really", "sure", "wanted", "ones", "person", "stuff",
    "seen", "similar", "previous", "real", "extra", "deal", "type",
    "home", "house", "large", "high", "low", "perfectly"
}

evaluative_outcome_tokens = {
    "waste_money", "not_worth", "disappointed", "disappointing",
    "terrible", "horrible", "awful", "useless", "worthless",
    "junk", "garbage", "trash", "worst"
}

delivery_or_fulfillment_tokens = {
    "late_delivery", "on_time_delivery", "shipping", "ship", "shipped",
    "arrived", "arrive", "arrival", "arrived_damaged", "wrong_item",
    "package", "packaging", "box", "seller", "sellers", "amazon",
    "customer_service", "tech_support"
}

remedy_or_transaction_tokens = {
    "return", "returns", "returned", "returning",
    "refund", "refunds", "refunded",
    "return_window", "return_policy",
    "warranty", "warranties",
    "buyer", "buyers", "beware",
    "false_advertising", "false_advertisement",
    "exchange", "restocking",
    "pay_shipping"
}


base_stopwords = set(ENGLISH_STOP_WORDS)

combined_stopwords = (
    base_stopwords
    .union(domain_stopwords)
    .union(month_stopwords)
    .union(low_value_noise_tokens)
    .union(artifact_stopwords)
    .union(generic_sentiment_tokens)
    .union(generic_context_tokens)
    .union(residual_generic_tokens)
)

protected_phrase_tokens = {
    "does_not_work", "did_not_work", "not_working", "stopped_working",
    "will_not_turn_on", "not_turn_on",
    "battery_life", "battery_died",
    "sound_quality", "picture_quality", "build_quality",
    "customer_service", "tech_support",
    "not_charge", "not_charging",
    "not_connect", "not_connecting", "keeps_disconnecting",
    "not_pair", "not_pairing",
    "not_compatible", "not_fit",
    "late_delivery", "on_time_delivery",
    "return_policy", "return_window",
    "dead_on_arrival", "noise_cancelling",
    "smart_tv", "usb_c", "usb_a", "sd_card", "micro_sd",
    "bluetooth", "wifi", "hard_drive", "poor_quality",
    "no_signal", "no_sound", "no_power",
    review_boundary_token
}

token_normalization_map = {
    "works": "work",
    "worked": "work",
    "working": "work",
    "charges": "charge",
    "charged": "charge",
    "charging": "charge",
    "connects": "connect",
    "connected": "connect",
    "connecting": "connect",
    "disconnects": "disconnect",
    "disconnected": "disconnect",
    "disconnecting": "disconnect",
    "installs": "install",
    "installed": "install",
    "installing": "install",
    "pairs": "pair",
    "paired": "pair",
    "pairing": "pair",
    "returns": "return",
    "returned": "return",
    "returning": "return",
    "replaced": "replace",
    "replacing": "replace",
    "tries": "try",
    "tried": "try",
    "trying": "try",
    "batteries": "battery",
    "cables": "cable",
    "devices": "device",
    "headphones": "headphone",
    "speakers": "speaker",
    "routers": "router",
    "monitors": "monitor",
    "cameras": "camera",
    "wast": "wasted",
    "avail": "available",
    "dissappointed": "disappointed",
    "dissapointed": "disappointed"
}

contraction_replacements = {
    r"\bwon(?:['’]t|\s+t)\b": "will not",
    r"\bcan(?:['’]t|\s+t)\b": "can not",
    r"\bdon(?:['’]t|\s+t)\b": "do not",
    r"\bdoesn(?:['’]t|\s+t)\b": "does not",
    r"\bdidn(?:['’]t|\s+t)\b": "did not",
    r"\bisn(?:['’]t|\s+t)\b": "is not",
    r"\baren(?:['’]t|\s+t)\b": "are not",
    r"\bwasn(?:['’]t|\s+t)\b": "was not",
    r"\bweren(?:['’]t|\s+t)\b": "were not",
    r"\bshouldn(?:['’]t|\s+t)\b": "should not",
    r"\bcouldn(?:['’]t|\s+t)\b": "could not",
    r"\bwouldn(?:['’]t|\s+t)\b": "would not",
    r"\bhasn(?:['’]t|\s+t)\b": "has not",
    r"\bhaven(?:['’]t|\s+t)\b": "have not",
    r"\bhadn(?:['’]t|\s+t)\b": "had not",
    r"\bit(?:['’]s|\s+s)\b": "it is",
    r"\bthat(?:['’]s|\s+s)\b": "that is",
    r"\bthere(?:['’]s|\s+s)\b": "there is",
    r"\bi(?:['’]m|\s+m)\b": "i am",
    r"\bi(?:['’]ve|\s+ve)\b": "i have",
    r"\bwe(?:['’]ve|\s+ve)\b": "we have",
    r"\bthey(?:['’]ve|\s+ve)\b": "they have",
    r"\byou(?:['’]ve|\s+ve)\b": "you have",
    r"\bi(?:['’]ll|\s+ll)\b": "i will",
    r"\bwe(?:['’]ll|\s+ll)\b": "we will",
    r"\bthey(?:['’]ll|\s+ll)\b": "they will",
    r"\byou(?:['’]ll|\s+ll)\b": "you will",
    r"\bthey(?:['’]re|\s+re)\b": "they are",
    r"\bwe(?:['’]re|\s+re)\b": "we are",
    r"\byou(?:['’]re|\s+re)\b": "you are"
}

spelling_normalizations = {
    r"\bstoped\b": "stopped",
    r"\bdisapointed\b": "disappointed",
    r"\bdissapointed\b": "disappointed",
    r"\bdissappointed\b": "disappointed",
    r"\bperfet\b": "perfect",
    r"\bmala\s+calidad\b": "poor_quality"
}

phrase_replacements = {
    r"\bsound\s+quality\b": "sound_quality",
    r"\bpicture\s+quality\b": "picture_quality",
    r"\bbuild\s+quality\b": "build_quality",
    r"\bbattery\s+life\b": "battery_life",
    r"\bcustomer\s+service\b": "customer_service",
    r"\bcustomer\s+support\b": "customer_service",
    r"\btech\s+support\b": "tech_support",
    r"\bhard\s+drive\b": "hard_drive",
    r"\bsolid\s+state\s+drive\b": "ssd",
    r"\bscreen\s+protector\b": "screen_protector",
    r"\bplug\s+and\s+play\b": "plug_and_play",
    r"\bset\s+up\b": "setup",
    r"\bwi\s*fi\b": "wifi",
    r"\bblu[\s\-]*ray\b": "bluray",
    r"\bblu[\s\-]*tooth\b": "bluetooth",
    r"\busb[\s\-]*c\b": "usb_c",
    r"\busb[\s\-]*a\b": "usb_a",
    r"\bmicro[\s\-]*sd(?:\s+card)?\b": "micro_sd",
    r"\bsd[\s\-]*card\b": "sd_card",
    r"\bnoise\s+cancell(?:ing|ation)\b": "noise_cancelling",
    r"\bsmart\s+tv\b": "smart_tv",
    r"\b4k\b": "four_k",
    r"\b1080p\b": "full_hd",
    r"\b2\s*4\s*ghz\b": "two_four_ghz",
    r"\b5\s*ghz\b": "five_ghz",
    r"\bdoes\s+not\s+work\b": "does_not_work",
    r"\bdid\s+not\s+work\b": "did_not_work",
    r"\bnot\s+working\b": "not_working",
    r"\bstopped\s+working\b": "stopped_working",
    r"\bwill\s+not\s+turn\s+on\b": "will_not_turn_on",
    r"\bnot\s+turn(?:ing)?\s+on\b": "not_turn_on",
    r"\bdead\s+on\s+arrival\b": "dead_on_arrival",
    r"\bbattery\s+died\b": "battery_died",
    r"\bwill\s+not\s+charge\b": "not_charge",
    r"\bdoes\s+not\s+charge\b": "not_charge",
    r"\bnot\s+charging\b": "not_charging",
    r"\bdoes\s+not\s+connect\b": "not_connect",
    r"\bnot\s+connecting\b": "not_connecting",
    r"\bkeeps\s+disconnecting\b": "keeps_disconnecting",
    r"\bdoes\s+not\s+pair\b": "not_pair",
    r"\bnot\s+pairing\b": "not_pairing",
    r"\bnot\s+compatible\b": "not_compatible",
    r"\bdoes\s+not\s+fit\b": "not_fit",
    r"\blate\s+delivery\b": "late_delivery",
    r"\barrived\s+late\b": "late_delivery",
    r"\bon\s+time\b": "on_time_delivery",
    r"\breturn\s+window\b": "return_window",
    r"\breturn\s+policy\b": "return_policy",
    r"\bpoor\s+quality\b": "poor_quality",
    r"\bno\s+signal\b": "no_signal",
    r"\bno\s+sound\b": "no_sound",
    r"\bno\s+power\b": "no_power",
    r"\bfalse\s+advertising\b": "false_advertising",
    r"\bnot\s+as\s+described\b": "not_as_described",
    r"\bnot\s+worth\b": "not_worth",
    r"\bwaste\s+of\s+money\b": "waste_money",
    r"\bcustomer\s+care\b": "customer_service",
    r"\btech\s+help\b": "tech_support",
    r"\barrived\s+damaged\b": "arrived_damaged",
    r"\bwrong\s+item\b": "wrong_item",
    r"\bdefective\s+unit\b": "defective_unit",
    r"\bstop(?:ped)?\s+charging\b": "not_charging",
    r"\bstop(?:ped)?\s+connecting\b": "not_connecting"
}

repeated_token_pattern = re.compile(r"\b(\w+)(?:\s+\1){3,}\b")

def collapse_repeated_tokens(text_value, max_consecutive_repeats = 2):
    if not text_value:
        return ""

    input_tokens = text_value.split()
    output_tokens = []
    previous_token = None
    repeat_count = 0

    for token in input_tokens:
        if token == previous_token:
            repeat_count += 1
        else:
            previous_token = token
            repeat_count = 1

        if repeat_count <= max_consecutive_repeats:
            output_tokens.append(token)

    return " ".join(output_tokens)

def clip_text_balanced_by_review_boundaries(
    text_value,
    boundary_token = review_boundary_token,
    max_reviews = 120,
    min_reviews_to_clip = 140
):
    if not text_value:
        return "", False

    review_parts = [
        part.strip()
        for part in str(text_value).split(boundary_token)
        if part.strip()
    ]

    review_count = len(review_parts)

    if review_count <= min_reviews_to_clip:
        return f" {boundary_token} ".join(review_parts), False

    first_block_size = max_reviews // 3
    middle_block_size = max_reviews // 3
    last_block_size = max_reviews - first_block_size - middle_block_size

    middle_start_index = max((review_count // 2) - (middle_block_size // 2), 0)
    middle_end_index = middle_start_index + middle_block_size

    clipped_reviews = (
        review_parts[ : first_block_size]
        + review_parts[middle_start_index:middle_end_index]
        + review_parts[-last_block_size:]
    )

    return f" {boundary_token} ".join(clipped_reviews), True


def clean_electronics_review_text(text_value):
    if pd.isna(text_value):
        return ""

    cleaned_text = str(text_value)
    cleaned_text = html.unescape(cleaned_text)
    cleaned_text = unicodedata.normalize("NFKC", cleaned_text)
    cleaned_text = cleaned_text.lower()

    cleaned_text = re.sub(rf"\b{review_boundary_token}\b", f" {review_boundary_token} ", cleaned_text)
    cleaned_text = cleaned_text.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", cleaned_text)

    for pattern, replacement in contraction_replacements.items():
        cleaned_text = re.sub(pattern, replacement, cleaned_text)

    for pattern, replacement in spelling_normalizations.items():
        cleaned_text = re.sub(pattern, replacement, cleaned_text)

    cleaned_text = re.sub(r"\bbr\b", " ", cleaned_text)
    cleaned_text = re.sub(r"\bvideoid\b", " ", cleaned_text)

    # Normalizing separators before phrase preservation so patterns like 
    # "return-window" and "blu-tooth" can still be captured.
    cleaned_text = re.sub(r"[-/]+", " ", cleaned_text)

    for pattern, replacement in phrase_replacements.items():
        cleaned_text = re.sub(pattern, replacement, cleaned_text)

    cleaned_text = re.sub(r"[~`]+", " ", cleaned_text)
    cleaned_text = re.sub(r"[^a-z0-9_\s]", " ", cleaned_text)
    cleaned_text = re.sub(r"\b[a-f0-9]{12,}\b", " ", cleaned_text)
    cleaned_text = re.sub(r"\b(?=\w*[a-z])(?=\w*\d)[a-z0-9_]{8,}\b", " ", cleaned_text)
    cleaned_text = re.sub(r"\b(?:amp|quot|nbsp)\b", " ", cleaned_text)
    cleaned_text = re.sub(r"\b\d{1,2}(?:st|nd|rd|th)\b", " ", cleaned_text)
    cleaned_text = re.sub(r"\b(?:year|month|day|week)s?\s+old\b", " ", cleaned_text)
    cleaned_text = re.sub(r"(?<![a-z_])\d+(?![a-z_])", " ", cleaned_text)
    cleaned_text = re.sub(r"\b[a-z]\b", " ", cleaned_text)
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()
    cleaned_text = collapse_repeated_tokens(cleaned_text, max_consecutive_repeats = 2)

    return cleaned_text

def build_modeling_text(text_value):
    if not text_value:
        return ""

    output_tokens = []

    for token in str(text_value).split():
        if token == review_boundary_token:
            output_tokens.append(review_boundary_token)
            continue

        token = token_normalization_map.get(token, token)

        if token.isdigit():
            continue

        if token in combined_stopwords:
            continue

        if token in protected_phrase_tokens:
            output_tokens.append(token)
            continue

        if len(token) >= 3 or token in short_technical_tokens:
            output_tokens.append(token)

    return " ".join(output_tokens)

diagnostic_sample_dataframe["modeling_text_clean"] = (
    diagnostic_sample_dataframe["aggregated_text_clean"]
    .fillna("")
    .apply(clean_electronics_review_text)
)

diagnostic_sample_dataframe["clean_word_count_preclip"] = (
    diagnostic_sample_dataframe["modeling_text_clean"].str.split().str.len()
)

clip_results = diagnostic_sample_dataframe["modeling_text_clean"].apply(
    lambda value: clip_text_balanced_by_review_boundaries(
        value,
        boundary_token = review_boundary_token,
        max_reviews = 120,
        min_reviews_to_clip = 140
    )
)

diagnostic_sample_dataframe["modeling_text_clipped"] = clip_results.apply(lambda value: value[0])
diagnostic_sample_dataframe["clipping_applied"] = clip_results.apply(lambda value: value[1])

diagnostic_sample_dataframe["modeling_text_for_vectorizer"] = (
    diagnostic_sample_dataframe["modeling_text_clipped"]
    .apply(build_modeling_text)
)

diagnostic_sample_dataframe["diagnostic_tokens"] = (
    diagnostic_sample_dataframe["modeling_text_for_vectorizer"]
    .apply(lambda value: [token for token in value.split() if token != review_boundary_token])
)

diagnostic_sample_dataframe["raw_word_count"] = (
    diagnostic_sample_dataframe["aggregated_text_clean"].str.split().str.len()
)

diagnostic_sample_dataframe["clean_word_count_postclip"] = (
    diagnostic_sample_dataframe["modeling_text_clipped"].str.split().str.len()
)

diagnostic_sample_dataframe["diagnostic_token_count"] = (
    diagnostic_sample_dataframe["diagnostic_tokens"].str.len()
)

diagnostic_sample_dataframe["unique_token_count"] = (
    diagnostic_sample_dataframe["diagnostic_tokens"].apply(lambda values: len(set(values)))
)

diagnostic_sample_dataframe["unique_token_ratio"] = (
    diagnostic_sample_dataframe["unique_token_count"] /
    diagnostic_sample_dataframe["diagnostic_token_count"].replace(0, np.nan)
)

diagnostic_sample_dataframe["cleaning_retention_ratio_preclip"] = (
    diagnostic_sample_dataframe["clean_word_count_preclip"] /
    diagnostic_sample_dataframe["raw_word_count"].replace(0, np.nan)
)

diagnostic_sample_dataframe["has_repeated_token_sequences"] = (
    diagnostic_sample_dataframe["modeling_text_clipped"]
    .apply(lambda value: bool(repeated_token_pattern.search(value)))
)

In [16]:
# Summarizing diagnostic quality checks for Electronics NLP documents

diagnostic_quality_summary_dataframe = pd.DataFrame({
    "metric": [
        "diagnostic_document_count",
        "blank_modeling_text_count",
        "documents_under_100_words_after_cleaning",
        "documents_under_50_diagnostic_tokens",
        "documents_with_repeated_token_sequences",
        "documents_with_cleaning_retention_ratio_below_0_40",
        "documents_with_unique_token_ratio_below_0_20",
        "documents_with_clipping_applied",
        "documents_with_br_after_cleaning",
        "documents_with_non_ascii_after_cleaning",
        "documents_with_no_signal_token",
        "documents_with_no_sound_token",
        "documents_with_no_power_token"
    ],
    "value": [
        len(diagnostic_sample_dataframe),
        diagnostic_sample_dataframe["modeling_text_clipped"].fillna("").str.strip().eq("").sum(),
        diagnostic_sample_dataframe["clean_word_count_postclip"].lt(100).sum(),
        diagnostic_sample_dataframe["diagnostic_token_count"].lt(50).sum(),
        diagnostic_sample_dataframe["has_repeated_token_sequences"].sum(),
        diagnostic_sample_dataframe["cleaning_retention_ratio_preclip"].lt(0.40).sum(),
        diagnostic_sample_dataframe["unique_token_ratio"].lt(0.20).sum(),
        diagnostic_sample_dataframe["clipping_applied"].sum(),
        diagnostic_sample_dataframe["modeling_text_clipped"].str.contains(r"\bbr\b", regex = True, na = False).sum(),
        diagnostic_sample_dataframe["modeling_text_clipped"].str.contains(r"[^\x00-\x7F]", regex = True, na = False).sum(),
        diagnostic_sample_dataframe["modeling_text_clipped"].str.contains(r"\bno_signal\b", regex = True, na = False).sum(),
        diagnostic_sample_dataframe["modeling_text_clipped"].str.contains(r"\bno_sound\b", regex = True, na = False).sum(),
        diagnostic_sample_dataframe["modeling_text_clipped"].str.contains(r"\bno_power\b", regex = True, na = False).sum()
    ]
})

phrase_validation_tokens = [
    "does_not_work",
    "did_not_work",
    "stopped_working",
    "battery_life",
    "customer_service",
    "return_window",
    "late_delivery",
    "false_advertising",
    "not_charging",
    "not_connecting",
    "not_compatible",
    "poor_quality",
    "no_signal",
    "no_sound",
    "no_power"
]

phrase_validation_rows = []

for token_value in phrase_validation_tokens:
    phrase_validation_rows.append({
        "token": token_value,
        "document_count": diagnostic_sample_dataframe["modeling_text_for_vectorizer"]
            .str.contains(rf"\b{re.escape(token_value)}\b", regex = True, na = False)
            .sum()
    })

phrase_validation_dataframe = pd.DataFrame(phrase_validation_rows).sort_values(
    by = "document_count",
    ascending = False
)


all_diagnostic_tokens = [
    token
    for token_list in diagnostic_sample_dataframe["diagnostic_tokens"]
    for token in token_list
]

overall_top_tokens_dataframe = pd.DataFrame(
    Counter(all_diagnostic_tokens).most_common(100),
    columns = ["token", "frequency"]
)

vectorizer = CountVectorizer(
    lowercase = False,
    stop_words = None,
    token_pattern = r"(?u)\b[a-z_][a-z0-9_]{2,}\b",
    ngram_range = (1, 2),
    min_df = 10
)

document_term_matrix = vectorizer.fit_transform(
    diagnostic_sample_dataframe["modeling_text_for_vectorizer"]
)

terms = np.array(vectorizer.get_feature_names_out())
valid_feature_mask = np.array([review_boundary_token not in term for term in terms])

document_term_matrix = document_term_matrix[ : , valid_feature_mask]
terms = terms[valid_feature_mask]

dissatisfaction_mask = diagnostic_sample_dataframe["analysis_pool_role"].eq("Topic modeling target").to_numpy()
satisfaction_mask = diagnostic_sample_dataframe["analysis_pool_role"].eq("Comparison baseline").to_numpy()

dissatisfaction_document_frequency = np.asarray(
    (document_term_matrix[dissatisfaction_mask] > 0).sum(axis = 0)
).ravel()

satisfaction_document_frequency = np.asarray(
    (document_term_matrix[satisfaction_mask] > 0).sum(axis = 0)
).ravel()

dissatisfaction_document_count = dissatisfaction_mask.sum()
satisfaction_document_count = satisfaction_mask.sum()

token_lift_dataframe = pd.DataFrame({
    "token_or_phrase": terms,
    "dissatisfaction_document_frequency": dissatisfaction_document_frequency,
    "dissatisfaction_document_frequency_ratio": dissatisfaction_document_frequency / dissatisfaction_document_count,
    "satisfaction_document_frequency": satisfaction_document_frequency,
    "satisfaction_document_frequency_ratio": satisfaction_document_frequency / satisfaction_document_count
})

token_lift_dataframe["dissatisfaction_vs_satisfaction_ratio"] = (
    (token_lift_dataframe["dissatisfaction_document_frequency_ratio"] + 1e-6) /
    (token_lift_dataframe["satisfaction_document_frequency_ratio"] + 1e-6)
)

token_lift_dataframe["absolute_ratio_gap"] = (
    token_lift_dataframe["dissatisfaction_document_frequency_ratio"] -
    token_lift_dataframe["satisfaction_document_frequency_ratio"]
)

differential_topic_terms_dataframe = (
    token_lift_dataframe
    .query("dissatisfaction_document_frequency >= 25 and satisfaction_document_frequency >= 5")
    .sort_values(
        by = ["dissatisfaction_vs_satisfaction_ratio", "absolute_ratio_gap"],
        ascending = [False, False]
    )
    .reset_index(drop = True)
)

generic_top_terms_dataframe = (
    token_lift_dataframe
    .sort_values(
        by = ["dissatisfaction_document_frequency_ratio", "satisfaction_document_frequency_ratio"],
        ascending = [False, False]
    )
    .reset_index(drop = True)
)

display(diagnostic_quality_summary_dataframe)
display(phrase_validation_dataframe)
display(overall_top_tokens_dataframe.head(80))
display(differential_topic_terms_dataframe.head(120))
display(generic_top_terms_dataframe.head(120))

,metric,value
0,diagnostic_document_count,7200
1,blank_modeling_text_count,0
2,documents_under_100_words_after_cleaning,0
3,documents_under_50_diagnostic_tokens,0
4,documents_with_repeated_token_sequences,0
5,documents_with_cleaning_retention_ratio_below_...,0
6,documents_with_unique_token_ratio_below_0_20,0
7,documents_with_clipping_applied,2474
8,documents_with_br_after_cleaning,0
9,documents_with_non_ascii_after_cleaning,0


,token,document_count
1,did_not_work,4057
0,does_not_work,4010
2,stopped_working,3543
4,customer_service,3536
5,return_window,2060
3,battery_life,1917
11,poor_quality,1770
10,not_compatible,1223
8,not_charging,918
13,no_sound,906


,token,frequency
0,work,232130
1,try,68661
2,bought,66587
3,camera,63889
4,charge,60858
5,sound,56073
6,return,55402
7,buy,53005
8,cable,52279
9,phone,51165


,token_or_phrase,dissatisfaction_document_frequency,dissatisfaction_document_frequency_ratio,satisfaction_document_frequency,satisfaction_document_frequency_ratio,dissatisfaction_vs_satisfaction_ratio,absolute_ratio_gap
0,total waste_money,568,0.157778,7,0.001944,81.101662,0.155833
1,missed return_window,363,0.100833,5,0.001389,72.548485,0.099444
2,wish return,553,0.153611,9,0.002500,61.420276,0.151111
3,complete waste_money,555,0.154167,10,0.002778,55.480387,0.151389
4,super disappointed,328,0.091111,6,0.001667,54.634486,0.089444
5,disappointed return,371,0.103056,7,0.001944,52.973271,0.101111
6,complete waste,260,0.072222,5,0.001389,51.963306,0.070833
7,terrible quality,305,0.084722,6,0.001667,50.803451,0.083056
8,definitely not_worth,301,0.083611,6,0.001667,50.137184,0.081944
9,huge disappointment,236,0.065556,5,0.001389,47.166760,0.064167


,token_or_phrase,dissatisfaction_document_frequency,dissatisfaction_document_frequency_ratio,satisfaction_document_frequency,satisfaction_document_frequency_ratio,dissatisfaction_vs_satisfaction_ratio,absolute_ratio_gap
0,return,3588,0.996667,1893,0.525833,1.895402,0.470833
1,work,3585,0.995833,3588,0.996667,0.999164,-0.000833
2,bought,3570,0.991667,3525,0.979167,1.012766,0.012500
3,try,3562,0.989444,3072,0.853333,1.159505,0.136111
4,buy,3552,0.986667,3433,0.953611,1.034664,0.033056
5,disappointed,3418,0.949444,1473,0.409167,2.320431,0.540278
6,purchased,3387,0.940833,3156,0.876667,1.073194,0.064167
7,amazon,3339,0.927500,2526,0.701667,1.321852,0.225833
8,bad,3320,0.922222,2230,0.619444,1.488788,0.302778
9,quality,3295,0.915278,3365,0.934722,0.979198,-0.019444


In [17]:
# Creating a balanced Electronics NLP sample for exploratory diagnostics and validation

sample_modeling_dataframe = diagnostic_sample_dataframe.loc[
    diagnostic_sample_dataframe["modeling_text_for_vectorizer"].fillna("").str.strip().ne("")
].copy()

sample_modeling_dataframe["is_dissatisfaction"] = (
    sample_modeling_dataframe["analysis_pool_role"] == "Topic modeling target"
)

print(f"Sample documents for pilot modeling : {len(sample_modeling_dataframe):,}")
print(f"Dissatisfaction documents : {sample_modeling_dataframe['is_dissatisfaction'].sum():,}")
print(f"Comparison baseline documents : {(~sample_modeling_dataframe['is_dissatisfaction']).sum():,}")

Sample documents for pilot modeling : 7,200
Dissatisfaction documents : 3,600
Comparison baseline documents : 3,600


In [18]:
# Creating dissatisfaction-focused Electronics review samples for issue-theme exploration

dissatisfaction_sample_dataframe = sample_modeling_dataframe.loc[
    sample_modeling_dataframe["is_dissatisfaction"]
].copy()

tfidf_vectorizer = TfidfVectorizer(
    lowercase = False,
    stop_words = None,
    token_pattern = r"(?u)\b[a-z_][a-z0-9_]{2,}\b",
    ngram_range = (1, 2),
    min_df = 15,
    max_df = 0.35,
    max_features = 20000,
    sublinear_tf = True
)

dissatisfaction_tfidf_matrix = tfidf_vectorizer.fit_transform(
    dissatisfaction_sample_dataframe["modeling_text_for_vectorizer"]
)

topic_count = 12

nmf_model = NMF(
    n_components = topic_count,
    init = "nndsvda",
    random_state = 42,
    max_iter = 400
)

dissatisfaction_topic_matrix = nmf_model.fit_transform(dissatisfaction_tfidf_matrix)
topic_term_matrix = nmf_model.components_
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

top_term_count = 15
topic_rows = []

for topic_index, topic_weights in enumerate(topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : top_term_count]
    topic_rows.append({
        "topic_id": topic_index,
        "top_terms": ", ".join(feature_names[top_term_indices])
    })

sample_topic_terms_dataframe = pd.DataFrame(topic_rows)
display(sample_topic_terms_dataframe)

,topic_id,top_terms
0,0,"router, wifi, network, netgear, modem, interne..."
1,1,"ear, headphone, earbuds, ears, buds, earbud, s..."
2,2,"review_boundary case, screen_protector, ipad, ..."
3,3,"chargers, charge phone, review_boundary cable,..."
4,4,"drive, drives, files, hard_drive, disk, window..."
5,5,"sound_quality, radio, music, bass, bluetooth, ..."
6,6,"camera, review_boundary camera, camera work, c..."
7,7,"mouse, keyboard, review_boundary mouse, keys, ..."
8,8,"hdmi, monitor, hdmi cable, display, four_k, hd..."
9,9,"fitbit, watch, review_boundary watch, heart ra..."


In [19]:
# Resetting dissatisfaction sample indexing for downstream processing consistency

dissatisfaction_sample_dataframe = dissatisfaction_sample_dataframe.reset_index(drop = True)

document_topic_assignment = dissatisfaction_topic_matrix.argmax(axis = 1)
document_topic_strength = dissatisfaction_topic_matrix.max(axis = 1)

dissatisfaction_sample_dataframe["topic_id"] = document_topic_assignment
dissatisfaction_sample_dataframe["topic_strength"] = document_topic_strength

representative_topic_documents_dataframe = (
    dissatisfaction_sample_dataframe[
        [
            "topic_id",
            "topic_strength",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "sampled_review_count",
            "modeling_text_for_vectorizer"
        ]
    ]
    .sort_values(["topic_id", "topic_strength"], ascending = [True, False])
    .groupby("topic_id", group_keys = False)
    .head(5)
    .reset_index(drop = True)
)

display(representative_topic_documents_dataframe)

,topic_id,topic_strength,price_band,issue_priority_level,product_display_name,sampled_review_count,modeling_text_for_vectorizer
0,0,0.147679,Very premium,Monitor,NETGEAR Nighthawk X6 Smart Wi-Fi Router (R8000...,250,pile scrap simplest activities router pass dns...
1,0,0.147193,Very premium,Monitor,NETGEAR Nighthawk X6 Smart Wi-Fi Router (R8000...,250,router cheapo linksys bought powerful router v...
2,0,0.144883,Premium,Monitor,NETGEAR Nighthawk Smart WiFi Router (R7000P) -...,250,drops signal constantly bad outside close yard...
3,0,0.143458,Very premium,Monitor,NETGEAR Orbi Whole Home Tri-band Mesh WiFi 6 S...,250,update2 replacement work satellite stopped_wor...
4,0,0.141907,Very premium,Monitor,NETGEAR Orbi Whole Home Tri-band Mesh WiFi 6 S...,250,bought linksys usb port clearly stated work ce...
5,1,0.202116,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,work ago earbuds cut watching listening music ...
6,1,0.201013,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,140,expecting charge case average size compared no...
7,1,0.199576,Premium,Priority review,Sony WF-SP800N Truly Wireless Sports In-Ear No...,121,sound_quality earpiece left earpiece try earpi...
8,1,0.196052,Lower mid,Monitor,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,250,ear synced piece work try resetting saved box ...
9,1,0.194319,Lower mid,Monitor,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,250,sound_quality okay lifespan short left earbud ...


In [20]:
# Creating the full Electronics NLP corpus for full-scale NLP modeling

full_corpus_dataframe = usable_electronics_nlp_documents_dataframe.copy()

full_corpus_dataframe["modeling_text_clean"] = (
    full_corpus_dataframe["aggregated_text_clean"]
    .fillna("")
    .apply(clean_electronics_review_text)
)

full_corpus_dataframe["clean_word_count_preclip"] = (
    full_corpus_dataframe["modeling_text_clean"].str.split().str.len()
)

full_clip_results = full_corpus_dataframe["modeling_text_clean"].apply(
    lambda value: clip_text_balanced_by_review_boundaries(
        value,
        boundary_token = review_boundary_token,
        max_reviews = 120,
        min_reviews_to_clip = 140
    )
)

full_corpus_dataframe["modeling_text_clipped"] = full_clip_results.apply(lambda value: value[0])
full_corpus_dataframe["clipping_applied"] = full_clip_results.apply(lambda value: value[1])

full_corpus_dataframe["modeling_text_for_vectorizer"] = (
    full_corpus_dataframe["modeling_text_clipped"]
    .apply(build_modeling_text)
    .str.replace(rf"\b{review_boundary_token}\b", " ", regex = True)
    .str.replace(r"\s+", " ", regex = True)
    .str.strip()
)

full_corpus_dataframe["modeling_tokens"] = (
    full_corpus_dataframe["modeling_text_for_vectorizer"]
    .apply(lambda value: value.split() if value else [])
)

full_corpus_dataframe["raw_word_count"] = (
    full_corpus_dataframe["aggregated_text_clean"].str.split().str.len()
)

full_corpus_dataframe["clean_word_count_postclip"] = (
    full_corpus_dataframe["modeling_text_clipped"].str.split().str.len()
)

full_corpus_dataframe["modeling_token_count"] = (
    full_corpus_dataframe["modeling_tokens"].str.len()
)

full_corpus_dataframe["unique_token_count"] = (
    full_corpus_dataframe["modeling_tokens"].apply(lambda values: len(set(values)))
)

full_corpus_dataframe["unique_token_ratio"] = (
    full_corpus_dataframe["unique_token_count"] /
    full_corpus_dataframe["modeling_token_count"].replace(0, np.nan)
)

full_corpus_dataframe["cleaning_retention_ratio_preclip"] = (
    full_corpus_dataframe["clean_word_count_preclip"] /
    full_corpus_dataframe["raw_word_count"].replace(0, np.nan)
)

full_corpus_dataframe["has_repeated_token_sequences"] = (
    full_corpus_dataframe["modeling_text_clipped"]
    .apply(lambda value: bool(repeated_token_pattern.search(value)))
)

print(f"Full corpus documents prepared : {len(full_corpus_dataframe):,}")

Full corpus documents prepared : 35,213


In [21]:
# Reviewing full-corpus text quality before modeling
full_corpus_quality_summary_dataframe = pd.DataFrame({
    "metric": [
        "full_document_count",
        "blank_modeling_text_count",
        "documents_under_100_words_after_cleaning",
        "documents_under_50_modeling_tokens",
        "documents_with_repeated_token_sequences",
        "documents_with_cleaning_retention_ratio_below_0_40",
        "documents_with_unique_token_ratio_below_0_20",
        "documents_with_clipping_applied",
        "documents_with_review_boundary_after_vectorizer_text"
    ],
    "value": [
        len(full_corpus_dataframe),
        full_corpus_dataframe["modeling_text_for_vectorizer"].fillna("").str.strip().eq("").sum(),
        full_corpus_dataframe["clean_word_count_postclip"].lt(100).sum(),
        full_corpus_dataframe["modeling_token_count"].lt(50).sum(),
        full_corpus_dataframe["has_repeated_token_sequences"].sum(),
        full_corpus_dataframe["cleaning_retention_ratio_preclip"].lt(0.40).sum(),
        full_corpus_dataframe["unique_token_ratio"].lt(0.20).sum(),
        full_corpus_dataframe["clipping_applied"].sum(),
        full_corpus_dataframe["modeling_text_for_vectorizer"].str.contains(
            rf"\b{review_boundary_token}\b",
            regex = True,
            na = False
        ).sum()
    ]
})

display(full_corpus_quality_summary_dataframe)

,metric,value
0,full_document_count,35213
1,blank_modeling_text_count,0
2,documents_under_100_words_after_cleaning,0
3,documents_under_50_modeling_tokens,0
4,documents_with_repeated_token_sequences,0
5,documents_with_cleaning_retention_ratio_below_...,0
6,documents_with_unique_token_ratio_below_0_20,0
7,documents_with_clipping_applied,10820
8,documents_with_review_boundary_after_vectorize...,0


In [22]:
# Creating the final modeling corpus with usable text records

full_corpus_modeling_dataframe = full_corpus_dataframe.loc[
    full_corpus_dataframe["modeling_text_for_vectorizer"].fillna("").str.strip().ne("")
].copy()

full_corpus_modeling_dataframe["is_dissatisfaction"] = (
    full_corpus_modeling_dataframe["analysis_pool_role"] == "Topic modeling target"
)

print(f"Full modeling documents : {len(full_corpus_modeling_dataframe):,}")
print("Dissatisfaction documents : "
      f"{full_corpus_modeling_dataframe['is_dissatisfaction'].sum():,}"
)
print("Comparison baseline documents : "
      f"{(~full_corpus_modeling_dataframe['is_dissatisfaction']).sum():,}"
)

Full modeling documents : 35,213
Dissatisfaction documents : 16,853
Comparison baseline documents : 18,360


In [23]:
# Building the dissatisfaction-only corpus for topic modeling

full_dissatisfaction_dataframe = full_corpus_modeling_dataframe.loc[
    full_corpus_modeling_dataframe["is_dissatisfaction"]
].copy()

full_tfidf_vectorizer = TfidfVectorizer(
    lowercase = False,
    stop_words = None,
    token_pattern = r"(?u)\b[a-z_][a-z0-9_]{2,}\b",
    ngram_range = (1, 2),
    min_df = 25,
    max_df = 0.35,
    max_features = 30000,
    sublinear_tf = True
)

full_dissatisfaction_tfidf_matrix = full_tfidf_vectorizer.fit_transform(
    full_dissatisfaction_dataframe["modeling_text_for_vectorizer"]
)

print(full_dissatisfaction_tfidf_matrix.shape)

(16853, 30000)


In [24]:
# Comparing different topic counts to choose a stable topic model

candidate_topic_counts = [10, 12, 15, 18]

topic_model_diagnostics = []

for topic_count in candidate_topic_counts:
    candidate_nmf_model = NMF(
        n_components = topic_count,
        init = "nndsvda",
        random_state = 42,
        max_iter = 400
    )

    candidate_document_topic_matrix = candidate_nmf_model.fit_transform(full_dissatisfaction_tfidf_matrix)

    topic_model_diagnostics.append({
        "topic_count": topic_count,
        "reconstruction_error": candidate_nmf_model.reconstruction_err_,
        "mean_max_topic_strength": candidate_document_topic_matrix.max(axis = 1).mean(),
        "median_max_topic_strength": np.median(candidate_document_topic_matrix.max(axis = 1))
    })

topic_model_diagnostics_dataframe = pd.DataFrame(topic_model_diagnostics)
display(topic_model_diagnostics_dataframe)

,topic_count,reconstruction_error,mean_max_topic_strength,median_max_topic_strength
0,10,122.206903,0.073579,0.066558
1,12,121.683329,0.075101,0.070779
2,15,121.035939,0.078982,0.075012
3,18,120.479698,0.081087,0.075200


In [25]:
# Training candidate topic models and collecting representative topic examples

candidate_topic_counts = [10, 12, 15, 18]

candidate_topic_models = {}
candidate_topic_review_rows = []

for topic_count in candidate_topic_counts:
    candidate_nmf_model = NMF(
        n_components = topic_count,
        init = "nndsvda",
        random_state = 42,
        max_iter = 400
    )

    candidate_document_topic_matrix = candidate_nmf_model.fit_transform(full_dissatisfaction_tfidf_matrix)
    candidate_topic_term_matrix = candidate_nmf_model.components_

    candidate_topic_models[topic_count] = {
        "nmf_model": candidate_nmf_model,
        "document_topic_matrix": candidate_document_topic_matrix,
        "topic_term_matrix": candidate_topic_term_matrix
    }

    dominant_topic_assignment = candidate_document_topic_matrix.argmax(axis = 1)
    dominant_topic_strength = candidate_document_topic_matrix.max(axis = 1)

    topic_size_dataframe = (
        pd.DataFrame({
            "topic_id": dominant_topic_assignment,
            "topic_strength": dominant_topic_strength
        })
        .groupby("topic_id", dropna = False)
        .agg(
            document_count = ("topic_id", "count"),
            average_topic_strength = ("topic_strength", "mean"),
            median_topic_strength = ("topic_strength", "median")
        )
        .reset_index()
    )

    topic_entropy_proxy = (
        topic_size_dataframe["document_count"] / topic_size_dataframe["document_count"].sum()
    )

    candidate_topic_review_rows.append({
        "topic_count": topic_count,
        "reconstruction_error": candidate_nmf_model.reconstruction_err_,
        "mean_max_topic_strength": dominant_topic_strength.mean(),
        "median_max_topic_strength": np.median(dominant_topic_strength),
        "smallest_topic_document_count": topic_size_dataframe["document_count"].min(),
        "largest_topic_document_count": topic_size_dataframe["document_count"].max(),
        "topic_size_ratio_max_to_min": (
            topic_size_dataframe["document_count"].max() /
            topic_size_dataframe["document_count"].min()
        )
    })

candidate_topic_review_dataframe = pd.DataFrame(candidate_topic_review_rows)
display(candidate_topic_review_dataframe.sort_values("topic_count"))

,topic_count,reconstruction_error,mean_max_topic_strength,median_max_topic_strength,smallest_topic_document_count,largest_topic_document_count,topic_size_ratio_max_to_min
0,10,122.206903,0.073579,0.066558,494,3125,6.325911
1,12,121.683329,0.075101,0.070779,510,2275,4.460784
2,15,121.035939,0.078982,0.075012,474,1937,4.086498
3,18,120.479698,0.081087,0.075200,421,1945,4.619952


In [26]:
# Selecting the final broad topic model for Electronics dissatisfaction analysis

selected_topic_count = 15

selected_nmf_model = candidate_topic_models[selected_topic_count]["nmf_model"]
selected_document_topic_matrix = candidate_topic_models[selected_topic_count]["document_topic_matrix"]
selected_topic_term_matrix = candidate_topic_models[selected_topic_count]["topic_term_matrix"]
selected_feature_names = np.array(full_tfidf_vectorizer.get_feature_names_out())

print(f"Selected topic count for detailed review : {selected_topic_count}")

Selected topic count for detailed review : 15


In [27]:
# Assigning selected topics to dissatisfied Electronics documents

full_dissatisfaction_topic_review_dataframe = full_dissatisfaction_dataframe.reset_index(drop = True).copy()

full_dissatisfaction_topic_review_dataframe["topic_id"] = selected_document_topic_matrix.argmax(axis = 1)
full_dissatisfaction_topic_review_dataframe["topic_strength"] = selected_document_topic_matrix.max(axis = 1)

print(f"Assigned topics to {len(full_dissatisfaction_topic_review_dataframe):,} dissatisfaction documents")

Assigned topics to 16,853 dissatisfaction documents


In [28]:
# Extracting top keywords for each selected topic

topic_top_term_rows = []

for topic_index, topic_weights in enumerate(selected_topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : 15]
    topic_top_term_rows.append({
        "topic_id": topic_index,
        "top_terms": ", ".join(selected_feature_names[top_term_indices])
    })

topic_top_terms_dataframe = pd.DataFrame(topic_top_term_rows)

topic_size_summary_dataframe = (
    full_dissatisfaction_topic_review_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean"),
        median_topic_strength = ("topic_strength", "median"),
        average_rating = ("average_rating", "mean"),
        average_helpful_vote = ("average_helpful_vote", "mean")
    )
    .reset_index()
    .sort_values("document_count", ascending = False)
)

full_topic_summary_dataframe = topic_size_summary_dataframe.merge(
    topic_top_terms_dataframe,
    on = "topic_id",
    how = "left"
)

display(full_topic_summary_dataframe)

,topic_id,document_count,total_sampled_reviews,average_topic_strength,median_topic_strength,average_rating,average_helpful_vote,top_terms
0,1,1937,285262,0.087805,0.089475,1.358950,1.290585,"ear, headphone, earbuds, ears, buds, earbud, s..."
1,2,1649,151277,0.075462,0.074127,1.377515,1.053450,"ipad, not_fit, protect, color, cases, fits, co..."
2,4,1545,183129,0.075803,0.074010,1.231458,2.416205,"drive, drives, files, disk, data, hard_drive, ..."
3,8,1508,185948,0.061545,0.061543,1.324395,1.637885,"speaker, sound_quality, music, bluetooth, bass..."
4,7,1339,156051,0.077219,0.075124,1.303286,3.327786,"camera, camera work, motion, recording, view, ..."
5,6,1246,105915,0.069002,0.062515,1.359285,1.684490,"screws, mount, screw, mounting, bracket, tight..."
6,3,1231,162709,0.086244,0.089526,1.256194,1.215423,"charge phone, cords, chargers, charger, not_ch..."
7,9,1109,116610,0.075755,0.080711,1.261656,1.617113,"hdmi, monitor, hdmi cable, display, adapter, p..."
8,12,1060,109780,0.071391,0.054123,1.256757,2.432281,"charger, charge battery, hold charge, battery ..."
9,5,945,124495,0.089500,0.094093,1.340732,1.209011,"mouse, keyboard, keys, logitech, mouse work, k..."


In [29]:
# Selecting representative documents for each topic

full_representative_topic_documents_dataframe = (
    full_dissatisfaction_topic_review_dataframe[
        [
            "topic_id",
            "topic_strength",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "sampled_review_count",
            "modeling_text_for_vectorizer"
        ]
    ]
    .sort_values(["topic_id", "topic_strength"], ascending = [True, False])
    .groupby("topic_id", group_keys = False)
    .head(5)
    .reset_index(drop = True)
)

display(full_representative_topic_documents_dataframe)

,topic_id,topic_strength,price_band,issue_priority_level,product_display_name,sampled_review_count,modeling_text_for_vectorizer
0,0,0.087883,Very premium,Monitor,NETGEAR Nighthawk X6 Smart Wi-Fi Router (R8000...,250,pile scrap simplest activities router pass dns...
1,0,0.087399,Premium,Priority review,NETGEAR R7500 Nighthawk X4 AC2350 Dual Band Wi...,108,constantly reset terrible router buy open sent...
2,0,0.087073,Premium,Monitor,NETGEAR Nighthawk X4S Smart WiFi Router (R7800...,250,piece crap mouth owning started continued conn...
3,0,0.086200,Premium,Priority review,NETGEAR Nighthawk 4-Stream AX4 Wi-fi 6 Router ...,250,purchased ago disappointed start constantly di...
4,0,0.085783,Premium,Monitor,NETGEAR Nighthawk X4S Smart WiFi Router (R7800...,250,wireless stopped broadcasting try factory rese...
5,1,0.137643,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,completely dead_on_arrival no_power whatsoever...
6,1,0.136635,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,work ago earbuds cut watching listening music ...
7,1,0.135819,Premium,Priority review,Sony WF-SP800N Truly Wireless Sports In-Ear No...,121,sound_quality earpiece left earpiece try earpi...
8,1,0.135364,Lower mid,Monitor,MEE audio M6 X1 Wired In-Ear Sports Headphones...,250,sounds tin iphone hear kind awkward secure ear...
9,1,0.134971,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,140,expecting charge case average size compared no...


In [30]:
# Reviewing topic distribution

topic_segment_distribution_dataframe = (
    full_dissatisfaction_topic_review_dataframe
    .groupby(["topic_id", "price_band", "issue_priority_level"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean")
    )
    .reset_index()
    .sort_values(["topic_id", "price_band", "issue_priority_level"])
)

display(topic_segment_distribution_dataframe)

,topic_id,price_band,issue_priority_level,document_count,total_sampled_reviews,average_topic_strength
0,0,Budget,Monitor,33,4398,0.039279
1,0,Budget,Priority review,17,3800,0.063073
2,0,Lower mid,Monitor,94,12647,0.041572
3,0,Lower mid,Priority review,44,6897,0.063079
4,0,Premium,Monitor,208,31662,0.066698
5,0,Premium,Priority review,108,16848,0.069171
6,0,Upper mid,Monitor,163,24033,0.059010
7,0,Upper mid,Priority review,131,20960,0.063930
8,0,Very premium,Monitor,32,4775,0.073187
9,1,Budget,Monitor,194,22368,0.075236


In [31]:
# Measuring how text clipping affects document preparation

clipping_impact_summary_dataframe = pd.DataFrame({
    "metric": [
        "full_document_count",
        "documents_with_clipping_applied",
        "clipping_applied_ratio",
        "average_raw_word_count",
        "average_clean_word_count_preclip",
        "average_clean_word_count_postclip",
        "average_modeling_token_count",
        "average_sampled_review_count"
    ],
    "value": [
        len(full_corpus_dataframe),
        full_corpus_dataframe["clipping_applied"].sum(),
        full_corpus_dataframe["clipping_applied"].mean(),
        full_corpus_dataframe["raw_word_count"].mean(),
        full_corpus_dataframe["clean_word_count_preclip"].mean(),
        full_corpus_dataframe["clean_word_count_postclip"].mean(),
        full_corpus_dataframe["modeling_token_count"].mean(),
        full_corpus_dataframe["sampled_review_count"].mean()
    ]
})

clipped_vs_unclipped_summary_dataframe = (
    full_corpus_dataframe
    .groupby("clipping_applied", dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum"),
        average_raw_word_count = ("raw_word_count", "mean"),
        average_clean_word_count_preclip = ("clean_word_count_preclip", "mean"),
        average_clean_word_count_postclip = ("clean_word_count_postclip", "mean"),
        average_modeling_token_count = ("modeling_token_count", "mean"),
        average_aggregated_text_length = ("aggregated_text_length", "mean")
    )
    .reset_index()
)

display(clipping_impact_summary_dataframe)
display(clipped_vs_unclipped_summary_dataframe)

,metric,value
0,full_document_count,35213.000000
1,documents_with_clipping_applied,10820.000000
2,clipping_applied_ratio,0.307273
3,average_raw_word_count,5626.417090
4,average_clean_word_count_preclip,5110.421180
5,average_clean_word_count_postclip,3686.933576
6,average_modeling_token_count,1300.040326
7,average_sampled_review_count,112.978133


,clipping_applied,document_count,total_sampled_reviews,average_raw_word_count,average_clean_word_count_preclip,average_clean_word_count_postclip,average_modeling_token_count,average_aggregated_text_length
0,False,24393,1563824,3236.758209,2940.436191,2940.436150,1043.963678,17165.49301
1,True,10820,2414475,11013.750647,10002.513956,5369.864418,1877.348799,58481.566451


In [32]:
# Applying the topic model to comparison documents for cohort analysis

full_comparison_dataframe = full_corpus_modeling_dataframe.loc[
    ~full_corpus_modeling_dataframe["is_dissatisfaction"]
].copy()

full_comparison_tfidf_matrix = full_tfidf_vectorizer.transform(
    full_comparison_dataframe["modeling_text_for_vectorizer"]
)

full_comparison_topic_matrix = selected_nmf_model.transform(full_comparison_tfidf_matrix)

full_comparison_dataframe = full_comparison_dataframe.reset_index(drop = True)
full_comparison_dataframe["topic_id"] = full_comparison_topic_matrix.argmax(axis = 1)
full_comparison_dataframe["topic_strength"] = full_comparison_topic_matrix.max(axis = 1)

full_topic_assignment_dataframe = pd.concat(
    [
        full_dissatisfaction_topic_review_dataframe.assign(cohort_role = "Dissatisfaction"),
        full_comparison_dataframe.assign(cohort_role = "Satisfaction")
    ],
    ignore_index = True
)

print(f"Combined topic assignment rows : {len(full_topic_assignment_dataframe):,}")
print(full_topic_assignment_dataframe["cohort_role"].value_counts(dropna = False))

Combined topic assignment rows : 35,213
cohort_role
Satisfaction       18360
Dissatisfaction    16853
Name: count, dtype: int64


In [33]:
# Preparing topic labels and top-term mappings for interpretation

topic_top_terms_map = {}

for topic_index, topic_weights in enumerate(selected_topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : 15]
    topic_top_terms_map[topic_index] = selected_feature_names[top_term_indices].tolist()

topic_overlap_rows = []

for left_topic_id in sorted(topic_top_terms_map.keys()):
    left_terms = set(topic_top_terms_map[left_topic_id])

    overlap_scores = []
    for right_topic_id in sorted(topic_top_terms_map.keys()):
        if left_topic_id == right_topic_id:
            continue

        right_terms = set(topic_top_terms_map[right_topic_id])
        overlap_score = len(left_terms.intersection(right_terms)) / len(left_terms.union(right_terms))
        overlap_scores.append(overlap_score)

    topic_overlap_rows.append({
        "topic_id": left_topic_id,
        "max_top_term_jaccard_overlap": max(overlap_scores) if overlap_scores else 0.0,
        "mean_top_term_jaccard_overlap": float(np.mean(overlap_scores)) if overlap_scores else 0.0
    })

topic_overlap_summary_dataframe = pd.DataFrame(topic_overlap_rows)

cohort_topic_summary_dataframe = (
    full_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean")
    )
    .reset_index()
)

cohort_totals_dataframe = (
    full_topic_assignment_dataframe
    .groupby("cohort_role", dropna = False)
    .agg(
        total_documents = ("document_id", "count"),
        total_sampled_reviews_cohort = ("sampled_review_count", "sum")
    )
    .reset_index()
)

cohort_topic_summary_dataframe = cohort_topic_summary_dataframe.merge(
    cohort_totals_dataframe,
    on = "cohort_role",
    how = "left"
)

cohort_topic_summary_dataframe["document_share"] = (
    cohort_topic_summary_dataframe["document_count"] /
    cohort_topic_summary_dataframe["total_documents"]
)

cohort_topic_summary_dataframe["sampled_review_share"] = (
    cohort_topic_summary_dataframe["sampled_reviews"] /
    cohort_topic_summary_dataframe["total_sampled_reviews_cohort"]
)

topic_comparison_pivot_dataframe = (
    cohort_topic_summary_dataframe[
        [
            "cohort_role",
            "topic_id",
            "document_count",
            "sampled_reviews",
            "document_share",
            "sampled_review_share",
            "average_topic_strength"
        ]
    ]
    .pivot(index = "topic_id", columns = "cohort_role")
)

topic_comparison_pivot_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in topic_comparison_pivot_dataframe.columns
]

topic_comparison_pivot_dataframe = topic_comparison_pivot_dataframe.reset_index()

topic_comparison_pivot_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] = (
    (topic_comparison_pivot_dataframe["document_share_dissatisfaction"] + 1e-6) /
    (topic_comparison_pivot_dataframe["document_share_satisfaction"] + 1e-6)
)

topic_comparison_pivot_dataframe["sampled_review_share_lift_dissatisfaction_vs_satisfaction"] = (
    (topic_comparison_pivot_dataframe["sampled_review_share_dissatisfaction"] + 1e-6) /
    (topic_comparison_pivot_dataframe["sampled_review_share_satisfaction"] + 1e-6)
)

topic_comparison_pivot_dataframe["document_share_gap"] = (
    topic_comparison_pivot_dataframe["document_share_dissatisfaction"] -
    topic_comparison_pivot_dataframe["document_share_satisfaction"]
)

topic_comparison_pivot_dataframe["top_terms"] = topic_comparison_pivot_dataframe["topic_id"].map(
    lambda topic_id: ", ".join(topic_top_terms_map[topic_id])
)

topic_comparison_summary_dataframe = (
    topic_comparison_pivot_dataframe
    .merge(topic_overlap_summary_dataframe, on = "topic_id", how = "left")
    .sort_values(
        by = ["document_share_lift_dissatisfaction_vs_satisfaction", "document_share_gap"],
        ascending = [False, False]
    )
    .reset_index(drop = True)
)

display(topic_comparison_summary_dataframe)

,topic_id,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,document_share_dissatisfaction,document_share_satisfaction,sampled_review_share_dissatisfaction,sampled_review_share_satisfaction,average_topic_strength_dissatisfaction,average_topic_strength_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,sampled_review_share_lift_dissatisfaction_vs_satisfaction,document_share_gap,top_terms,max_top_term_jaccard_overlap,mean_top_term_jaccard_overlap
0,12,1060,740,109780,60861,0.062897,0.040305,0.055073,0.030661,0.071391,0.064315,1.560507,1.796167,0.022592,"charger, charge battery, hold charge, battery ...",0.111111,0.007937
1,3,1231,1171,162709,130872,0.073043,0.063780,0.081626,0.065932,0.086244,0.068559,1.145238,1.238034,0.009263,"charge phone, cords, chargers, charger, not_ch...",0.111111,0.012863
2,5,945,933,124495,110320,0.056073,0.050817,0.062455,0.055578,0.089500,0.077449,1.103430,1.123739,0.005256,"mouse, keyboard, keys, logitech, mouse work, k...",0.000000,0.000000
3,0,830,822,126020,113852,0.049249,0.044771,0.06322,0.057357,0.061121,0.053412,1.100021,1.102217,0.004478,"router, wifi, network, internet, modem, netgea...",0.034483,0.004926
4,1,1937,1926,285262,253887,0.114935,0.104902,0.143107,0.127906,0.087805,0.088255,1.095641,1.11885,0.010033,"ear, headphone, earbuds, ears, buds, earbud, s...",0.071429,0.005102
5,11,915,923,116085,103688,0.054293,0.050272,0.058236,0.052237,0.077447,0.070782,1.079976,1.114848,0.004021,"remote, remote work, remotes, buttons, remote ...",0.000000,0.000000
6,10,507,515,51465,49783,0.030084,0.028050,0.025818,0.02508,0.115678,0.096046,1.072495,1.029436,0.002034,"bubbles, screen_protector, protectors, screen ...",0.000000,0.000000
7,4,1545,1571,183129,171371,0.091675,0.085566,0.09187,0.086335,0.075803,0.063656,1.071390,1.064114,0.006109,"drive, drives, files, disk, data, hard_drive, ...",0.000000,0.000000
8,9,1109,1163,116610,116659,0.065804,0.063344,0.0585,0.058772,0.075755,0.068694,1.038836,0.995374,0.002460,"hdmi, monitor, hdmi cable, display, adapter, p...",0.034483,0.007389
9,13,558,593,66470,67822,0.033110,0.032298,0.033346,0.034168,0.100459,0.096746,1.025120,0.975943,0.000811,"antenna, stations, reception, channels, miles,...",0.034483,0.007389


In [34]:
# Comparing topic exposure across cohorts, price bands, and priority groups

topic_segment_comparison_dataframe = (
    full_topic_assignment_dataframe
    .groupby(
        ["cohort_role", "topic_id", "price_band", "issue_priority_level"],
        dropna = False
    )
    .agg(
        document_count = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean")
    )
    .reset_index()
)

display(topic_segment_comparison_dataframe.sort_values(
    ["topic_id", "cohort_role", "price_band", "issue_priority_level"]
))

full_representative_documents_both_cohorts_dataframe = (
    full_topic_assignment_dataframe[
        [
            "cohort_role",
            "topic_id",
            "topic_strength",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "sampled_review_count",
            "modeling_text_for_vectorizer"
        ]
    ]
    .sort_values(
        ["cohort_role", "topic_id", "topic_strength"],
        ascending = [True, True, False]
    )
    .groupby(["cohort_role", "topic_id"], group_keys = False)
    .head(3)
    .reset_index(drop = True)
)

display(full_representative_documents_both_cohorts_dataframe)

,cohort_role,topic_id,price_band,issue_priority_level,document_count,total_sampled_reviews,average_topic_strength
0,Dissatisfaction,0,Budget,Monitor,33,4398,0.039279
1,Dissatisfaction,0,Budget,Priority review,17,3800,0.063073
2,Dissatisfaction,0,Lower mid,Monitor,94,12647,0.041572
3,Dissatisfaction,0,Lower mid,Priority review,44,6897,0.063079
4,Dissatisfaction,0,Premium,Monitor,208,31662,0.066698
5,Dissatisfaction,0,Premium,Priority review,108,16848,0.069171
6,Dissatisfaction,0,Upper mid,Monitor,163,24033,0.059010
7,Dissatisfaction,0,Upper mid,Priority review,131,20960,0.063930
8,Dissatisfaction,0,Very premium,Monitor,32,4775,0.073187
134,Satisfaction,0,Budget,Monitor,78,9898,0.032675


,cohort_role,topic_id,topic_strength,price_band,issue_priority_level,product_display_name,sampled_review_count,modeling_text_for_vectorizer
0,Dissatisfaction,0,0.087883,Very premium,Monitor,NETGEAR Nighthawk X6 Smart Wi-Fi Router (R8000...,250,pile scrap simplest activities router pass dns...
1,Dissatisfaction,0,0.087399,Premium,Priority review,NETGEAR R7500 Nighthawk X4 AC2350 Dual Band Wi...,108,constantly reset terrible router buy open sent...
2,Dissatisfaction,0,0.087073,Premium,Monitor,NETGEAR Nighthawk X4S Smart WiFi Router (R7800...,250,piece crap mouth owning started continued conn...
3,Dissatisfaction,1,0.137643,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,completely dead_on_arrival no_power whatsoever...
4,Dissatisfaction,1,0.136635,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,work ago earbuds cut watching listening music ...
5,Dissatisfaction,1,0.135819,Premium,Priority review,Sony WF-SP800N Truly Wireless Sports In-Ear No...,121,sound_quality earpiece left earpiece try earpi...
6,Dissatisfaction,2,0.138012,Budget,Monitor,MoKo Case Fit 2018/2017 iPad 9.7 6th/5th Gener...,250,case title misleading not_fit ipad release cou...
7,Dissatisfaction,2,0.137992,Lower mid,Monitor,JETech Case for iPad Air 1st Edition (NOT for ...,250,hole camera lens line feel case crazy flaw tip...
8,Dissatisfaction,2,0.137101,Budget,Monitor,"Fintie Case for iPad Air (3rd Gen) 10.5"" 2019 ...",141,generation ipad fit frustrating cool case leav...
9,Dissatisfaction,3,0.150794,Budget,Monitor,JSAUX USB-C to USB A Cable 3.1A Fast Charging ...,250,cord randomly stopped_working super disappoint...


In [35]:
# Calculating topic assignment confidence for each document

def compute_assignment_margin(topic_matrix):
    sorted_strengths = np.sort(topic_matrix, axis = 1)
    max_strength = sorted_strengths[ : , -1]
    second_strength = sorted_strengths[ : , -2]
    return max_strength - second_strength


full_dissatisfaction_topic_review_dataframe["topic_margin"] = compute_assignment_margin(
    selected_document_topic_matrix
)

full_comparison_dataframe["topic_margin"] = compute_assignment_margin(
    full_comparison_topic_matrix
)

assignment_confidence_summary_dataframe = (
    pd.concat(
        [
            full_dissatisfaction_topic_review_dataframe.assign(cohort_role = "Dissatisfaction"),
            full_comparison_dataframe.assign(cohort_role = "Satisfaction")
        ],
        ignore_index = True
    )
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        average_topic_strength = ("topic_strength", "mean"),
        median_topic_strength = ("topic_strength", "median"),
        average_topic_margin = ("topic_margin", "mean"),
        median_topic_margin = ("topic_margin", "median"),
        low_margin_document_count = ("topic_margin", lambda values: (values < 0.015).sum())
    )
    .reset_index()
    .sort_values(["cohort_role", "topic_id"])
)

display(assignment_confidence_summary_dataframe)

,cohort_role,topic_id,document_count,average_topic_strength,median_topic_strength,average_topic_margin,median_topic_margin,low_margin_document_count
0,Dissatisfaction,0,830,0.061121,0.066148,0.048389,0.055147,126
1,Dissatisfaction,1,1937,0.087805,0.089475,0.065651,0.067718,160
2,Dissatisfaction,2,1649,0.075462,0.074127,0.055432,0.055434,218
3,Dissatisfaction,3,1231,0.086244,0.089526,0.067411,0.072081,330
4,Dissatisfaction,4,1545,0.075803,0.074010,0.055108,0.044309,313
5,Dissatisfaction,5,945,0.089500,0.094093,0.068815,0.076124,158
6,Dissatisfaction,6,1246,0.069002,0.062515,0.047992,0.036532,300
7,Dissatisfaction,7,1339,0.077219,0.075124,0.053167,0.047307,297
8,Dissatisfaction,8,1508,0.061545,0.061543,0.033555,0.030727,386
9,Dissatisfaction,9,1109,0.075755,0.080711,0.050545,0.053225,258


In [36]:
# Checking whether text clipping is biasing topic assignments

topic_clipping_bias_dataframe = (
    full_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id", "clipping_applied"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

topic_clipping_totals_dataframe = (
    topic_clipping_bias_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        total_document_count = ("document_count", "sum"),
        total_sampled_reviews = ("sampled_reviews", "sum")
    )
    .reset_index()
)

topic_clipping_bias_dataframe = topic_clipping_bias_dataframe.merge(
    topic_clipping_totals_dataframe,
    on = ["cohort_role", "topic_id"],
    how = "left"
)

topic_clipping_bias_dataframe["document_share_within_topic"] = (
    topic_clipping_bias_dataframe["document_count"] /
    topic_clipping_bias_dataframe["total_document_count"]
)

display(
    topic_clipping_bias_dataframe.sort_values(
        ["cohort_role", "topic_id", "clipping_applied"]
    )
)

,cohort_role,topic_id,clipping_applied,document_count,sampled_reviews,total_document_count,total_sampled_reviews,document_share_within_topic
0,Dissatisfaction,0,False,413,29761,830,126020,0.497590
1,Dissatisfaction,0,True,417,96259,830,126020,0.502410
2,Dissatisfaction,1,False,1007,72968,1937,285262,0.519876
3,Dissatisfaction,1,True,930,212294,1937,285262,0.480124
4,Dissatisfaction,2,False,1319,79567,1649,151277,0.799879
5,Dissatisfaction,2,True,330,71710,1649,151277,0.200121
6,Dissatisfaction,3,False,740,53337,1231,162709,0.601137
7,Dissatisfaction,3,True,491,109372,1231,162709,0.398863
8,Dissatisfaction,4,False,1038,69331,1545,183129,0.671845
9,Dissatisfaction,4,True,507,113798,1545,183129,0.328155


In [37]:
# Checking topic segment reliability for reporting use

topic_segment_reporting_eligibility_dataframe = (
    full_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id", "price_band", "issue_priority_level"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
    .pivot(
        index = ["topic_id", "price_band", "issue_priority_level"],
        columns = "cohort_role",
        values = ["document_count", "sampled_reviews"]
    )
)

topic_segment_reporting_eligibility_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in topic_segment_reporting_eligibility_dataframe.columns
]

topic_segment_reporting_eligibility_dataframe = (
    topic_segment_reporting_eligibility_dataframe
    .reset_index()
    .fillna(0)
)

topic_segment_reporting_eligibility_dataframe["is_segment_reporting_eligible"] = (
    (topic_segment_reporting_eligibility_dataframe["document_count_dissatisfaction"] >= 25)
    & (topic_segment_reporting_eligibility_dataframe["document_count_satisfaction"] >= 25)
    & (topic_segment_reporting_eligibility_dataframe["sampled_reviews_dissatisfaction"] >= 3000)
    & (topic_segment_reporting_eligibility_dataframe["sampled_reviews_satisfaction"] >= 3000)
)

display(
    topic_segment_reporting_eligibility_dataframe.sort_values(
        ["topic_id", "price_band", "issue_priority_level"]
    )
)

/var/folders/f1/ws24m5bs3cx9_1d_nclvhd1w0000gn/T/ipykernel_51314/1094106930.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


,topic_id,price_band,issue_priority_level,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,is_segment_reporting_eligible
0,0,Budget,Monitor,33,78,4398,9898,True
1,0,Budget,Priority review,17,21,3800,4831,False
2,0,Lower mid,Monitor,94,87,12647,8530,True
3,0,Lower mid,Priority review,44,52,6897,9354,True
4,0,Premium,Monitor,208,170,31662,21919,True
5,0,Premium,Priority review,108,111,16848,17245,True
6,0,Upper mid,Monitor,163,142,24033,15990,True
7,0,Upper mid,Priority review,131,135,20960,23164,True
8,0,Very premium,Monitor,32,26,4775,2921,False
9,1,Budget,Monitor,194,214,22368,21397,True


In [38]:
# Measuring overall topic weight share across dissatisfied documents

dissatisfaction_topic_weight_share = (
    selected_document_topic_matrix.sum(axis = 0) /
    selected_document_topic_matrix.sum()
)

satisfaction_topic_weight_share = (
    full_comparison_topic_matrix.sum(axis = 0) /
    full_comparison_topic_matrix.sum()
)

soft_topic_comparison_dataframe = pd.DataFrame({
    "topic_id": np.arange(selected_topic_count),
    "topic_weight_share_dissatisfaction": dissatisfaction_topic_weight_share,
    "topic_weight_share_satisfaction": satisfaction_topic_weight_share
})

soft_topic_comparison_dataframe["topic_weight_share_lift_dissatisfaction_vs_satisfaction"] = (
    (soft_topic_comparison_dataframe["topic_weight_share_dissatisfaction"] + 1e-6) /
    (soft_topic_comparison_dataframe["topic_weight_share_satisfaction"] + 1e-6)
)

soft_topic_comparison_dataframe["topic_weight_share_gap"] = (
    soft_topic_comparison_dataframe["topic_weight_share_dissatisfaction"] -
    soft_topic_comparison_dataframe["topic_weight_share_satisfaction"]
)

display(
    soft_topic_comparison_dataframe.sort_values(
        by = "topic_weight_share_lift_dissatisfaction_vs_satisfaction",
        ascending = False
    )
)

,topic_id,topic_weight_share_dissatisfaction,topic_weight_share_satisfaction,topic_weight_share_lift_dissatisfaction_vs_satisfaction,topic_weight_share_gap
12,12,0.082268,0.050882,1.616824,0.031386
3,3,0.076694,0.059047,1.298866,0.017647
0,0,0.043807,0.034378,1.274283,0.009430
4,4,0.079433,0.069354,1.145317,0.010078
10,10,0.048469,0.044053,1.100252,0.004416
11,11,0.060327,0.055149,1.093902,0.005179
5,5,0.059885,0.055270,1.083497,0.004615
9,9,0.069385,0.068057,1.019514,0.001328
1,1,0.089549,0.093391,0.958862,-0.003842
8,8,0.071682,0.080550,0.889906,-0.008868


In [39]:
# Building topic-level driver metrics for business interpretation

topic_driver_core_dataframe = (
    full_topic_summary_dataframe[
        [
            "topic_id",
            "document_count",
            "total_sampled_reviews",
            "average_topic_strength",
            "average_rating",
            "average_helpful_vote",
            "top_terms"
        ]
    ]
    .merge(
        topic_comparison_summary_dataframe[
            [
                "topic_id",
                "document_count_dissatisfaction",
                "document_count_satisfaction",
                "sampled_reviews_dissatisfaction",
                "sampled_reviews_satisfaction",
                "document_share_dissatisfaction",
                "document_share_satisfaction",
                "document_share_lift_dissatisfaction_vs_satisfaction",
                "sampled_review_share_lift_dissatisfaction_vs_satisfaction",
                "document_share_gap",
                "max_top_term_jaccard_overlap"
            ]
        ],
        on = "topic_id",
        how = "left"
    )
    .merge(
        assignment_confidence_summary_dataframe.loc[
            assignment_confidence_summary_dataframe["cohort_role"] == "Dissatisfaction",
            [
                "topic_id",
                "average_topic_margin",
                "median_topic_margin",
                "low_margin_document_count"
            ]
        ],
        on = "topic_id",
        how = "left"
    )
    .merge(
        soft_topic_comparison_dataframe[
            [
                "topic_id",
                "topic_weight_share_dissatisfaction",
                "topic_weight_share_satisfaction",
                "topic_weight_share_lift_dissatisfaction_vs_satisfaction",
                "topic_weight_share_gap"
            ]
        ],
        on = "topic_id",
        how = "left"
    )
)

topic_driver_core_dataframe["low_margin_document_ratio"] = (
    topic_driver_core_dataframe["low_margin_document_count"] /
    topic_driver_core_dataframe["document_count"].replace(0, np.nan)
)

display(
    topic_driver_core_dataframe.sort_values(
        by = [
            "document_share_lift_dissatisfaction_vs_satisfaction",
            "topic_weight_share_lift_dissatisfaction_vs_satisfaction",
            "document_count"
        ],
        ascending = [False, False, False]
    )
)

,topic_id,document_count,total_sampled_reviews,average_topic_strength,average_rating,average_helpful_vote,top_terms,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,document_share_dissatisfaction,document_share_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,sampled_review_share_lift_dissatisfaction_vs_satisfaction,document_share_gap,max_top_term_jaccard_overlap,average_topic_margin,median_topic_margin,low_margin_document_count,topic_weight_share_dissatisfaction,topic_weight_share_satisfaction,topic_weight_share_lift_dissatisfaction_vs_satisfaction,topic_weight_share_gap,low_margin_document_ratio
8,12,1060,109780,0.071391,1.256757,2.432281,"charger, charge battery, hold charge, battery ...",1060,740,109780,60861,0.062897,0.040305,1.560507,1.796167,0.022592,0.111111,0.048695,0.023479,391,0.082268,0.050882,1.616824,0.031386,0.368868
6,3,1231,162709,0.086244,1.256194,1.215423,"charge phone, cords, chargers, charger, not_ch...",1231,1171,162709,130872,0.073043,0.063780,1.145238,1.238034,0.009263,0.111111,0.067411,0.072081,330,0.076694,0.059047,1.298866,0.017647,0.268075
9,5,945,124495,0.089500,1.340732,1.209011,"mouse, keyboard, keys, logitech, mouse work, k...",945,933,124495,110320,0.056073,0.050817,1.103430,1.123739,0.005256,0.000000,0.068815,0.076124,158,0.059885,0.055270,1.083497,0.004615,0.167196
11,0,830,126020,0.061121,1.267146,2.398637,"router, wifi, network, internet, modem, netgea...",830,822,126020,113852,0.049249,0.044771,1.100021,1.102217,0.004478,0.034483,0.048389,0.055147,126,0.043807,0.034378,1.274283,0.009430,0.151807
0,1,1937,285262,0.087805,1.358950,1.290585,"ear, headphone, earbuds, ears, buds, earbud, s...",1937,1926,285262,253887,0.114935,0.104902,1.095641,1.11885,0.010033,0.071429,0.065651,0.067718,160,0.089549,0.093391,0.958862,-0.003842,0.082602
10,11,915,116085,0.077447,1.276203,2.052309,"remote, remote work, remotes, buttons, remote ...",915,923,116085,103688,0.054293,0.050272,1.079976,1.114848,0.004021,0.000000,0.055333,0.058762,224,0.060327,0.055149,1.093902,0.005179,0.244809
13,10,507,51465,0.115678,1.300156,1.057393,"bubbles, screen_protector, protectors, screen ...",507,515,51465,49783,0.030084,0.028050,1.072495,1.029436,0.002034,0.000000,0.101886,0.128618,90,0.048469,0.044053,1.100252,0.004416,0.177515
2,4,1545,183129,0.075803,1.231458,2.416205,"drive, drives, files, disk, data, hard_drive, ...",1545,1571,183129,171371,0.091675,0.085566,1.071390,1.064114,0.006109,0.000000,0.055108,0.044309,313,0.079433,0.069354,1.145317,0.010078,0.202589
7,9,1109,116610,0.075755,1.261656,1.617113,"hdmi, monitor, hdmi cable, display, adapter, p...",1109,1163,116610,116659,0.065804,0.063344,1.038836,0.995374,0.002460,0.034483,0.050545,0.053225,258,0.069385,0.068057,1.019514,0.001328,0.232642
12,13,558,66470,0.100459,1.314494,2.045112,"antenna, stations, reception, channels, miles,...",558,593,66470,67822,0.033110,0.032298,1.025120,0.975943,0.000811,0.034483,0.073351,0.044054,137,0.046565,0.052410,0.888472,-0.005845,0.245520


In [40]:
# Reviewing dissatisfied topic distribution across price and priority segments
dissatisfaction_topic_segment_distribution_dataframe = (
    full_dissatisfaction_topic_review_dataframe
    .groupby(["topic_id", "price_band", "issue_priority_level"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

dissatisfaction_topic_segment_distribution_dataframe["segment_document_share_within_topic"] = (
    dissatisfaction_topic_segment_distribution_dataframe["document_count"] /
    dissatisfaction_topic_segment_distribution_dataframe.groupby("topic_id")["document_count"].transform("sum")
)

topic_breadth_summary_dataframe = (
    dissatisfaction_topic_segment_distribution_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        covered_segment_count = ("segment_document_share_within_topic", "count"),
        segment_count_above_05_share = (
            "segment_document_share_within_topic",
            lambda values: (values >= 0.05).sum()
        ),
        segment_count_above_10_share = (
            "segment_document_share_within_topic",
            lambda values: (values >= 0.10).sum()
        ),
        max_segment_share = ("segment_document_share_within_topic", "max"),
        concentration_hhi = (
            "segment_document_share_within_topic",
            lambda values: np.square(values).sum()
        )
    )
    .reset_index()
)

display(
    topic_breadth_summary_dataframe.sort_values(
        by = ["concentration_hhi", "max_segment_share"],
        ascending = [False, False]
    )
)

,topic_id,covered_segment_count,segment_count_above_05_share,segment_count_above_10_share,max_segment_share,concentration_hhi
10,10,8,3,3,0.568047,0.380935
3,3,9,4,3,0.465475,0.304157
2,2,9,4,3,0.402668,0.266621
6,6,9,4,4,0.303371,0.209671
7,7,9,6,4,0.296490,0.182711
4,4,9,6,4,0.209709,0.163492
0,0,9,6,5,0.250602,0.162334
12,12,9,7,4,0.240566,0.160048
8,8,9,6,4,0.234748,0.159123
9,9,9,7,4,0.259693,0.156795


In [41]:
# Comparing topic exposure by price band across customer cohorts

topic_price_band_comparison_dataframe = (
    full_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id", "price_band"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

topic_price_band_totals_dataframe = (
    topic_price_band_comparison_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        total_documents = ("document_count", "sum"),
        total_sampled_reviews = ("sampled_reviews", "sum")
    )
    .reset_index()
)

topic_price_band_comparison_dataframe = topic_price_band_comparison_dataframe.merge(
    topic_price_band_totals_dataframe,
    on = ["cohort_role", "topic_id"],
    how = "left"
)

topic_price_band_comparison_dataframe["document_share_within_topic"] = (
    topic_price_band_comparison_dataframe["document_count"] /
    topic_price_band_comparison_dataframe["total_documents"].replace(0, np.nan)
)

topic_price_band_comparison_dataframe = (
    topic_price_band_comparison_dataframe
    .pivot_table(
        index = ["topic_id", "price_band"],
        columns = "cohort_role",
        values = ["document_count", "sampled_reviews", "document_share_within_topic"],
        fill_value = 0
    )
)

topic_price_band_comparison_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in topic_price_band_comparison_dataframe.columns
]

topic_price_band_comparison_dataframe = (
    topic_price_band_comparison_dataframe
    .reset_index()
    .infer_objects(copy = False)
)

topic_price_band_comparison_dataframe["price_band_document_share_lift_dissatisfaction_vs_satisfaction"] = (
    (topic_price_band_comparison_dataframe["document_share_within_topic_dissatisfaction"] + 1e-6) /
    (topic_price_band_comparison_dataframe["document_share_within_topic_satisfaction"] + 1e-6)
)

display(
    topic_price_band_comparison_dataframe.sort_values(
        by = ["topic_id", "price_band"]
    )
)

,topic_id,price_band,document_count_dissatisfaction,document_count_satisfaction,document_share_within_topic_dissatisfaction,document_share_within_topic_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,price_band_document_share_lift_dissatisfaction_vs_satisfaction
0,0,Budget,50.0,99.0,0.060241,0.120438,8198.0,14729.0,0.500187
1,0,Lower mid,138.0,139.0,0.166265,0.169100,19544.0,17884.0,0.983237
2,0,Premium,316.0,281.0,0.380723,0.341849,48510.0,39164.0,1.113716
3,0,Upper mid,294.0,277.0,0.354217,0.336983,44993.0,39154.0,1.051142
4,0,Very premium,32.0,26.0,0.038554,0.031630,4775.0,2921.0,1.218899
5,1,Budget,356.0,369.0,0.183789,0.191589,47971.0,42863.0,0.959291
6,1,Lower mid,560.0,533.0,0.289107,0.276739,87316.0,74121.0,1.044690
7,1,Premium,404.0,405.0,0.208570,0.210280,63138.0,56583.0,0.991866
8,1,Upper mid,574.0,573.0,0.296335,0.297508,82519.0,76212.0,0.996056
9,1,Very premium,43.0,46.0,0.022199,0.023884,4318.0,4108.0,0.929477


In [42]:
# Comparing topic exposure by product priority level across customer cohorts

topic_priority_comparison_dataframe = (
    full_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id", "issue_priority_level"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

topic_priority_totals_dataframe = (
    topic_priority_comparison_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        total_documents = ("document_count", "sum")
    )
    .reset_index()
)

topic_priority_comparison_dataframe = topic_priority_comparison_dataframe.merge(
    topic_priority_totals_dataframe,
    on = ["cohort_role", "topic_id"],
    how = "left"
)

topic_priority_comparison_dataframe["document_share_within_topic"] = (
    topic_priority_comparison_dataframe["document_count"] /
    topic_priority_comparison_dataframe["total_documents"].replace(0, np.nan)
)

topic_priority_comparison_dataframe = (
    topic_priority_comparison_dataframe
    .pivot_table(
        index = ["topic_id", "issue_priority_level"],
        columns = "cohort_role",
        values = ["document_count", "sampled_reviews", "document_share_within_topic"],
        fill_value = 0
    )
)

topic_priority_comparison_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in topic_priority_comparison_dataframe.columns
]

topic_priority_comparison_dataframe = (
    topic_priority_comparison_dataframe
    .reset_index()
    .infer_objects(copy = False)
)

display(
    topic_priority_comparison_dataframe.sort_values(
        by = ["topic_id", "issue_priority_level"]
    )
)

,topic_id,issue_priority_level,document_count_dissatisfaction,document_count_satisfaction,document_share_within_topic_dissatisfaction,document_share_within_topic_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction
0,0,Monitor,530.0,503.0,0.638554,0.611922,77515.0,59258.0
1,0,Priority review,300.0,319.0,0.361446,0.388078,48505.0,54594.0
2,1,Monitor,1195.0,1177.0,0.616933,0.611111,163411.0,138068.0
3,1,Priority review,742.0,749.0,0.383067,0.388889,121851.0,115819.0
4,2,Monitor,1432.0,2021.0,0.868405,0.891093,120421.0,180242.0
5,2,Priority review,217.0,247.0,0.131595,0.108907,30856.0,35914.0
6,3,Monitor,877.0,862.0,0.712429,0.736123,104886.0,83308.0
7,3,Priority review,354.0,309.0,0.287571,0.263877,57823.0,47564.0
8,4,Monitor,1296.0,1328.0,0.838835,0.845321,148119.0,138454.0
9,4,Priority review,249.0,243.0,0.161165,0.154679,35010.0,32917.0


In [43]:
# Extracting frequent phrases from the full Electronics corpus

full_phrase_vectorizer = CountVectorizer(
    lowercase = False,
    stop_words = None,
    token_pattern = r"(?u)\b[a-z_][a-z0-9_]{2,}\b",
    ngram_range = (2, 3),
    min_df = 20,
    max_df = 0.20,
    max_features = 40000
)

full_phrase_matrix = full_phrase_vectorizer.fit_transform(
    full_topic_assignment_dataframe["modeling_text_for_vectorizer"]
)

full_phrase_names = np.array(full_phrase_vectorizer.get_feature_names_out())

topic_phrase_comparison_rows = []

for topic_id in sorted(full_topic_assignment_dataframe["topic_id"].unique()):
    topic_mask = full_topic_assignment_dataframe["topic_id"].eq(topic_id).to_numpy()
    dissatisfaction_mask = topic_mask & full_topic_assignment_dataframe["cohort_role"].eq("Dissatisfaction").to_numpy()
    satisfaction_mask = topic_mask & full_topic_assignment_dataframe["cohort_role"].eq("Satisfaction").to_numpy()

    dissatisfaction_document_count = dissatisfaction_mask.sum()
    satisfaction_document_count = satisfaction_mask.sum()

    if dissatisfaction_document_count == 0 or satisfaction_document_count == 0:
        continue

    dissatisfaction_document_frequency = np.asarray(
        (full_phrase_matrix[dissatisfaction_mask] > 0).sum(axis = 0)
    ).ravel()

    satisfaction_document_frequency = np.asarray(
        (full_phrase_matrix[satisfaction_mask] > 0).sum(axis = 0)
    ).ravel()

    topic_phrase_dataframe = pd.DataFrame({
        "topic_id": topic_id,
        "phrase_token": full_phrase_names,
        "dissatisfaction_document_frequency": dissatisfaction_document_frequency,
        "satisfaction_document_frequency": satisfaction_document_frequency
    })

    topic_phrase_dataframe["combined_document_frequency"] = (
        topic_phrase_dataframe["dissatisfaction_document_frequency"] +
        topic_phrase_dataframe["satisfaction_document_frequency"]
    )

    topic_phrase_dataframe["dissatisfaction_document_ratio"] = (
        topic_phrase_dataframe["dissatisfaction_document_frequency"] /
        dissatisfaction_document_count
    )

    topic_phrase_dataframe["satisfaction_document_ratio"] = (
        topic_phrase_dataframe["satisfaction_document_frequency"] /
        satisfaction_document_count
    )

    topic_phrase_dataframe["phrase_ratio_lift"] = (
        (topic_phrase_dataframe["dissatisfaction_document_ratio"] + 1e-6) /
        (topic_phrase_dataframe["satisfaction_document_ratio"] + 1e-6)
    )

    topic_phrase_dataframe["phrase_ratio_gap"] = (
        topic_phrase_dataframe["dissatisfaction_document_ratio"] -
        topic_phrase_dataframe["satisfaction_document_ratio"]
    )

    topic_phrase_comparison_rows.append(topic_phrase_dataframe)

topic_phrase_comparison_dataframe = pd.concat(
    topic_phrase_comparison_rows,
    ignore_index = True
)

accepted_topic_phrase_comparison_dataframe = (
    topic_phrase_comparison_dataframe
    .query(
        "dissatisfaction_document_frequency >= 20 "
        "and combined_document_frequency >= 30 "
        "and phrase_ratio_gap >= 0.015"
    )
    .sort_values(
        by = ["topic_id", "phrase_ratio_lift", "phrase_ratio_gap", "dissatisfaction_document_frequency"],
        ascending = [True, False, False, False]
    )
    .reset_index(drop = True)
)

accepted_topic_phrase_summary_dataframe = (
    accepted_topic_phrase_comparison_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        accepted_phrase_count = ("phrase_token", "count"),
        top_accepted_phrases = ("phrase_token", lambda values: ", ".join(values.head(12)))
    )
    .reset_index()
)

display(accepted_topic_phrase_comparison_dataframe.head(120))
display(accepted_topic_phrase_summary_dataframe.sort_values("topic_id"))

,topic_id,phrase_token,dissatisfaction_document_frequency,satisfaction_document_frequency,combined_document_frequency,dissatisfaction_document_ratio,satisfaction_document_ratio,phrase_ratio_lift,phrase_ratio_gap
0,0,outside return_window,117,0,117,0.140964,0.0,140964.855422,0.140964
1,0,past return_window,107,0,107,0.128916,0.0,128916.662651,0.128916
2,0,complete waste,102,0,102,0.122892,0.0,122892.566265,0.122892
3,0,amazon return_window,98,0,98,0.118072,0.0,118073.289157,0.118072
4,0,stopped_working completely,98,0,98,0.118072,0.0,118073.289157,0.118072
5,0,piece trash,80,0,80,0.096386,0.0,96386.542169,0.096386
6,0,return waste,77,0,77,0.092771,0.0,92772.084337,0.092771
7,0,unable return,77,0,77,0.092771,0.0,92772.084337,0.092771
8,0,constantly loses,76,0,76,0.091566,0.0,91567.265060,0.091566
9,0,tech_support useless,76,0,76,0.091566,0.0,91567.265060,0.091566


,topic_id,accepted_phrase_count,top_accepted_phrases
0,0,3284,"outside return_window, past return_window, com..."
1,1,3711,"work waste_money, return waste, return not_wor..."
2,2,1781,"total waste_money, waste_money case, case poor..."
3,3,2605,"stopped_working waste_money, return try, retur..."
4,4,3223,"outside return_window, try multiple computers,..."
5,5,2316,"stopped_working disappointed, disappointed ret..."
6,6,1175,"complete waste_money, super disappointed, retu..."
7,7,3272,"customer_service horrible, does_not_work adver..."
8,8,2953,"return immediately, waste work, return waste, ..."
9,9,2453,"does_not_work advertised, waste bought, return..."


In [44]:
# Grouping frequent phrases into interpretable business signal families

phrase_family_definitions = {
    "transaction_or_remedy": {
        "return", "return_window", "return_policy", "refund", "restocking",
        "warranty", "buyer", "buyers", "beware", "false_advertising",
        "false_advertisement", "exchange"
    },
    "delivery_or_marketplace": {
        "late_delivery", "on_time_delivery", "shipping", "ship", "shipped",
        "arrived", "arrive", "arrival", "arrived_damaged", "wrong_item",
        "package", "packaging", "box", "seller", "amazon",
        "customer_service", "tech_support"
    },
    "product_failure_or_breakdown": {
        "does_not_work", "did_not_work", "stopped_working", "dead_on_arrival",
        "not_charge", "not_charging", "not_connect", "not_connecting",
        "not_pair", "not_pairing", "not_turn_on", "will_not_turn_on",
        "no_power", "no_sound", "no_signal", "battery_died",
        "defective_unit", "defective", "broken", "broke"
    },
    "compatibility_or_fit": {
        "not_compatible", "not_fit", "compatible", "adapter", "hdmi",
        "screen_protector", "usb_c", "usb_a", "micro_sd", "sd_card", "case"
    },
    "setup_or_connectivity": {
        "connect", "disconnect", "keeps_disconnecting", "wifi", "bluetooth",
        "pair", "setup", "router", "network", "internet", "modem",
        "signal", "reception", "channels"
    },
    "quality_or_durability": {
        "poor_quality", "quality", "cheap", "junk", "garbage", "trash",
        "cracked", "cracking", "bubbles", "flimsy", "strap", "zipper",
        "screws", "mount", "mounting"
    },
    "audio_or_visual_performance": {
        "sound_quality", "picture_quality", "audio", "speaker", "speakers",
        "earbuds", "headphone", "headphones", "bass", "display", "monitor",
        "camera", "recording", "motion"
    }
}


def phrase_contains_family_signal(phrase_value, family_terms):
    phrase_tokens = set(str(phrase_value).split())
    return len(phrase_tokens.intersection(family_terms)) > 0


topic_phrase_family_rows = []

for topic_id in sorted(accepted_topic_phrase_comparison_dataframe["topic_id"].unique()):
    topic_phrase_subset_dataframe = accepted_topic_phrase_comparison_dataframe.loc[
        accepted_topic_phrase_comparison_dataframe["topic_id"] == topic_id
    ].copy()

    topic_result = {
        "topic_id": topic_id,
        "accepted_phrase_count": len(topic_phrase_subset_dataframe)
    }

    total_gap_score = topic_phrase_subset_dataframe["phrase_ratio_gap"].sum()

    for family_name, family_terms in phrase_family_definitions.items():
        family_mask = topic_phrase_subset_dataframe["phrase_token"].apply(
            lambda value: phrase_contains_family_signal(value, family_terms)
        )

        family_phrase_count = family_mask.sum()
        family_gap_score = topic_phrase_subset_dataframe.loc[
            family_mask,
            "phrase_ratio_gap"
        ].sum()

        topic_result[f"{family_name}_phrase_count"] = family_phrase_count
        topic_result[f"{family_name}_gap_score"] = family_gap_score
        topic_result[f"{family_name}_gap_share"] = (
            family_gap_score / total_gap_score if total_gap_score > 0 else 0.0
        )

    family_share_columns = [
        column_name
        for column_name in topic_result.keys()
        if column_name.endswith("_gap_share")
    ]

    strongest_family_column = max(
        family_share_columns,
        key = lambda column_name: topic_result[column_name]
    )

    strongest_family_share = topic_result[strongest_family_column]

    if strongest_family_share >= 0.40:
        dominant_driver_family = strongest_family_column.replace("_gap_share", "")
    else:
        dominant_driver_family = "Mixed"

    topic_result["dominant_driver_family"] = dominant_driver_family
    topic_result["dominant_driver_family_share"] = strongest_family_share
    topic_result["total_phrase_gap_score"] = total_gap_score

    topic_phrase_family_rows.append(topic_result)

topic_phrase_family_summary_dataframe = pd.DataFrame(topic_phrase_family_rows)

display(topic_phrase_family_summary_dataframe.sort_values("topic_id"))

,topic_id,accepted_phrase_count,transaction_or_remedy_phrase_count,transaction_or_remedy_gap_score,transaction_or_remedy_gap_share,delivery_or_marketplace_phrase_count,delivery_or_marketplace_gap_score,delivery_or_marketplace_gap_share,product_failure_or_breakdown_phrase_count,product_failure_or_breakdown_gap_score,product_failure_or_breakdown_gap_share,compatibility_or_fit_phrase_count,compatibility_or_fit_gap_score,compatibility_or_fit_gap_share,setup_or_connectivity_phrase_count,setup_or_connectivity_gap_score,setup_or_connectivity_gap_share,quality_or_durability_phrase_count,quality_or_durability_gap_score,quality_or_durability_gap_share,audio_or_visual_performance_phrase_count,audio_or_visual_performance_gap_score,audio_or_visual_performance_gap_share,dominant_driver_family,dominant_driver_family_share,total_phrase_gap_score
0,0,3284,273,22.223759,0.117320,192,12.578014,0.066400,109,7.618676,0.040219,17,0.531296,0.002805,667,40.039396,0.211370,46,3.407610,0.017989,0,0.000000,0.000000,Mixed,0.211370,189.428253
1,1,3711,399,17.097389,0.120397,138,4.590974,0.032329,433,17.797937,0.125330,36,0.978679,0.006892,273,11.568286,0.081462,178,6.339378,0.044641,275,12.689355,0.089356,Mixed,0.125330,142.008387
2,2,1781,208,8.742860,0.126230,55,1.639866,0.023676,147,5.757577,0.083128,368,18.349154,0.264926,3,0.075318,0.001087,141,7.392076,0.106727,12,0.315889,0.004561,Mixed,0.264926,69.261557
3,3,2605,170,9.451748,0.075105,84,2.971944,0.023615,404,25.624154,0.203612,37,1.096855,0.008716,64,2.875793,0.022851,110,4.938919,0.039245,8,0.166494,0.001323,Mixed,0.203612,125.847917
4,4,3223,349,16.675277,0.133299,223,8.042196,0.064288,241,9.338921,0.074654,39,1.112768,0.008895,76,2.714723,0.021701,87,3.643642,0.029127,20,0.471372,0.003768,Mixed,0.133299,125.096777
5,5,2316,198,14.032182,0.116304,103,4.381949,0.036319,201,14.315412,0.118652,11,0.254744,0.002111,91,4.061798,0.033666,77,4.616656,0.038265,0,0.000000,0.000000,Mixed,0.118652,120.650460
6,6,1175,144,6.655610,0.150053,84,3.087447,0.069608,108,4.194723,0.094572,11,0.247664,0.005584,7,0.149336,0.003367,159,7.095841,0.159978,34,0.814897,0.018372,Mixed,0.159978,44.354976
7,7,3272,307,15.351038,0.116674,224,8.703249,0.066148,149,6.795586,0.051649,15,0.362043,0.002752,235,9.531743,0.072445,91,4.716155,0.035845,476,26.793040,0.203637,Mixed,0.203637,131.572272
8,8,2953,344,16.109991,0.144601,177,6.170199,0.055383,264,10.103149,0.090685,1,0.020455,0.000184,163,5.187015,0.046558,119,4.926655,0.044221,235,11.698091,0.105001,Mixed,0.144601,111.409623
9,9,2453,227,13.757990,0.128272,139,5.260565,0.049047,272,15.686143,0.146249,82,3.175997,0.029611,97,4.007765,0.037366,82,3.791491,0.035350,149,6.627011,0.061786,Mixed,0.146249,107.256621


In [45]:
# Enriching topic outputs with premium-price and priority indicators

premium_price_bands = {"Premium", "Very premium"}

topic_price_band_enriched_dataframe = topic_price_band_comparison_dataframe.copy()
topic_price_band_enriched_dataframe["is_premium_band"] = (
    topic_price_band_enriched_dataframe["price_band"].isin(premium_price_bands)
)

premium_topic_summary_dataframe = (
    topic_price_band_enriched_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        premium_document_count_dissatisfaction = (
            "document_count_dissatisfaction",
            lambda values: values[
                topic_price_band_enriched_dataframe.loc[values.index, "is_premium_band"]
            ].sum()
        ),
        premium_document_count_satisfaction = (
            "document_count_satisfaction",
            lambda values: values[
                topic_price_band_enriched_dataframe.loc[values.index, "is_premium_band"]
            ].sum()
        ),
        total_document_count_dissatisfaction = ("document_count_dissatisfaction", "sum"),
        total_document_count_satisfaction = ("document_count_satisfaction", "sum")
    )
    .reset_index()
)

premium_topic_summary_dataframe["premium_document_share_dissatisfaction"] = (
    premium_topic_summary_dataframe["premium_document_count_dissatisfaction"] /
    premium_topic_summary_dataframe["total_document_count_dissatisfaction"].replace(0, np.nan)
)

premium_topic_summary_dataframe["premium_document_share_satisfaction"] = (
    premium_topic_summary_dataframe["premium_document_count_satisfaction"] /
    premium_topic_summary_dataframe["total_document_count_satisfaction"].replace(0, np.nan)
)

premium_topic_summary_dataframe["premium_share_lift_dissatisfaction_vs_satisfaction"] = (
    (premium_topic_summary_dataframe["premium_document_share_dissatisfaction"] + 1e-6) /
    (premium_topic_summary_dataframe["premium_document_share_satisfaction"] + 1e-6)
)

priority_topic_summary_dataframe = (
    topic_priority_comparison_dataframe.loc[
        topic_priority_comparison_dataframe["issue_priority_level"] == "Priority review",
        [
            "topic_id",
            "document_count_dissatisfaction",
            "document_count_satisfaction",
            "sampled_reviews_dissatisfaction",
            "sampled_reviews_satisfaction",
            "document_share_within_topic_dissatisfaction",
            "document_share_within_topic_satisfaction"
        ]
    ]
    .rename(columns = {
        "document_share_within_topic_dissatisfaction": "priority_document_share_dissatisfaction",
        "document_share_within_topic_satisfaction": "priority_document_share_satisfaction"
    })
    .copy()
)

priority_topic_summary_dataframe["priority_share_lift_dissatisfaction_vs_satisfaction"] = (
    (priority_topic_summary_dataframe["priority_document_share_dissatisfaction"] + 1e-6) /
    (priority_topic_summary_dataframe["priority_document_share_satisfaction"] + 1e-6)
)

topic_reporting_coverage_dataframe = (
    topic_segment_reporting_eligibility_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        eligible_segment_count = ("is_segment_reporting_eligible", "sum"),
        total_segment_count = ("is_segment_reporting_eligible", "count")
    )
    .reset_index()
)

topic_reporting_coverage_dataframe["eligible_segment_ratio"] = (
    topic_reporting_coverage_dataframe["eligible_segment_count"] /
    topic_reporting_coverage_dataframe["total_segment_count"].replace(0, np.nan)
)

topic_insight_mechanics_dataframe = (
    topic_driver_core_dataframe
    .merge(topic_breadth_summary_dataframe, on = "topic_id", how = "left")
    .merge(
        premium_topic_summary_dataframe[
            [
                "topic_id",
                "premium_document_share_dissatisfaction",
                "premium_document_share_satisfaction",
                "premium_share_lift_dissatisfaction_vs_satisfaction"
            ]
        ],
        on = "topic_id",
        how = "left"
    )
    .merge(
        priority_topic_summary_dataframe[
            [
                "topic_id",
                "priority_document_share_dissatisfaction",
                "priority_document_share_satisfaction",
                "priority_share_lift_dissatisfaction_vs_satisfaction"
            ]
        ],
        on = "topic_id",
        how = "left"
    )
    .merge(topic_reporting_coverage_dataframe, on = "topic_id", how = "left")
    .merge(topic_phrase_family_summary_dataframe, on = "topic_id", how = "left")
    .merge(accepted_topic_phrase_summary_dataframe, on = "topic_id", how = "left")
)

topic_insight_mechanics_dataframe["topic_breadth_type"] = np.select(
    [
        topic_insight_mechanics_dataframe["concentration_hhi"] >= 0.25,
        topic_insight_mechanics_dataframe["segment_count_above_05_share"] >= 6
    ],
    [
        "Concentrated",
        "Broad"
    ],
    default = "Moderate"
)

topic_insight_mechanics_dataframe["dissatisfaction_signal_strength"] = np.select(
    [
        topic_insight_mechanics_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.15,
        topic_insight_mechanics_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.08
    ],
    [
        "Strong",
        "Moderate"
    ],
    default = "Weak"
)

topic_insight_mechanics_dataframe["topic_assignment_confidence"] = np.select(
    [
        (
            topic_insight_mechanics_dataframe["average_topic_margin"] >= 0.06
        ) & (
            topic_insight_mechanics_dataframe["low_margin_document_ratio"] <= 0.25
        ),
        (
            topic_insight_mechanics_dataframe["average_topic_margin"] >= 0.04
        ) & (
            topic_insight_mechanics_dataframe["low_margin_document_ratio"] <= 0.35
        )
    ],
    [
        "High",
        "Moderate"
    ],
    default = "Watch"
)

signal_strength_rank_map = {
    "Strong": 3,
    "Moderate": 2,
    "Weak": 1
}

breadth_rank_map = {
    "Broad": 3,
    "Moderate": 2,
    "Concentrated": 1
}

confidence_rank_map = {
    "High": 3,
    "Moderate": 2,
    "Watch": 1
}

topic_insight_mechanics_dataframe["dissatisfaction_signal_strength_rank"] = (
    topic_insight_mechanics_dataframe["dissatisfaction_signal_strength"].map(signal_strength_rank_map)
)

topic_insight_mechanics_dataframe["topic_breadth_rank"] = (
    topic_insight_mechanics_dataframe["topic_breadth_type"].map(breadth_rank_map)
)

topic_insight_mechanics_dataframe["topic_assignment_confidence_rank"] = (
    topic_insight_mechanics_dataframe["topic_assignment_confidence"].map(confidence_rank_map)
)

display(
    topic_insight_mechanics_dataframe.sort_values(
        by = [
            "dissatisfaction_signal_strength_rank",
            "eligible_segment_ratio",
            "topic_assignment_confidence_rank",
            "document_share_lift_dissatisfaction_vs_satisfaction",
            "document_count"
        ],
        ascending = [False, False, False, False, False]
    )
)

,topic_id,document_count,total_sampled_reviews,average_topic_strength,average_rating,average_helpful_vote,top_terms,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,document_share_dissatisfaction,document_share_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,sampled_review_share_lift_dissatisfaction_vs_satisfaction,document_share_gap,max_top_term_jaccard_overlap,average_topic_margin,median_topic_margin,low_margin_document_count,topic_weight_share_dissatisfaction,topic_weight_share_satisfaction,topic_weight_share_lift_dissatisfaction_vs_satisfaction,topic_weight_share_gap,low_margin_document_ratio,covered_segment_count,segment_count_above_05_share,segment_count_above_10_share,max_segment_share,concentration_hhi,premium_document_share_dissatisfaction,premium_document_share_satisfaction,premium_share_lift_dissatisfaction_vs_satisfaction,priority_document_share_dissatisfaction,priority_document_share_satisfaction,priority_share_lift_dissatisfaction_vs_satisfaction,eligible_segment_count,total_segment_count,eligible_segment_ratio,accepted_phrase_count_x,transaction_or_remedy_phrase_count,transaction_or_remedy_gap_score,transaction_or_remedy_gap_share,delivery_or_marketplace_phrase_count,delivery_or_marketplace_gap_score,delivery_or_marketplace_gap_share,product_failure_or_breakdown_phrase_count,product_failure_or_breakdown_gap_score,product_failure_or_breakdown_gap_share,compatibility_or_fit_phrase_count,compatibility_or_fit_gap_score,compatibility_or_fit_gap_share,setup_or_connectivity_phrase_count,setup_or_connectivity_gap_score,setup_or_connectivity_gap_share,quality_or_durability_phrase_count,quality_or_durability_gap_score,quality_or_durability_gap_share,audio_or_visual_performance_phrase_count,audio_or_visual_performance_gap_score,audio_or_visual_performance_gap_share,dominant_driver_family,dominant_driver_family_share,total_phrase_gap_score,accepted_phrase_count_y,top_accepted_phrases,topic_breadth_type,dissatisfaction_signal_strength,topic_assignment_confidence,dissatisfaction_signal_strength_rank,topic_breadth_rank,topic_assignment_confidence_rank
8,12,1060,109780,0.071391,1.256757,2.432281,"charger, charge battery, hold charge, battery ...",1060,740,109780,60861,0.062897,0.040305,1.560507,1.796167,0.022592,0.111111,0.048695,0.023479,391,0.082268,0.050882,1.616824,0.031386,0.368868,9,7,4,0.240566,0.160048,0.226415,0.120270,1.882545,0.234906,0.183784,1.278162,7,9,0.777778,1933,207,12.294289,0.146357,114,4.976747,0.059245,167,8.608134,0.102475,6,0.160581,0.001912,21,0.783988,0.009333,57,2.876798,0.034247,5,0.099567,0.001185,Mixed,0.146357,84.002295,1933,"total waste_money, complete waste_money, wish ...",Broad,Strong,Watch,3,3,1
0,1,1937,285262,0.087805,1.358950,1.290585,"ear, headphone, earbuds, ears, buds, earbud, s...",1937,1926,285262,253887,0.114935,0.104902,1.095641,1.11885,0.010033,0.071429,0.065651,0.067718,160,0.089549,0.093391,0.958862,-0.003842,0.082602,9,8,6,0.207537,0.135442,0.230769,0.234164,0.985502,0.383067,0.388889,0.985028,9,9,1.000000,3711,399,17.097389,0.120397,138,4.590974,0.032329,433,17.797937,0.125330,36,0.978679,0.006892,273,11.568286,0.081462,178,6.339378,0.044641,275,12.689355,0.089356,Mixed,0.125330,142.008387,3711,"work waste_money, return waste, return not_wor...",Broad,Moderate,High,2,3,3
9,5,945,124495,0.089500,1.340732,1.209011,"mouse, keyboard, keys, logitech, mouse work, k...",945,933,124495,110320,0.056073,0.050817,1.103430,1.123739,0.005256,0.000000,0.068815,0.076124,158,0.059885,0.055270,1.083497,0.004615,0.167196,9,7,5,0.214815,0.138913,0.185185,0.210075,0.881520,0.334392,0.350482,0.954090,8,9,0.888889,2316,198,14.032182,0.116304,103,4.381949,0.036319,201,14.315412,0.118652,11,0.254744,0.002111,91,4.061798,0.033666,77,4.616656,0.038265,0,0.000000,0.000000,Mixed,0.118652,120.650460,2316,"stopped_working disappointed, disappointed ret...",Broad,Moderate,High,2,3,3
11,0,830,126020,0.061121,1.267146,2.39

In [46]:
# Scoring topics for root-cause and product-failure relevance

driver_evaluation_dataframe = topic_insight_mechanics_dataframe.copy()

driver_evaluation_dataframe["root_cause_gap_score"] = (
    driver_evaluation_dataframe["product_failure_or_breakdown_gap_score"]
    + driver_evaluation_dataframe["compatibility_or_fit_gap_score"]
    + driver_evaluation_dataframe["setup_or_connectivity_gap_score"]
    + driver_evaluation_dataframe["quality_or_durability_gap_score"]
    + driver_evaluation_dataframe["audio_or_visual_performance_gap_score"]
)

driver_evaluation_dataframe["remedy_gap_score"] = (
    driver_evaluation_dataframe["transaction_or_remedy_gap_score"]
)

driver_evaluation_dataframe["total_driver_gap_score"] = (
    driver_evaluation_dataframe["root_cause_gap_score"]
    + driver_evaluation_dataframe["remedy_gap_score"]
)

driver_evaluation_dataframe["root_cause_share"] = (
    driver_evaluation_dataframe["root_cause_gap_score"] /
    driver_evaluation_dataframe["total_driver_gap_score"].replace(0, np.nan)
)

driver_evaluation_dataframe["remedy_share"] = (
    driver_evaluation_dataframe["remedy_gap_score"] /
    driver_evaluation_dataframe["total_driver_gap_score"].replace(0, np.nan)
)

driver_evaluation_dataframe["topic_breadth_type"] = np.select(
    [
        driver_evaluation_dataframe["concentration_hhi"] >= 0.25,
        driver_evaluation_dataframe["segment_count_above_05_share"] >= 6
    ],
    [
        "Concentrated",
        "Broad"
    ],
    default = "Moderate"
)

driver_evaluation_dataframe["dissatisfaction_signal_strength"] = np.select(
    [
        (
            driver_evaluation_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.12
        ) & (
            driver_evaluation_dataframe["topic_weight_share_lift_dissatisfaction_vs_satisfaction"] >= 1.08
        ),
        (
            driver_evaluation_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.05
        ) & (
            driver_evaluation_dataframe["topic_weight_share_lift_dissatisfaction_vs_satisfaction"] >= 1.03
        )
    ],
    [
        "Strong",
        "Moderate"
    ],
    default = "Weak"
)

driver_evaluation_dataframe["topic_assignment_confidence"] = np.select(
    [
        (
            driver_evaluation_dataframe["average_topic_margin"] >= 0.06
        ) & (
            driver_evaluation_dataframe["low_margin_document_ratio"] <= 0.25
        ),
        (
            driver_evaluation_dataframe["average_topic_margin"] >= 0.04
        ) & (
            driver_evaluation_dataframe["low_margin_document_ratio"] <= 0.35
        )
    ],
    [
        "High",
        "Moderate"
    ],
    default = "Watch"
)

driver_evaluation_dataframe["premium_exposure_level"] = np.select(
    [
        driver_evaluation_dataframe["premium_share_lift_dissatisfaction_vs_satisfaction"] >= 1.08,
        driver_evaluation_dataframe["premium_share_lift_dissatisfaction_vs_satisfaction"] <= 0.92
    ],
    [
        "Premium-tilted",
        "Non-premium-tilted"
    ],
    default = "Balanced"
)

driver_evaluation_dataframe["priority_exposure_level"] = np.select(
    [
        driver_evaluation_dataframe["priority_share_lift_dissatisfaction_vs_satisfaction"] >= 1.08,
        driver_evaluation_dataframe["priority_share_lift_dissatisfaction_vs_satisfaction"] <= 0.92
    ],
    [
        "Priority-product-tilted",
        "Monitor-product-tilted"
    ],
    default = "Balanced"
)

driver_evaluation_dataframe["driver_balance_type"] = np.select(
    [
        driver_evaluation_dataframe["root_cause_share"] >= 0.60,
        driver_evaluation_dataframe["remedy_share"] >= 0.60
    ],
    [
        "Root-cause heavy",
        "Remedy-heavy"
    ],
    default = "Mixed"
)

driver_evaluation_dataframe["is_high_coverage_topic"] = (
    driver_evaluation_dataframe["eligible_segment_ratio"] >= 0.75
)

driver_evaluation_dataframe["is_sufficiently_large"] = (
    driver_evaluation_dataframe["document_count_dissatisfaction"] >= 800
)

driver_evaluation_dataframe["is_assignment_reliable"] = (
    driver_evaluation_dataframe["topic_assignment_confidence"].isin(["High", "Moderate"])
)

driver_evaluation_dataframe["is_driver_candidate"] = (
    driver_evaluation_dataframe["is_high_coverage_topic"]
    & driver_evaluation_dataframe["is_sufficiently_large"]
    & driver_evaluation_dataframe["is_assignment_reliable"]
)

driver_evaluation_dataframe["driver_interpretation_status"] = np.select(
    [
        (
            driver_evaluation_dataframe["is_driver_candidate"]
            & driver_evaluation_dataframe["dissatisfaction_signal_strength"].eq("Strong")
            & driver_evaluation_dataframe["driver_balance_type"].eq("Root-cause heavy")
        ),
        (
            driver_evaluation_dataframe["is_driver_candidate"]
            & driver_evaluation_dataframe["dissatisfaction_signal_strength"].isin(["Strong", "Moderate"])
            & driver_evaluation_dataframe["driver_balance_type"].eq("Mixed")
        ),
        (
            driver_evaluation_dataframe["is_driver_candidate"]
            & driver_evaluation_dataframe["driver_balance_type"].eq("Remedy-heavy")
        )
    ],
    [
        "Root-cause dominant candidate",
        "Mixed candidate",
        "Remedy-heavy candidate"
    ],
    default = "Watch / low-confidence candidate"
)

driver_evaluation_result_dataframe = (
    driver_evaluation_dataframe[
        [
            "topic_id",
            "top_terms",
            "document_count_dissatisfaction",
            "document_count_satisfaction",
            "document_share_lift_dissatisfaction_vs_satisfaction",
            "topic_weight_share_lift_dissatisfaction_vs_satisfaction",
            "eligible_segment_ratio",
            "topic_assignment_confidence",
            "topic_breadth_type",
            "dominant_driver_family",
            "dominant_driver_family_share",
            "root_cause_share",
            "remedy_share",
            "premium_exposure_level",
            "priority_exposure_level",
            "driver_balance_type",
            "driver_interpretation_status",
            "top_accepted_phrases"
        ]
    ]
    .sort_values(
        by = [
            "driver_interpretation_status",
            "document_share_lift_dissatisfaction_vs_satisfaction",
            "eligible_segment_ratio",
            "document_count_dissatisfaction"
        ],
        ascending = [True, False, False, False]
    )
    .reset_index(drop = True)
)

display(driver_evaluation_result_dataframe)

,topic_id,top_terms,document_count_dissatisfaction,document_count_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,topic_weight_share_lift_dissatisfaction_vs_satisfaction,eligible_segment_ratio,topic_assignment_confidence,topic_breadth_type,dominant_driver_family,dominant_driver_family_share,root_cause_share,remedy_share,premium_exposure_level,priority_exposure_level,driver_balance_type,driver_interpretation_status,top_accepted_phrases
0,11,"remote, remote work, remotes, buttons, remote ...",915,923,1.079976,1.093902,1.000000,Moderate,Broad,Mixed,0.130775,0.575865,0.424135,Balanced,Balanced,Mixed,Mixed candidate,"late return, total waste_money, dont buy, miss..."
1,4,"drive, drives, files, disk, data, hard_drive, ...",1545,1571,1.071390,1.145317,1.000000,Moderate,Broad,Mixed,0.133299,0.508925,0.491075,Balanced,Balanced,Mixed,Mixed candidate,"outside return_window, try multiple computers,..."
2,12,"charger, charge battery, hold charge, battery ...",1060,740,1.560507,1.616824,0.777778,Watch,Broad,Mixed,0.146357,0.504729,0.495271,Premium-tilted,Priority-product-tilted,Mixed,Watch / low-confidence candidate,"total waste_money, complete waste_money, wish ..."
3,3,"charge phone, cords, chargers, charger, not_ch...",1231,1171,1.145238,1.298866,0.555556,Moderate,Concentrated,Mixed,0.203612,0.785937,0.214063,Non-premium-tilted,Priority-product-tilted,Root-cause heavy,Watch / low-confidence candidate,"stopped_working waste_money, return try, retur..."
4,5,"mouse, keyboard, keys, logitech, mouse work, k...",945,933,1.103430,1.083497,0.888889,High,Broad,Mixed,0.118652,0.623608,0.376392,Non-premium-tilted,Balanced,Root-cause heavy,Watch / low-confidence candidate,"stopped_working disappointed, disappointed ret..."
5,0,"router, wifi, network, internet, modem, netgea...",830,822,1.100021,1.274283,0.777778,Moderate,Broad,Mixed,0.211370,0.698950,0.301050,Premium-tilted,Balanced,Root-cause heavy,Watch / low-confidence candidate,"outside return_window, past return_window, com..."
6,1,"ear, headphone, earbuds, ears, buds, earbud, s...",1937,1926,1.095641,0.958862,1.000000,High,Broad,Mixed,0.125330,0.742784,0.257216,Balanced,Balanced,Root-cause heavy,Watch / low-confidence candidate,"work waste_money, return waste, return not_wor..."
7,10,"bubbles, screen_protector, protectors, screen ...",507,515,1.072495,1.100252,0.333333,High,Concentrated,Mixed,0.145281,0.861071,0.138929,Non-premium-tilted,Balanced,Root-cause heavy,Watch / low-confidence candidate,"complete waste_money, buyer beware, total wast..."
8,9,"hdmi, monitor, hdmi cable, display, adapter, p...",1109,1163,1.038836,1.019514,1.000000,Moderate,Broad,Mixed,0.146249,0.707565,0.292435,Balanced,Balanced,Root-cause heavy,Watch / low-confidence candidate,"does_not_work advertised, waste bought, return..."
9,13,"antenna, stations, reception, channels, miles,...",558,593,1.025120,0.888472,0.888889,High,Broad,Mixed,0.204672,0.523067,0.476933,Balanced,Balanced,Mixed,Watch / low-confidence candidate,"window return, dont waste, work waste, does_no..."


In [47]:
# Removing non-root-cause language from topic modeling text
root_cause_non_cause_tokens = (
    remedy_or_transaction_tokens
    .union(evaluative_outcome_tokens)
)

def build_root_cause_text(modeling_text_value):
    if not modeling_text_value:
        return ""

    filtered_tokens = [
        token
        for token in str(modeling_text_value).split()
        if token not in root_cause_non_cause_tokens
    ]

    return " ".join(filtered_tokens)

full_corpus_modeling_dataframe["root_cause_modeling_text"] = (
    full_corpus_modeling_dataframe["modeling_text_for_vectorizer"]
    .fillna("")
    .apply(build_root_cause_text)
    .str.replace(r"\s+", " ", regex = True)
    .str.strip()
)

full_corpus_modeling_dataframe["root_cause_token_count"] = (
    full_corpus_modeling_dataframe["root_cause_modeling_text"]
    .str.split()
    .str.len()
)

root_cause_text_quality_summary_dataframe = pd.DataFrame({
    "metric": [
        "full_modeling_document_count",
        "blank_root_cause_text_count",
        "documents_under_30_root_cause_tokens",
        "documents_under_50_root_cause_tokens",
        "average_root_cause_token_count",
        "median_root_cause_token_count"
    ],
    "value": [
        len(full_corpus_modeling_dataframe),
        full_corpus_modeling_dataframe["root_cause_modeling_text"].eq("").sum(),
        full_corpus_modeling_dataframe["root_cause_token_count"].lt(30).sum(),
        full_corpus_modeling_dataframe["root_cause_token_count"].lt(50).sum(),
        full_corpus_modeling_dataframe["root_cause_token_count"].mean(),
        full_corpus_modeling_dataframe["root_cause_token_count"].median()
    ]
})

display(root_cause_text_quality_summary_dataframe)

,metric,value
0,full_modeling_document_count,35213.000000
1,blank_root_cause_text_count,0.000000
2,documents_under_30_root_cause_tokens,0.000000
3,documents_under_50_root_cause_tokens,0.000000
4,average_root_cause_token_count,1277.601284
5,median_root_cause_token_count,1120.000000


In [48]:
# Preparing root-cause-focused dissatisfied documents for topic modeling
root_cause_dissatisfaction_dataframe = full_corpus_modeling_dataframe.loc[
    full_corpus_modeling_dataframe["analysis_pool_role"].eq("Topic modeling target")
    & full_corpus_modeling_dataframe["root_cause_modeling_text"].ne("")
].copy()

root_cause_tfidf_vectorizer = TfidfVectorizer(
    lowercase = False,
    stop_words = None,
    token_pattern = r"(?u)\b[a-z_][a-z0-9_]{2,}\b",
    ngram_range = (1, 2),
    min_df = 25,
    max_df = 0.35,
    max_features = 30000,
    sublinear_tf = True
)

root_cause_dissatisfaction_tfidf_matrix = root_cause_tfidf_vectorizer.fit_transform(
    root_cause_dissatisfaction_dataframe["root_cause_modeling_text"]
)

root_cause_selected_topic_count = 15

root_cause_nmf_model = NMF(
    n_components = root_cause_selected_topic_count,
    init = "nndsvda",
    random_state = 42,
    max_iter = 400
)

root_cause_document_topic_matrix = root_cause_nmf_model.fit_transform(
    root_cause_dissatisfaction_tfidf_matrix
)

root_cause_topic_term_matrix = root_cause_nmf_model.components_
root_cause_feature_names = np.array(root_cause_tfidf_vectorizer.get_feature_names_out())

root_cause_topic_rows = []

for topic_index, topic_weights in enumerate(root_cause_topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : 15]
    root_cause_topic_rows.append({
        "topic_id": topic_index,
        "top_terms": ", ".join(root_cause_feature_names[top_term_indices])
    })

root_cause_topic_terms_dataframe = pd.DataFrame(root_cause_topic_rows)
display(root_cause_topic_terms_dataframe)

,topic_id,top_terms
0,0,"router, wifi, network, internet, modem, netgea..."
1,1,"ear, headphone, earbuds, ears, buds, earbud, s..."
2,2,"ipad, not_fit, protect, color, cases, fits, co..."
3,3,"charge phone, cords, chargers, charger, not_ch..."
4,4,"drive, drives, files, disk, data, hard_drive, ..."
5,5,"mouse, keyboard, keys, logitech, mouse work, k..."
6,6,"screws, mount, screw, mounting, bracket, tight..."
7,7,"camera, camera work, motion, recording, view, ..."
8,8,"speaker, sound_quality, music, bluetooth, bass..."
9,9,"hdmi, monitor, hdmi cable, display, adapter, p..."


In [49]:
# Comparing topic counts for the root-cause-focused model

candidate_root_cause_topic_counts = [10, 12, 15, 18]

root_cause_topic_model_diagnostics = []

for topic_count in candidate_root_cause_topic_counts:
    candidate_root_cause_nmf_model = NMF(
        n_components = topic_count,
        init = "nndsvda",
        random_state = 42,
        max_iter = 400
    )

    candidate_root_cause_document_topic_matrix = candidate_root_cause_nmf_model.fit_transform(
        root_cause_dissatisfaction_tfidf_matrix
    )

    dominant_topic_assignment = candidate_root_cause_document_topic_matrix.argmax(axis = 1)
    dominant_topic_strength = candidate_root_cause_document_topic_matrix.max(axis = 1)

    topic_size_dataframe = (
        pd.DataFrame({
            "topic_id": dominant_topic_assignment,
            "topic_strength": dominant_topic_strength
        })
        .groupby("topic_id", dropna = False)
        .agg(
            document_count = ("topic_id", "count"),
            average_topic_strength = ("topic_strength", "mean"),
            median_topic_strength = ("topic_strength", "median")
        )
        .reset_index()
    )

    root_cause_topic_model_diagnostics.append({
        "topic_count": topic_count,
        "reconstruction_error": candidate_root_cause_nmf_model.reconstruction_err_,
        "mean_max_topic_strength": dominant_topic_strength.mean(),
        "median_max_topic_strength": np.median(dominant_topic_strength),
        "smallest_topic_document_count": topic_size_dataframe["document_count"].min(),
        "largest_topic_document_count": topic_size_dataframe["document_count"].max(),
        "topic_size_ratio_max_to_min": (
            topic_size_dataframe["document_count"].max() /
            topic_size_dataframe["document_count"].min()
        )
    })

root_cause_topic_model_diagnostics_dataframe = pd.DataFrame(root_cause_topic_model_diagnostics)
display(root_cause_topic_model_diagnostics_dataframe.sort_values("topic_count"))

,topic_count,reconstruction_error,mean_max_topic_strength,median_max_topic_strength,smallest_topic_document_count,largest_topic_document_count,topic_size_ratio_max_to_min
0,10,122.173674,0.073915,0.066940,493,3098,6.283976
1,12,121.625818,0.076147,0.072939,506,1987,3.926877
2,15,120.981810,0.078689,0.074678,468,1947,4.160256
3,18,120.415896,0.080843,0.075005,419,1952,4.658711


In [50]:
# Applying the root-cause topic model to comparison documents

root_cause_comparison_dataframe = full_corpus_modeling_dataframe.loc[
    ~full_corpus_modeling_dataframe["is_dissatisfaction"]
].copy()

root_cause_comparison_tfidf_matrix = root_cause_tfidf_vectorizer.transform(
    root_cause_comparison_dataframe["root_cause_modeling_text"]
)

root_cause_comparison_topic_matrix = root_cause_nmf_model.transform(
    root_cause_comparison_tfidf_matrix
)

root_cause_dissatisfaction_dataframe = root_cause_dissatisfaction_dataframe.reset_index(drop = True)
root_cause_comparison_dataframe = root_cause_comparison_dataframe.reset_index(drop = True)

root_cause_dissatisfaction_dataframe["topic_id"] = root_cause_document_topic_matrix.argmax(axis = 1)
root_cause_dissatisfaction_dataframe["topic_strength"] = root_cause_document_topic_matrix.max(axis = 1)

root_cause_comparison_dataframe["topic_id"] = root_cause_comparison_topic_matrix.argmax(axis = 1)
root_cause_comparison_dataframe["topic_strength"] = root_cause_comparison_topic_matrix.max(axis = 1)

root_cause_topic_assignment_dataframe = pd.concat(
    [
        root_cause_dissatisfaction_dataframe.assign(cohort_role = "Dissatisfaction"),
        root_cause_comparison_dataframe.assign(cohort_role = "Satisfaction")
    ],
    ignore_index = True
)

root_cause_topic_rows = []

for topic_index, topic_weights in enumerate(root_cause_topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : 15]
    root_cause_topic_rows.append({
        "topic_id": topic_index,
        "top_terms": ", ".join(root_cause_feature_names[top_term_indices])
    })

root_cause_topic_terms_dataframe = pd.DataFrame(root_cause_topic_rows)

display(root_cause_topic_terms_dataframe)
print(f"Root-cause topic assignment rows : {len(root_cause_topic_assignment_dataframe):,}")
display(root_cause_topic_assignment_dataframe["cohort_role"].value_counts(dropna = False))

,topic_id,top_terms
0,0,"router, wifi, network, internet, modem, netgea..."
1,1,"ear, headphone, earbuds, ears, buds, earbud, s..."
2,2,"ipad, not_fit, protect, color, cases, fits, co..."
3,3,"charge phone, cords, chargers, charger, not_ch..."
4,4,"drive, drives, files, disk, data, hard_drive, ..."
5,5,"mouse, keyboard, keys, logitech, mouse work, k..."
6,6,"screws, mount, screw, mounting, bracket, tight..."
7,7,"camera, camera work, motion, recording, view, ..."
8,8,"speaker, sound_quality, music, bluetooth, bass..."
9,9,"hdmi, monitor, hdmi cable, display, adapter, p..."


Root-cause topic assignment rows : 35,213


cohort_role
Satisfaction       18360
Dissatisfaction    16853
Name: count, dtype: int64

In [51]:
# Reviewing root-cause topic distribution across dissatisfied segments

root_cause_dissatisfaction_segment_distribution_dataframe = (
    root_cause_topic_assignment_dataframe.loc[
        root_cause_topic_assignment_dataframe["cohort_role"] == "Dissatisfaction"
    ]
    .groupby(["topic_id", "price_band", "issue_priority_level"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

root_cause_dissatisfaction_segment_distribution_dataframe["segment_document_share_within_topic"] = (
    root_cause_dissatisfaction_segment_distribution_dataframe["document_count"] /
    root_cause_dissatisfaction_segment_distribution_dataframe.groupby("topic_id")["document_count"].transform("sum")
)

root_cause_topic_breadth_summary_dataframe = (
    root_cause_dissatisfaction_segment_distribution_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        covered_segment_count = ("segment_document_share_within_topic", "count"),
        segment_count_above_05_share = ("segment_document_share_within_topic", lambda values: (values >= 0.05).sum()),
        segment_count_above_10_share = ("segment_document_share_within_topic", lambda values: (values >= 0.10).sum()),
        max_segment_share = ("segment_document_share_within_topic", "max"),
        concentration_hhi = ("segment_document_share_within_topic", lambda values: np.square(values).sum())
    )
    .reset_index()
)

display(
    root_cause_topic_breadth_summary_dataframe.sort_values(
        by = ["concentration_hhi", "max_segment_share"],
        ascending = [False, False]
    )
)

,topic_id,covered_segment_count,segment_count_above_05_share,segment_count_above_10_share,max_segment_share,concentration_hhi
10,10,8,3,3,0.570020,0.382149
3,3,9,4,3,0.462158,0.300806
2,2,9,4,3,0.402056,0.265955
6,6,9,4,4,0.305778,0.210719
7,7,9,6,4,0.295356,0.181152
4,4,9,6,4,0.210560,0.162824
0,0,9,6,5,0.250301,0.161870
12,12,9,7,4,0.245703,0.160750
8,8,9,6,4,0.236231,0.159741
9,9,9,7,4,0.259160,0.156537


In [52]:
# Preparing root-cause topic labels and top-term mappings

root_cause_topic_top_terms_map = {}

for topic_index, topic_weights in enumerate(root_cause_topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : 15]
    root_cause_topic_top_terms_map[topic_index] = root_cause_feature_names[top_term_indices].tolist()

root_cause_topic_assignment_dataframe["topic_margin"] = compute_assignment_margin(
    np.vstack([
        root_cause_document_topic_matrix,
        root_cause_comparison_topic_matrix
    ])
)

root_cause_topic_assignment_dataframe["is_topic_core_document"] = (
    (root_cause_topic_assignment_dataframe["topic_strength"] >= 0.06)
    & (root_cause_topic_assignment_dataframe["topic_margin"] >= 0.03)
)

root_cause_topic_core_summary_dataframe = (
    root_cause_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        topic_core_document_count = ("is_topic_core_document", "sum"),
        average_topic_strength = ("topic_strength", "mean"),
        average_topic_margin = ("topic_margin", "mean")
    )
    .reset_index()
)

root_cause_topic_core_summary_dataframe["topic_core_ratio"] = (
    root_cause_topic_core_summary_dataframe["topic_core_document_count"] /
    root_cause_topic_core_summary_dataframe["document_count"].replace(0, np.nan)
)

display(
    root_cause_topic_core_summary_dataframe.sort_values(
        ["cohort_role", "topic_id"]
    )
)

,cohort_role,topic_id,document_count,topic_core_document_count,average_topic_strength,average_topic_margin,topic_core_ratio
0,Dissatisfaction,0,831,509,0.061015,0.048547,0.612515
1,Dissatisfaction,1,1947,1535,0.088046,0.066550,0.788392
2,Dissatisfaction,2,1654,1089,0.075338,0.055272,0.658404
3,Dissatisfaction,3,1242,746,0.086178,0.067602,0.600644
4,Dissatisfaction,4,1553,896,0.075913,0.055225,0.576948
5,Dissatisfaction,5,946,700,0.089248,0.069135,0.739958
6,Dissatisfaction,6,1246,592,0.068028,0.047184,0.475120
7,Dissatisfaction,7,1378,831,0.076247,0.052545,0.603048
8,Dissatisfaction,8,1507,618,0.061000,0.033495,0.410086
9,Dissatisfaction,9,1119,715,0.076120,0.051045,0.638963


In [53]:
# Defining terms to remove for pure product-cause analysis

pure_product_non_cause_text_tokens = (
    remedy_or_transaction_tokens
    .union(delivery_or_fulfillment_tokens)
    .union(evaluative_outcome_tokens)
)

def build_pure_product_cause_text(modeling_text_value):
    if not modeling_text_value:
        return ""

    filtered_tokens = [
        token
        for token in str(modeling_text_value).split()
        if token not in pure_product_non_cause_text_tokens
    ]

    return " ".join(filtered_tokens)

In [54]:
# Creating product-cause-focused text for issue analysis

full_corpus_modeling_dataframe["pure_product_cause_text"] = (
    full_corpus_modeling_dataframe["modeling_text_for_vectorizer"]
    .fillna("")
    .apply(build_pure_product_cause_text)
    .str.replace(r"\s+", " ", regex = True)
    .str.strip()
)

full_corpus_modeling_dataframe["pure_product_cause_token_count"] = (
    full_corpus_modeling_dataframe["pure_product_cause_text"].str.split().str.len()
)

pure_product_cause_quality_summary_dataframe = pd.DataFrame({
    "metric": [
        "document_count",
        "blank_pure_product_cause_text_count",
        "documents_under_30_tokens",
        "documents_under_50_tokens",
        "average_token_count",
        "median_token_count"
    ],
    "value": [
        len(full_corpus_modeling_dataframe),
        full_corpus_modeling_dataframe["pure_product_cause_text"].eq("").sum(),
        full_corpus_modeling_dataframe["pure_product_cause_token_count"].lt(30).sum(),
        full_corpus_modeling_dataframe["pure_product_cause_token_count"].lt(50).sum(),
        full_corpus_modeling_dataframe["pure_product_cause_token_count"].mean(),
        full_corpus_modeling_dataframe["pure_product_cause_token_count"].median()
    ]
})

display(pure_product_cause_quality_summary_dataframe)

,metric,value
0,document_count,35213.000000
1,blank_pure_product_cause_text_count,0.000000
2,documents_under_30_tokens,0.000000
3,documents_under_50_tokens,0.000000
4,average_token_count,1263.107602
5,median_token_count,1108.000000


In [55]:
# Checking whether non-product-cause language remains in the cleaned text
contamination_audit_rows = []

for token_family_name, token_family in {
    "remedy_or_transaction": remedy_or_transaction_tokens,
    "delivery_or_fulfillment": delivery_or_fulfillment_tokens
}.items():
    contamination_audit_rows.append({
        "token_family": token_family_name,
        "documents_with_family_signal": full_corpus_modeling_dataframe["modeling_text_for_vectorizer"]
            .apply(lambda value: any(token in str(value).split() for token in token_family))
            .sum(),
        "documents_with_family_signal_ratio": full_corpus_modeling_dataframe["modeling_text_for_vectorizer"]
            .apply(lambda value: any(token in str(value).split() for token in token_family))
            .mean()
    })

contamination_audit_dataframe = pd.DataFrame(contamination_audit_rows)
display(contamination_audit_dataframe)

,token_family,documents_with_family_signal,documents_with_family_signal_ratio
0,remedy_or_transaction,27979,0.794565
1,delivery_or_fulfillment,34476,0.979070


In [56]:
# Reviewing protected technical and business terms retained in the text

protected_and_business_tokens = sorted(
    protected_phrase_tokens.union(short_technical_tokens)
)

token_policy_audit_rows = []

for token_value in protected_and_business_tokens:
    document_count = full_corpus_modeling_dataframe["modeling_text_for_vectorizer"].str.contains(
        rf"\b{re.escape(token_value)}\b",
        regex = True,
        na = False
    ).sum()

    if document_count > 0:
        token_policy_audit_rows.append({
            "token": token_value,
            "document_count": document_count,
            "document_ratio": document_count / len(full_corpus_modeling_dataframe)
        })

token_policy_audit_dataframe = (
    pd.DataFrame(token_policy_audit_rows)
    .sort_values(["document_count", "token"], ascending = [False, True])
    .reset_index(drop = True)
)

full_clipping_effect_summary_dataframe = (
    full_topic_assignment_dataframe
    .groupby(["topic_id", "cohort_role", "clipping_applied"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

display(token_policy_audit_dataframe.head(120))
display(full_clipping_effect_summary_dataframe.sort_values(["topic_id", "cohort_role", "clipping_applied"]))

,token,document_count,document_ratio
0,did_not_work,18028,0.511970
1,does_not_work,17850,0.506915
2,customer_service,15865,0.450544
3,stopped_working,15553,0.441683
4,usb,15550,0.441598
5,pc,12287,0.348934
6,tv,12164,0.345441
7,not_working,11146,0.316531
8,bluetooth,10060,0.285690
9,sound_quality,9240,0.262403


,topic_id,cohort_role,clipping_applied,document_count,sampled_reviews
0,0,Dissatisfaction,False,413,29761
1,0,Dissatisfaction,True,417,96259
2,0,Satisfaction,False,459,31658
3,0,Satisfaction,True,363,82194
4,1,Dissatisfaction,False,1007,72968
5,1,Dissatisfaction,True,930,212294
6,1,Satisfaction,False,1157,79855
7,1,Satisfaction,True,769,174032
8,2,Dissatisfaction,False,1319,79567
9,2,Dissatisfaction,True,330,71710


In [57]:
# Summarizing root-cause topic exposure and cohort differences

root_cause_topic_summary_dataframe = (
    root_cause_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean"),
        average_topic_margin = ("topic_margin", "mean")
    )
    .reset_index()
)

root_cause_cohort_totals_dataframe = (
    root_cause_topic_assignment_dataframe
    .groupby("cohort_role", dropna = False)
    .agg(
        total_documents = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

root_cause_topic_summary_dataframe = root_cause_topic_summary_dataframe.merge(
    root_cause_cohort_totals_dataframe,
    on = "cohort_role",
    how = "left"
)

root_cause_topic_summary_dataframe["document_share"] = (
    root_cause_topic_summary_dataframe["document_count"] /
    root_cause_topic_summary_dataframe["total_documents"]
)

root_cause_topic_summary_pivot_dataframe = (
    root_cause_topic_summary_dataframe[
        [
            "cohort_role",
            "topic_id",
            "document_count",
            "sampled_reviews",
            "document_share",
            "average_topic_strength",
            "average_topic_margin"
        ]
    ]
    .pivot(index = "topic_id", columns = "cohort_role")
)

root_cause_topic_summary_pivot_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in root_cause_topic_summary_pivot_dataframe.columns
]

root_cause_topic_summary_pivot_dataframe = root_cause_topic_summary_pivot_dataframe.reset_index()

root_cause_topic_summary_pivot_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] = (
    (root_cause_topic_summary_pivot_dataframe["document_share_dissatisfaction"] + 1e-6) /
    (root_cause_topic_summary_pivot_dataframe["document_share_satisfaction"] + 1e-6)
)

root_cause_topic_summary_pivot_dataframe["document_share_gap"] = (
    root_cause_topic_summary_pivot_dataframe["document_share_dissatisfaction"] -
    root_cause_topic_summary_pivot_dataframe["document_share_satisfaction"]
)

root_cause_topic_driver_dataframe = (
    root_cause_topic_summary_pivot_dataframe
    .merge(root_cause_topic_terms_dataframe, on = "topic_id", how = "left")
    .merge(
        root_cause_topic_breadth_summary_dataframe,
        on = "topic_id",
        how = "left"
    )
    .merge(
        root_cause_topic_core_summary_dataframe.loc[
            root_cause_topic_core_summary_dataframe["cohort_role"] == "Dissatisfaction",
            ["topic_id", "topic_core_ratio"]
        ],
        on = "topic_id",
        how = "left"
    )
)

root_cause_topic_driver_dataframe["breadth_type"] = np.select(
    [
        root_cause_topic_driver_dataframe["concentration_hhi"] >= 0.25,
        root_cause_topic_driver_dataframe["segment_count_above_05_share"] >= 6
    ],
    [
        "Concentrated",
        "Broad"
    ],
    default = "Moderate"
)

root_cause_topic_driver_dataframe["signal_strength"] = np.select(
    [
        root_cause_topic_driver_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.12,
        root_cause_topic_driver_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.05
    ],
    [
        "Strong",
        "Moderate"
    ],
    default = "Weak"
)

root_cause_topic_driver_dataframe["confidence_level"] = np.select(
    [
        (
            root_cause_topic_driver_dataframe["average_topic_margin_dissatisfaction"] >= 0.06
        ) & (
            root_cause_topic_driver_dataframe["topic_core_ratio"] >= 0.60
        ),
        (
            root_cause_topic_driver_dataframe["average_topic_margin_dissatisfaction"] >= 0.04
        ) & (
            root_cause_topic_driver_dataframe["topic_core_ratio"] >= 0.45
        )
    ],
    [
        "High",
        "Moderate"
    ],
    default = "Watch"
)

display(
    root_cause_topic_driver_dataframe.sort_values(
        by = [
            "signal_strength",
            "confidence_level",
            "document_share_lift_dissatisfaction_vs_satisfaction",
            "document_count_dissatisfaction"
        ],
        ascending = [True, True, False, False]
    )
)

,topic_id,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,document_share_dissatisfaction,document_share_satisfaction,average_topic_strength_dissatisfaction,average_topic_strength_satisfaction,average_topic_margin_dissatisfaction,average_topic_margin_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,document_share_gap,top_terms,covered_segment_count,segment_count_above_05_share,segment_count_above_10_share,max_segment_share,concentration_hhi,topic_core_ratio,breadth_type,signal_strength,confidence_level
5,5,946,930,124571,110335,0.056132,0.050654,0.089248,0.076905,0.069135,0.057840,1.108161,0.005479,"mouse, keyboard, keys, logitech, mouse work, k...",9,7,5,0.213531,0.138520,0.739958,Broad,Moderate,High
1,1,1947,1930,286298,254459,0.115528,0.105120,0.088046,0.087748,0.066550,0.070896,1.099015,0.010409,"ear, headphone, earbuds, ears, buds, earbud, s...",9,8,6,0.206471,0.135249,0.788392,Broad,Moderate,High
10,10,507,509,51105,48732,0.030084,0.027723,0.115506,0.095682,0.101770,0.081768,1.085137,0.002360,"bubbles, screen_protector, protectors, screen ...",8,3,3,0.570020,0.382149,0.690335,Concentrated,Moderate,High
0,0,831,820,126258,113611,0.049309,0.044662,0.061015,0.053132,0.048547,0.038469,1.104032,0.004646,"router, wifi, network, internet, modem, netgea...",9,6,5,0.250301,0.161870,0.612515,Broad,Moderate,Moderate
11,11,910,917,115719,103240,0.053996,0.049946,0.077667,0.070613,0.055715,0.051122,1.081102,0.004051,"remote, remote work, remotes, buttons, remote ...",9,8,4,0.210989,0.144821,0.637363,Broad,Moderate,Moderate
4,4,1553,1577,184418,171614,0.092150,0.085893,0.075913,0.063353,0.055225,0.043534,1.072840,0.006257,"drive, drives, files, disk, data, hard_drive, ...",9,6,4,0.210560,0.162824,0.576948,Broad,Moderate,Moderate
3,3,1242,1185,163692,131985,0.073696,0.064542,0.086178,0.067779,0.067602,0.050756,1.141821,0.009154,"charge phone, cords, chargers, charger, not_ch...",9,4,3,0.462158,0.300806,0.600644,Concentrated,Strong,High
12,12,989,711,101780,58266,0.058684,0.038725,0.070092,0.062641,0.048132,0.044222,1.515369,0.019958,"charger, charge battery, hold charge, battery ...",9,7,4,0.245703,0.160750,0.406471,Broad,Strong,Watch
14,14,468,539,51535,55977,0.027770,0.029357,0.106479,0.099458,0.086168,0.084357,0.945918,-0.001588,"fitbit, band, wrist, watch, bands, heart rate,...",9,7,6,0.235043,0.142861,0.773504,Broad,Weak,High
9,9,1119,1175,117394,117420,0.066398,0.063998,0.076120,0.068395,0.051045,0.044586,1.037498,0.002400,"hdmi, monitor, hdmi cable, display, adapter, p...",9,7,4,0.259160,0.156537,0.638963,Broad,Weak,Moderate


In [58]:
# Preparing product-cause dissatisfied documents for final issue analysis

pure_product_cause_dissatisfaction_dataframe = full_corpus_modeling_dataframe.loc[
    full_corpus_modeling_dataframe["analysis_pool_role"].eq("Topic modeling target")
    & full_corpus_modeling_dataframe["pure_product_cause_text"].ne("")
].copy()

pure_product_cause_comparison_dataframe = full_corpus_modeling_dataframe.loc[
    ~full_corpus_modeling_dataframe["is_dissatisfaction"]
    & full_corpus_modeling_dataframe["pure_product_cause_text"].ne("")
].copy()

print(f"Dissatisfaction documents : {len(pure_product_cause_dissatisfaction_dataframe):,}")
print(f"Satisfaction documents : {len(pure_product_cause_comparison_dataframe):,}")

Dissatisfaction documents : 16,853
Satisfaction documents : 18,360


In [59]:
# Vectorizing product-cause text for topic modeling

pure_product_cause_tfidf_vectorizer = TfidfVectorizer(
    lowercase = False,
    stop_words = None,
    token_pattern = r"(?u)\b[a-z_][a-z0-9_]{2,}\b",
    ngram_range = (1, 2),
    min_df = 25,
    max_df = 0.35,
    max_features = 30000,
    sublinear_tf = True
)

pure_product_cause_dissatisfaction_tfidf_matrix = pure_product_cause_tfidf_vectorizer.fit_transform(
    pure_product_cause_dissatisfaction_dataframe["pure_product_cause_text"]
)

candidate_pure_product_topic_counts = [10, 12, 15, 18]

pure_product_topic_model_diagnostics = []

for topic_count in candidate_pure_product_topic_counts:
    candidate_model = NMF(
        n_components = topic_count,
        init = "nndsvda",
        random_state = 42,
        max_iter = 400
    )

    candidate_document_topic_matrix = candidate_model.fit_transform(
        pure_product_cause_dissatisfaction_tfidf_matrix
    )

    dominant_topic_assignment = candidate_document_topic_matrix.argmax(axis = 1)
    dominant_topic_strength = candidate_document_topic_matrix.max(axis = 1)

    topic_size_dataframe = (
        pd.DataFrame({
            "topic_id": dominant_topic_assignment,
            "topic_strength": dominant_topic_strength
        })
        .groupby("topic_id", dropna = False)
        .agg(
            document_count = ("topic_id", "count"),
            average_topic_strength = ("topic_strength", "mean"),
            median_topic_strength = ("topic_strength", "median")
        )
        .reset_index()
    )

    pure_product_topic_model_diagnostics.append({
        "topic_count": topic_count,
        "reconstruction_error": candidate_model.reconstruction_err_,
        "mean_max_topic_strength": dominant_topic_strength.mean(),
        "median_max_topic_strength": np.median(dominant_topic_strength),
        "smallest_topic_document_count": topic_size_dataframe["document_count"].min(),
        "largest_topic_document_count": topic_size_dataframe["document_count"].max(),
        "topic_size_ratio_max_to_min": (
            topic_size_dataframe["document_count"].max() /
            topic_size_dataframe["document_count"].min()
        )
    })

pure_product_topic_model_diagnostics_dataframe = pd.DataFrame(
    pure_product_topic_model_diagnostics
)

display(pure_product_topic_model_diagnostics_dataframe.sort_values("topic_count"))

,topic_count,reconstruction_error,mean_max_topic_strength,median_max_topic_strength,smallest_topic_document_count,largest_topic_document_count,topic_size_ratio_max_to_min
0,10,122.159686,0.074099,0.067141,493,3089,6.265720
1,12,121.607258,0.076324,0.072984,508,1989,3.915354
2,15,120.958521,0.080229,0.075667,504,1943,3.855159
3,18,120.388863,0.081156,0.075329,418,1953,4.672249


In [60]:
# Training the selected product-cause topic model

selected_pure_product_topic_count = 15

pure_product_cause_nmf_model = NMF(
    n_components = selected_pure_product_topic_count,
    init = "nndsvda",
    random_state = 42,
    max_iter = 400
)

pure_product_cause_document_topic_matrix = pure_product_cause_nmf_model.fit_transform(
    pure_product_cause_dissatisfaction_tfidf_matrix
)

pure_product_cause_topic_term_matrix = pure_product_cause_nmf_model.components_
pure_product_cause_feature_names = np.array(
    pure_product_cause_tfidf_vectorizer.get_feature_names_out()
)

pure_product_cause_comparison_tfidf_matrix = pure_product_cause_tfidf_vectorizer.transform(
    pure_product_cause_comparison_dataframe["pure_product_cause_text"]
)

pure_product_cause_comparison_topic_matrix = pure_product_cause_nmf_model.transform(
    pure_product_cause_comparison_tfidf_matrix
)

pure_product_cause_dissatisfaction_dataframe = pure_product_cause_dissatisfaction_dataframe.reset_index(drop = True)
pure_product_cause_comparison_dataframe = pure_product_cause_comparison_dataframe.reset_index(drop = True)

pure_product_cause_dissatisfaction_dataframe["topic_id"] = pure_product_cause_document_topic_matrix.argmax(axis = 1)
pure_product_cause_dissatisfaction_dataframe["topic_strength"] = pure_product_cause_document_topic_matrix.max(axis = 1)

pure_product_cause_comparison_dataframe["topic_id"] = pure_product_cause_comparison_topic_matrix.argmax(axis = 1)
pure_product_cause_comparison_dataframe["topic_strength"] = pure_product_cause_comparison_topic_matrix.max(axis = 1)

pure_product_cause_topic_rows = []

for topic_index, topic_weights in enumerate(pure_product_cause_topic_term_matrix):
    top_term_indices = topic_weights.argsort()[ : : -1][ : 15]
    pure_product_cause_topic_rows.append({
        "topic_id": topic_index,
        "top_terms": ", ".join(pure_product_cause_feature_names[top_term_indices])
    })

pure_product_cause_topic_terms_dataframe = pd.DataFrame(pure_product_cause_topic_rows)

display(pure_product_cause_topic_terms_dataframe)

,topic_id,top_terms
0,0,"router, wifi, network, internet, modem, netgea..."
1,1,"ear, headphone, earbuds, ears, buds, earbud, s..."
2,2,"ipad, not_fit, protect, color, cases, fits, ca..."
3,3,"charge phone, cords, chargers, charger, not_ch..."
4,4,"drive, drives, files, disk, data, hard_drive, ..."
5,5,"mouse, keyboard, keys, logitech, mouse work, k..."
6,6,"screws, mount, screw, mounting, bracket, tight..."
7,7,"camera, camera work, motion, recording, view, ..."
8,8,"speaker, sound_quality, music, bluetooth, bass..."
9,9,"hdmi, monitor, hdmi cable, display, adapter, f..."


In [61]:
# Assigning product-cause topics across dissatisfied and satisfied documents

pure_product_cause_topic_assignment_dataframe = pd.concat(
    [
        pure_product_cause_dissatisfaction_dataframe.assign(cohort_role = "Dissatisfaction"),
        pure_product_cause_comparison_dataframe.assign(cohort_role = "Satisfaction")
    ],
    ignore_index = True
)

pure_product_cause_topic_summary_dataframe = (
    pure_product_cause_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean")
    )
    .reset_index()
)

pure_product_cause_cohort_totals_dataframe = (
    pure_product_cause_topic_assignment_dataframe
    .groupby("cohort_role", dropna = False)
    .agg(
        total_documents = ("document_id", "count"),
        total_sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

pure_product_cause_topic_summary_dataframe = pure_product_cause_topic_summary_dataframe.merge(
    pure_product_cause_cohort_totals_dataframe,
    on = "cohort_role",
    how = "left"
)

pure_product_cause_topic_summary_dataframe["document_share"] = (
    pure_product_cause_topic_summary_dataframe["document_count"] /
    pure_product_cause_topic_summary_dataframe["total_documents"]
)

pure_product_cause_topic_summary_pivot_dataframe = (
    pure_product_cause_topic_summary_dataframe[
        [
            "cohort_role",
            "topic_id",
            "document_count",
            "sampled_reviews",
            "document_share",
            "average_topic_strength"
        ]
    ]
    .pivot(index = "topic_id", columns = "cohort_role")
)

pure_product_cause_topic_summary_pivot_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in pure_product_cause_topic_summary_pivot_dataframe.columns
]

pure_product_cause_topic_summary_pivot_dataframe = (
    pure_product_cause_topic_summary_pivot_dataframe.reset_index()
)

pure_product_cause_topic_summary_pivot_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] = (
    (pure_product_cause_topic_summary_pivot_dataframe["document_share_dissatisfaction"] + 1e-6) /
    (pure_product_cause_topic_summary_pivot_dataframe["document_share_satisfaction"] + 1e-6)
)

pure_product_cause_topic_summary_pivot_dataframe["document_share_gap"] = (
    pure_product_cause_topic_summary_pivot_dataframe["document_share_dissatisfaction"] -
    pure_product_cause_topic_summary_pivot_dataframe["document_share_satisfaction"]
)

pure_product_cause_topic_driver_dataframe = (
    pure_product_cause_topic_summary_pivot_dataframe
    .merge(pure_product_cause_topic_terms_dataframe, on = "topic_id", how = "left")
    .sort_values(
        by = ["document_share_lift_dissatisfaction_vs_satisfaction", "document_share_gap"],
        ascending = [False, False]
    )
    .reset_index(drop = True)
)

display(pure_product_cause_topic_driver_dataframe)

,topic_id,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,document_share_dissatisfaction,document_share_satisfaction,average_topic_strength_dissatisfaction,average_topic_strength_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,document_share_gap,top_terms
0,14,987,708,101346,57240,0.058565,0.038562,0.072934,0.065367,1.518712,0.020003,"charger, charge battery, hold charge, battery ..."
1,3,1235,1183,163189,131865,0.073281,0.064434,0.087539,0.068773,1.137305,0.008847,"charge phone, cords, chargers, charger, not_ch..."
2,5,948,935,124455,110844,0.056251,0.050926,0.090618,0.077804,1.104565,0.005325,"mouse, keyboard, keys, logitech, mouse work, k..."
3,1,1943,1923,285840,253556,0.115291,0.104739,0.088229,0.087839,1.100750,0.010552,"ear, headphone, earbuds, ears, buds, earbud, s..."
4,0,829,822,126048,113848,0.049190,0.044771,0.061512,0.053579,1.098695,0.004419,"router, wifi, network, internet, modem, netgea..."
5,11,889,899,112541,100945,0.052750,0.048965,0.079903,0.072212,1.077301,0.003785,"remote, remote work, remotes, buttons, remote ..."
6,10,504,510,50688,48753,0.029906,0.027778,0.116487,0.096013,1.076601,0.002128,"bubbles, screen_protector, protectors, screen ..."
7,4,1563,1588,185105,172334,0.092743,0.086492,0.077481,0.064503,1.072269,0.006251,"drive, drives, files, disk, data, hard_drive, ..."
8,9,1115,1172,117624,117609,0.066160,0.063834,0.077246,0.069280,1.036436,0.002326,"hdmi, monitor, hdmi cable, display, adapter, f..."
9,13,559,591,65891,67434,0.033169,0.032190,0.104388,0.099674,1.030432,0.000980,"antenna, stations, reception, channels, miles,..."


In [62]:
# Comparing broad, root-cause and product-cause topic models

model_comparison_summary_dataframe = pd.DataFrame({
    "model_name": [
        "full_topic_model",
        "root_cause_topic_model",
        "pure_product_cause_topic_model"
    ],
    "topic_count": [
        selected_topic_count,
        root_cause_selected_topic_count,
        selected_pure_product_topic_count
    ],
    "dissatisfaction_document_count": [
        len(full_dissatisfaction_topic_review_dataframe),
        len(root_cause_dissatisfaction_dataframe),
        len(pure_product_cause_dissatisfaction_dataframe)
    ],
    "top_signal_lift": [
        topic_comparison_summary_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"].max(),
        root_cause_topic_driver_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"].max(),
        pure_product_cause_topic_driver_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"].max()
    ],
    "median_signal_lift": [
        topic_comparison_summary_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"].median(),
        root_cause_topic_driver_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"].median(),
        pure_product_cause_topic_driver_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"].median()
    ]
})

display(model_comparison_summary_dataframe)

,model_name,topic_count,dissatisfaction_document_count,top_signal_lift,median_signal_lift
0,full_topic_model,15,16853,1.560507,1.071390
1,root_cause_topic_model,15,16853,1.515369,1.072840
2,pure_product_cause_topic_model,15,16853,1.518712,1.072269


In [63]:
# Measuring confidence for product-cause topic assignments

pure_product_cause_topic_assignment_dataframe["topic_margin"] = compute_assignment_margin(
    np.vstack([
        pure_product_cause_document_topic_matrix,
        pure_product_cause_comparison_topic_matrix
    ])
)

pure_product_cause_topic_assignment_dataframe["is_topic_core_document"] = (
    (pure_product_cause_topic_assignment_dataframe["topic_strength"] >= 0.06)
    & (pure_product_cause_topic_assignment_dataframe["topic_margin"] >= 0.03)
)

pure_product_cause_topic_core_summary_dataframe = (
    pure_product_cause_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        topic_core_document_count = ("is_topic_core_document", "sum"),
        average_topic_strength = ("topic_strength", "mean"),
        median_topic_strength = ("topic_strength", "median"),
        average_topic_margin = ("topic_margin", "mean"),
        median_topic_margin = ("topic_margin", "median")
    )
    .reset_index()
)

pure_product_cause_topic_core_summary_dataframe["topic_core_ratio"] = (
    pure_product_cause_topic_core_summary_dataframe["topic_core_document_count"] /
    pure_product_cause_topic_core_summary_dataframe["document_count"].replace(0, np.nan)
)

display(
    pure_product_cause_topic_core_summary_dataframe.sort_values(
        ["cohort_role", "topic_id"]
    )
)

,cohort_role,topic_id,document_count,topic_core_document_count,average_topic_strength,median_topic_strength,average_topic_margin,median_topic_margin,topic_core_ratio
0,Dissatisfaction,0,829,512,0.061512,0.066629,0.048766,0.055668,0.617612
1,Dissatisfaction,1,1943,1533,0.088229,0.090027,0.066252,0.068180,0.788986
2,Dissatisfaction,2,1649,1082,0.075505,0.074454,0.054879,0.055389,0.656155
3,Dissatisfaction,3,1235,749,0.087539,0.091018,0.068660,0.073553,0.606478
4,Dissatisfaction,4,1563,914,0.077481,0.075784,0.056516,0.045806,0.584773
5,Dissatisfaction,5,948,702,0.090618,0.095105,0.070006,0.077528,0.740506
6,Dissatisfaction,6,1244,615,0.069644,0.062694,0.048504,0.036919,0.494373
7,Dissatisfaction,7,1340,820,0.076480,0.074334,0.052507,0.047110,0.611940
8,Dissatisfaction,8,1508,640,0.062071,0.062093,0.033923,0.031151,0.424403
9,Dissatisfaction,9,1115,725,0.077246,0.082235,0.051858,0.054196,0.650224


In [64]:
# Reviewing product-cause topic distribution by segment

pure_product_cause_segment_distribution_dataframe = (
    pure_product_cause_topic_assignment_dataframe
    .groupby(
        ["cohort_role", "topic_id", "price_band", "issue_priority_level"],
        dropna = False
    )
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum"),
        average_topic_strength = ("topic_strength", "mean")
    )
    .reset_index()
)

pure_product_cause_dissatisfaction_segment_distribution_dataframe = (
    pure_product_cause_segment_distribution_dataframe.loc[
        pure_product_cause_segment_distribution_dataframe["cohort_role"] == "Dissatisfaction"
    ].copy()
)

pure_product_cause_dissatisfaction_segment_distribution_dataframe["segment_document_share_within_topic"] = (
    pure_product_cause_dissatisfaction_segment_distribution_dataframe["document_count"] /
    pure_product_cause_dissatisfaction_segment_distribution_dataframe.groupby("topic_id")["document_count"].transform("sum")
)

pure_product_cause_topic_breadth_summary_dataframe = (
    pure_product_cause_dissatisfaction_segment_distribution_dataframe
    .groupby("topic_id", dropna = False)
    .agg(
        covered_segment_count = ("segment_document_share_within_topic", "count"),
        segment_count_above_05_share = (
            "segment_document_share_within_topic",
            lambda values: (values >= 0.05).sum()
        ),
        segment_count_above_10_share = (
            "segment_document_share_within_topic",
            lambda values: (values >= 0.10).sum()
        ),
        max_segment_share = ("segment_document_share_within_topic", "max"),
        concentration_hhi = (
            "segment_document_share_within_topic",
            lambda values: np.square(values).sum()
        )
    )
    .reset_index()
)

pure_product_cause_segment_reporting_eligibility_dataframe = (
    pure_product_cause_segment_distribution_dataframe
    .pivot(
        index = ["topic_id", "price_band", "issue_priority_level"],
        columns = "cohort_role",
        values = ["document_count", "sampled_reviews"]
    )
)

pure_product_cause_segment_reporting_eligibility_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in pure_product_cause_segment_reporting_eligibility_dataframe.columns
]

pure_product_cause_segment_reporting_eligibility_dataframe = (
    pure_product_cause_segment_reporting_eligibility_dataframe
    .reset_index()
    .fillna(0)
)

pure_product_cause_segment_reporting_eligibility_dataframe["is_segment_reporting_eligible"] = (
    (pure_product_cause_segment_reporting_eligibility_dataframe["document_count_dissatisfaction"] >= 25)
    & (pure_product_cause_segment_reporting_eligibility_dataframe["document_count_satisfaction"] >= 25)
    & (pure_product_cause_segment_reporting_eligibility_dataframe["sampled_reviews_dissatisfaction"] >= 3000)
    & (pure_product_cause_segment_reporting_eligibility_dataframe["sampled_reviews_satisfaction"] >= 3000)
)

display(pure_product_cause_topic_breadth_summary_dataframe.sort_values("topic_id"))
display(
    pure_product_cause_segment_reporting_eligibility_dataframe.sort_values(
        ["topic_id", "price_band", "issue_priority_level"]
    )
)

/var/folders/f1/ws24m5bs3cx9_1d_nclvhd1w0000gn/T/ipykernel_51314/1973121574.py:67: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(0)


,topic_id,covered_segment_count,segment_count_above_05_share,segment_count_above_10_share,max_segment_share,concentration_hhi
0,0,9,6,5,0.249698,0.161432
1,1,9,8,6,0.206382,0.135297
2,2,9,4,3,0.402668,0.266479
3,3,9,4,3,0.460729,0.300784
4,4,9,6,4,0.212412,0.163149
5,5,9,7,5,0.213080,0.138477
6,6,9,4,4,0.303859,0.210864
7,7,9,6,4,0.297015,0.182403
8,8,9,6,4,0.235411,0.159585
9,9,9,7,4,0.257399,0.155804


,topic_id,price_band,issue_priority_level,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,is_segment_reporting_eligible
0,0,Budget,Monitor,36,79,4727,9939,True
1,0,Budget,Priority review,16,21,3733,4831,False
2,0,Lower mid,Monitor,94,88,12647,8681,True
3,0,Lower mid,Priority review,44,52,6897,9354,True
4,0,Premium,Monitor,207,171,31599,21993,True
5,0,Premium,Priority review,108,109,16848,17030,True
6,0,Upper mid,Monitor,161,141,23862,15935,True
7,0,Upper mid,Priority review,131,135,20960,23164,True
8,0,Very premium,Monitor,32,26,4775,2921,False
9,1,Budget,Monitor,196,214,22508,21397,True


In [65]:
# Comparing product-cause topic exposure across price bands
pure_product_cause_price_band_comparison_dataframe = (
    pure_product_cause_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id", "price_band"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

pure_product_cause_price_band_totals_dataframe = (
    pure_product_cause_price_band_comparison_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        total_documents = ("document_count", "sum")
    )
    .reset_index()
)

pure_product_cause_price_band_comparison_dataframe = pure_product_cause_price_band_comparison_dataframe.merge(
    pure_product_cause_price_band_totals_dataframe,
    on = ["cohort_role", "topic_id"],
    how = "left"
)

pure_product_cause_price_band_comparison_dataframe["document_share_within_topic"] = (
    pure_product_cause_price_band_comparison_dataframe["document_count"] /
    pure_product_cause_price_band_comparison_dataframe["total_documents"].replace(0, np.nan)
)

pure_product_cause_price_band_comparison_dataframe = (
    pure_product_cause_price_band_comparison_dataframe
    .pivot_table(
        index = ["topic_id", "price_band"],
        columns = "cohort_role",
        values = ["document_count", "sampled_reviews", "document_share_within_topic"],
        fill_value = 0
    )
)

pure_product_cause_price_band_comparison_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in pure_product_cause_price_band_comparison_dataframe.columns
]

pure_product_cause_price_band_comparison_dataframe = (
    pure_product_cause_price_band_comparison_dataframe
    .reset_index()
    .infer_objects(copy = False)
)

pure_product_cause_price_band_comparison_dataframe["price_band_document_share_lift_dissatisfaction_vs_satisfaction"] = (
    (pure_product_cause_price_band_comparison_dataframe["document_share_within_topic_dissatisfaction"] + 1e-6) /
    (pure_product_cause_price_band_comparison_dataframe["document_share_within_topic_satisfaction"] + 1e-6)
)

pure_product_cause_priority_comparison_dataframe = (
    pure_product_cause_topic_assignment_dataframe
    .groupby(["cohort_role", "topic_id", "issue_priority_level"], dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        sampled_reviews = ("sampled_review_count", "sum")
    )
    .reset_index()
)

pure_product_cause_priority_totals_dataframe = (
    pure_product_cause_priority_comparison_dataframe
    .groupby(["cohort_role", "topic_id"], dropna = False)
    .agg(
        total_documents = ("document_count", "sum")
    )
    .reset_index()
)

pure_product_cause_priority_comparison_dataframe = pure_product_cause_priority_comparison_dataframe.merge(
    pure_product_cause_priority_totals_dataframe,
    on = ["cohort_role", "topic_id"],
    how = "left"
)

pure_product_cause_priority_comparison_dataframe["document_share_within_topic"] = (
    pure_product_cause_priority_comparison_dataframe["document_count"] /
    pure_product_cause_priority_comparison_dataframe["total_documents"].replace(0, np.nan)
)

pure_product_cause_priority_comparison_dataframe = (
    pure_product_cause_priority_comparison_dataframe
    .pivot_table(
        index = ["topic_id", "issue_priority_level"],
        columns = "cohort_role",
        values = ["document_count", "sampled_reviews", "document_share_within_topic"],
        fill_value = 0
    )
)

pure_product_cause_priority_comparison_dataframe.columns = [
    f"{metric_name}_{cohort_name.lower()}"
    for metric_name, cohort_name in pure_product_cause_priority_comparison_dataframe.columns
]

pure_product_cause_priority_comparison_dataframe = (
    pure_product_cause_priority_comparison_dataframe
    .reset_index()
    .infer_objects(copy = False)
)

display(pure_product_cause_price_band_comparison_dataframe.sort_values(["topic_id", "price_band"]))
display(pure_product_cause_priority_comparison_dataframe.sort_values(["topic_id", "issue_priority_level"]))

,topic_id,price_band,document_count_dissatisfaction,document_count_satisfaction,document_share_within_topic_dissatisfaction,document_share_within_topic_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,price_band_document_share_lift_dissatisfaction_vs_satisfaction
0,0,Budget,52.0,100.0,0.062726,0.121655,8460.0,14770.0,0.515613
1,0,Lower mid,138.0,140.0,0.166466,0.170316,19544.0,18035.0,0.977391
2,0,Premium,315.0,280.0,0.379976,0.340633,48447.0,39023.0,1.115500
3,0,Upper mid,292.0,276.0,0.352232,0.335766,44822.0,39099.0,1.049037
4,0,Very premium,32.0,26.0,0.038601,0.031630,4775.0,2921.0,1.220370
5,1,Budget,359.0,368.0,0.184766,0.191368,48212.0,42712.0,0.965502
6,1,Lower mid,559.0,532.0,0.287699,0.276651,87224.0,74092.0,1.039936
7,1,Premium,404.0,404.0,0.207926,0.210088,63138.0,56432.0,0.989707
8,1,Upper mid,578.0,573.0,0.297478,0.297972,82948.0,76212.0,0.998343
9,1,Very premium,43.0,46.0,0.022131,0.023921,4318.0,4108.0,0.925164


,topic_id,issue_priority_level,document_count_dissatisfaction,document_count_satisfaction,document_share_within_topic_dissatisfaction,document_share_within_topic_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction
0,0,Monitor,530.0,505.0,0.639324,0.614355,77610.0,59469.0
1,0,Priority review,299.0,317.0,0.360676,0.385645,48438.0,54379.0
2,1,Monitor,1198.0,1176.0,0.616572,0.611544,163578.0,138039.0
3,1,Priority review,745.0,747.0,0.383428,0.388456,122262.0,115517.0
4,2,Monitor,1431.0,2016.0,0.867799,0.891641,120747.0,179741.0
5,2,Priority review,218.0,245.0,0.132201,0.108359,31126.0,35578.0
6,3,Monitor,880.0,874.0,0.712551,0.738800,105385.0,84396.0
7,3,Priority review,355.0,309.0,0.287449,0.261200,57804.0,47469.0
8,4,Monitor,1307.0,1344.0,0.836212,0.846348,149214.0,139229.0
9,4,Priority review,256.0,244.0,0.163788,0.153652,35891.0,33105.0


In [66]:
# Summarizing product-cause topics in premium price bands

pure_product_premium_price_bands = {"Premium", "Very premium"}

pure_product_premium_topic_summary_dataframe = (
    pure_product_cause_price_band_comparison_dataframe.assign(
        is_premium_band = lambda dataframe: dataframe["price_band"].isin(pure_product_premium_price_bands)
    )
    .groupby("topic_id", dropna = False)
    .agg(
        premium_document_count_dissatisfaction = (
            "document_count_dissatisfaction",
            lambda values: values[
                pure_product_cause_price_band_comparison_dataframe.loc[values.index, "price_band"].isin(pure_product_premium_price_bands)
            ].sum()
        ),
        premium_document_count_satisfaction = (
            "document_count_satisfaction",
            lambda values: values[
                pure_product_cause_price_band_comparison_dataframe.loc[values.index, "price_band"].isin(pure_product_premium_price_bands)
            ].sum()
        ),
        total_document_count_dissatisfaction = ("document_count_dissatisfaction", "sum"),
        total_document_count_satisfaction = ("document_count_satisfaction", "sum")
    )
    .reset_index()
)

pure_product_premium_topic_summary_dataframe["premium_document_share_dissatisfaction"] = (
    pure_product_premium_topic_summary_dataframe["premium_document_count_dissatisfaction"] /
    pure_product_premium_topic_summary_dataframe["total_document_count_dissatisfaction"].replace(0, np.nan)
)

pure_product_premium_topic_summary_dataframe["premium_document_share_satisfaction"] = (
    pure_product_premium_topic_summary_dataframe["premium_document_count_satisfaction"] /
    pure_product_premium_topic_summary_dataframe["total_document_count_satisfaction"].replace(0, np.nan)
)

pure_product_premium_topic_summary_dataframe["premium_share_lift_dissatisfaction_vs_satisfaction"] = (
    (pure_product_premium_topic_summary_dataframe["premium_document_share_dissatisfaction"] + 1e-6) /
    (pure_product_premium_topic_summary_dataframe["premium_document_share_satisfaction"] + 1e-6)
)

pure_product_priority_topic_summary_dataframe = (
    pure_product_cause_priority_comparison_dataframe.loc[
        pure_product_cause_priority_comparison_dataframe["issue_priority_level"] == "Priority review"
    ].copy()
)

pure_product_priority_topic_summary_dataframe["priority_share_lift_dissatisfaction_vs_satisfaction"] = (
    (pure_product_priority_topic_summary_dataframe["document_share_within_topic_dissatisfaction"] + 1e-6) /
    (pure_product_priority_topic_summary_dataframe["document_share_within_topic_satisfaction"] + 1e-6)
)

display(pure_product_premium_topic_summary_dataframe.sort_values("topic_id"))
display(pure_product_priority_topic_summary_dataframe.sort_values("topic_id"))

,topic_id,premium_document_count_dissatisfaction,premium_document_count_satisfaction,total_document_count_dissatisfaction,total_document_count_satisfaction,premium_document_share_dissatisfaction,premium_document_share_satisfaction,premium_share_lift_dissatisfaction_vs_satisfaction
0,0,347.0,306.0,829.0,822.0,0.418577,0.372263,1.124411
1,1,447.0,450.0,1943.0,1923.0,0.230057,0.234009,0.983109
2,2,57.0,124.0,1649.0,2261.0,0.034566,0.054843,0.630286
3,3,15.0,22.0,1235.0,1183.0,0.012146,0.018597,0.653129
4,4,503.0,539.0,1563.0,1588.0,0.321817,0.339421,0.948136
5,5,177.0,194.0,948.0,935.0,0.186709,0.207487,0.899860
6,6,213.0,287.0,1244.0,1655.0,0.171222,0.173414,0.987360
7,7,712.0,952.0,1340.0,1753.0,0.531343,0.543069,0.978408
8,8,533.0,657.0,1508.0,1767.0,0.353448,0.371817,0.950598
9,9,337.0,344.0,1115.0,1172.0,0.302242,0.293515,1.029732


,topic_id,issue_priority_level,document_count_dissatisfaction,document_count_satisfaction,document_share_within_topic_dissatisfaction,document_share_within_topic_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,priority_share_lift_dissatisfaction_vs_satisfaction
1,0,Priority review,299.0,317.0,0.360676,0.385645,48438.0,54379.0,0.935253
3,1,Priority review,745.0,747.0,0.383428,0.388456,122262.0,115517.0,0.987057
5,2,Priority review,218.0,245.0,0.132201,0.108359,31126.0,35578.0,1.220027
7,3,Priority review,355.0,309.0,0.287449,0.261200,57804.0,47469.0,1.100494
9,4,Priority review,256.0,244.0,0.163788,0.153652,35891.0,33105.0,1.065961
11,5,Priority review,316.0,330.0,0.333333,0.352941,48517.0,51861.0,0.944445
13,6,Priority review,145.0,154.0,0.116559,0.093051,19263.0,21020.0,1.252633
15,7,Priority review,420.0,461.0,0.313433,0.262978,67469.0,73162.0,1.191860
17,8,Priority review,384.0,434.0,0.254642,0.245614,55153.0,63448.0,1.036756
19,9,Priority review,247.0,243.0,0.221525,0.207338,35810.0,36341.0,1.068423


In [67]:
# Combining product-cause topic evidence for interpretation
pure_product_topic_evidence_dataframe = (
    pure_product_cause_topic_driver_dataframe
    .merge(
        pure_product_cause_topic_breadth_summary_dataframe,
        on = "topic_id",
        how = "left"
    )
    .merge(
        pure_product_cause_topic_core_summary_dataframe.loc[
            pure_product_cause_topic_core_summary_dataframe["cohort_role"] == "Dissatisfaction",
            ["topic_id", "topic_core_ratio", "average_topic_margin", "median_topic_margin"]
        ].rename(
            columns = {
                "average_topic_margin": "average_topic_margin_dissatisfaction",
                "median_topic_margin": "median_topic_margin_dissatisfaction"
            }
        ),
        on = "topic_id",
        how = "left"
    )
    .merge(
        pure_product_premium_topic_summary_dataframe[
            [
                "topic_id",
                "premium_document_share_dissatisfaction",
                "premium_document_share_satisfaction",
                "premium_share_lift_dissatisfaction_vs_satisfaction"
            ]
        ],
        on = "topic_id",
        how = "left"
    )
    .merge(
        pure_product_priority_topic_summary_dataframe[
            [
                "topic_id",
                "document_share_within_topic_dissatisfaction",
                "document_share_within_topic_satisfaction",
                "priority_share_lift_dissatisfaction_vs_satisfaction"
            ]
        ].rename(
            columns = {
                "document_share_within_topic_dissatisfaction": "priority_document_share_dissatisfaction",
                "document_share_within_topic_satisfaction": "priority_document_share_satisfaction"
            }
        ),
        on = "topic_id",
        how = "left"
    )
    .merge(
        pure_product_cause_segment_reporting_eligibility_dataframe
        .groupby("topic_id", dropna = False)
        .agg(
            eligible_segment_count = ("is_segment_reporting_eligible", "sum"),
            total_segment_count = ("is_segment_reporting_eligible", "count")
        )
        .reset_index(),
        on = "topic_id",
        how = "left"
    )
)

pure_product_topic_evidence_dataframe["eligible_segment_ratio"] = (
    pure_product_topic_evidence_dataframe["eligible_segment_count"] /
    pure_product_topic_evidence_dataframe["total_segment_count"].replace(0, np.nan)
)

pure_product_topic_evidence_dataframe["breadth_type"] = np.select(
    [
        pure_product_topic_evidence_dataframe["concentration_hhi"] >= 0.25,
        pure_product_topic_evidence_dataframe["segment_count_above_05_share"] >= 6
    ],
    [
        "Concentrated",
        "Broad"
    ],
    default = "Moderate"
)

pure_product_topic_evidence_dataframe["signal_strength"] = np.select(
    [
        pure_product_topic_evidence_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.12,
        pure_product_topic_evidence_dataframe["document_share_lift_dissatisfaction_vs_satisfaction"] >= 1.05
    ],
    [
        "Strong",
        "Moderate"
    ],
    default = "Weak"
)

pure_product_topic_evidence_dataframe["confidence_level"] = np.select(
    [
        (
            pure_product_topic_evidence_dataframe["average_topic_margin_dissatisfaction"] >= 0.06
        ) & (
            pure_product_topic_evidence_dataframe["topic_core_ratio"] >= 0.60
        ),
        (
            pure_product_topic_evidence_dataframe["average_topic_margin_dissatisfaction"] >= 0.04
        ) & (
            pure_product_topic_evidence_dataframe["topic_core_ratio"] >= 0.45
        )
    ],
    [
        "High",
        "Moderate"
    ],
    default = "Watch"
)

pure_product_topic_evidence_dataframe["premium_exposure_type"] = np.select(
    [
        pure_product_topic_evidence_dataframe["premium_share_lift_dissatisfaction_vs_satisfaction"] >= 1.08,
        pure_product_topic_evidence_dataframe["premium_share_lift_dissatisfaction_vs_satisfaction"] <= 0.92
    ],
    [
        "Premium-tilted",
        "Non-premium-tilted"
    ],
    default = "Balanced"
)

pure_product_topic_evidence_dataframe["priority_exposure_type"] = np.select(
    [
        pure_product_topic_evidence_dataframe["priority_share_lift_dissatisfaction_vs_satisfaction"] >= 1.08,
        pure_product_topic_evidence_dataframe["priority_share_lift_dissatisfaction_vs_satisfaction"] <= 0.92
    ],
    [
        "Priority-product-tilted",
        "Monitor-product-tilted"
    ],
    default = "Balanced"
)

display(
    pure_product_topic_evidence_dataframe.sort_values(
        by = [
            "signal_strength",
            "confidence_level",
            "eligible_segment_ratio",
            "document_share_lift_dissatisfaction_vs_satisfaction",
            "document_count_dissatisfaction"
        ],
        ascending = [False, False, False, False, False]
    )
)

,topic_id,document_count_dissatisfaction,document_count_satisfaction,sampled_reviews_dissatisfaction,sampled_reviews_satisfaction,document_share_dissatisfaction,document_share_satisfaction,average_topic_strength_dissatisfaction,average_topic_strength_satisfaction,document_share_lift_dissatisfaction_vs_satisfaction,document_share_gap,top_terms,covered_segment_count,segment_count_above_05_share,segment_count_above_10_share,max_segment_share,concentration_hhi,topic_core_ratio,average_topic_margin_dissatisfaction,median_topic_margin_dissatisfaction,premium_document_share_dissatisfaction,premium_document_share_satisfaction,premium_share_lift_dissatisfaction_vs_satisfaction,priority_document_share_dissatisfaction,priority_document_share_satisfaction,priority_share_lift_dissatisfaction_vs_satisfaction,eligible_segment_count,total_segment_count,eligible_segment_ratio,breadth_type,signal_strength,confidence_level,premium_exposure_type,priority_exposure_type
11,8,1508,1767,186343,207786,0.089480,0.096242,0.062071,0.061152,0.929738,-0.006762,"speaker, sound_quality, music, bluetooth, bass...",9,6,4,0.235411,0.159585,0.424403,0.033923,0.031151,0.353448,0.371817,0.950598,0.254642,0.245614,1.036756,9,9,1.000000,Broad,Weak,Watch,Balanced,Balanced
8,9,1115,1172,117624,117609,0.066160,0.063834,0.077246,0.069280,1.036436,0.002326,"hdmi, monitor, hdmi cable, display, adapter, f...",9,7,4,0.257399,0.155804,0.650224,0.051858,0.054196,0.302242,0.293515,1.029732,0.221525,0.207338,1.068423,9,9,1.000000,Broad,Weak,Moderate,Balanced,Balanced
12,7,1340,1753,154727,181569,0.079511,0.095479,0.076480,0.068051,0.832759,-0.015968,"camera, camera work, motion, recording, view, ...",9,6,4,0.297015,0.182403,0.611940,0.052507,0.047110,0.531343,0.543069,0.978408,0.313433,0.262978,1.191860,8,9,0.888889,Broad,Weak,Moderate,Balanced,Priority-product-tilted
13,6,1244,1655,106045,142748,0.073815,0.090142,0.069644,0.059319,0.818877,-0.016327,"screws, mount, screw, mounting, bracket, tight...",9,4,4,0.303859,0.210864,0.494373,0.048504,0.036919,0.171222,0.173414,0.987360,0.116559,0.093051,1.252633,7,9,0.777778,Moderate,Weak,Moderate,Balanced,Priority-product-tilted
14,2,1649,2261,151873,215319,0.097846,0.123148,0.075505,0.065433,0.794541,-0.025302,"ipad, not_fit, protect, color, cases, fits, ca...",9,4,3,0.402668,0.266479,0.656155,0.054879,0.055389,0.034566,0.054843,0.630286,0.132201,0.108359,1.220027,6,9,0.666667,Concentrated,Weak,Moderate,Non-premium-tilted,Priority-product-tilted
9,13,559,591,65891,67434,0.033169,0.032190,0.104388,0.099674,1.030432,0.000980,"antenna, stations, reception, channels, miles,...",9,8,4,0.254025,0.148019,0.613596,0.076557,0.046962,0.262970,0.252115,1.043054,0.343470,0.319797,1.074027,8,9,0.888889,Broad,Weak,High,Balanced,Balanced
10,12,540,593,61628,63106,0.032042,0.032298,0.113165,0.108941,0.992052,-0.000257,"fitbit, band, wrist, watch, bands, heart rate,...",9,7,6,0.220370,0.140384,0.698148,0.091330,0.116824,0.344444,0.328836,1.047464,0.375926,0.359191,1.046592,7,9,0.777778,Broad,Weak,High,Balanced,Balanced
0,14,987,708,101346,57240,0.058565,0.038562,0.072934,0.065367,1.518712,0.020003,"charger, charge battery, hold charge, battery ...",9,7,4,0.244174,0.161470,0.418440,0.050471,0.025075,0.219858,0.117232,1.875409,0.231003,0.183616,1.258077,6,9,0.666667,Broad,Strong,Watch,Premium-tilted,Priority-product-tilted
1,3,1235,1183,163189,131865,0.073281,0.064434,0.087539,0.068773,1.137305,0.008847,"charge phone, cords, chargers, charger, not_ch...",9,4,3,0.460729,0.300784,0.606478,0.068660,0.073553,0.012146,0.018597,0.653129,0.287449,0.261200,1.100494,5,9,0.555556,Concentrated,Strong,High,Non-premium-tilted,Priority-product-tilted
5,11,889,899,112541,100945,0.052750,0.048965,0.079903,0.072212,1.077301,0.003785,"remote, remote work, remotes, buttons, remote ...",9,8,3,0.205849,0.144348,0.652418,0.057704,0.060459,0.363330,0.347052,1.046901,0.292463,0.275862,1.060180,9,9,1.000000,Broad,Moderate,Moderate,Balanced,Balanced


In [68]:
# Selecting representative product-cause documents for manual review
pure_product_representative_documents_dataframe = (
    pure_product_cause_topic_assignment_dataframe[
        [
            "cohort_role",
            "topic_id",
            "topic_strength",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "sampled_review_count",
            "pure_product_cause_text"
        ]
    ]
    .sort_values(
        ["cohort_role", "topic_id", "topic_strength"],
        ascending = [True, True, False]
    )
    .groupby(["cohort_role", "topic_id"], group_keys = False)
    .head(5)
    .reset_index(drop = True)
)

pure_product_representative_documents_dataframe["text_preview"] = (
    pure_product_representative_documents_dataframe["pure_product_cause_text"]
    .str.slice(0, 600)
)

display(
    pure_product_representative_documents_dataframe[
        [
            "cohort_role",
            "topic_id",
            "topic_strength",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "sampled_review_count",
            "text_preview"
        ]
    ]
)

,cohort_role,topic_id,topic_strength,price_band,issue_priority_level,product_display_name,sampled_review_count,text_preview
0,Dissatisfaction,0,0.088021,Very premium,Monitor,NETGEAR Nighthawk X6 Smart Wi-Fi Router (R8000...,250,pile scrap simplest activities router pass dns...
1,Dissatisfaction,0,0.087991,Premium,Priority review,NETGEAR R7500 Nighthawk X4 AC2350 Dual Band Wi...,108,constantly reset router buy open sent bought v...
2,Dissatisfaction,0,0.086958,Premium,Monitor,NETGEAR Nighthawk X4S Smart WiFi Router (R7800...,250,piece crap mouth owning started continued conn...
3,Dissatisfaction,0,0.086019,Premium,Priority review,NETGEAR Nighthawk 4-Stream AX4 Wi-fi 6 Router ...,250,purchased ago start constantly disconnect wife...
4,Dissatisfaction,0,0.085944,Very premium,Monitor,NETGEAR Nighthawk X6 Smart Wi-Fi Router (R8000...,250,router cheapo linksys bought powerful router v...
5,Dissatisfaction,1,0.137328,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,work ago earbuds cut watching listening music ...
6,Dissatisfaction,1,0.137180,Upper mid,Monitor,TOZO NC2 Hybrid Active Noise Cancelling Wirele...,250,completely dead_on_arrival no_power whatsoever...
7,Dissatisfaction,1,0.135325,Premium,Priority review,Sony WF-SP800N Truly Wireless Sports In-Ear No...,121,sound_quality earpiece left earpiece try earpi...
8,Dissatisfaction,1,0.134889,Lower mid,Monitor,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,250,thought came case handy strap work try pair ip...
9,Dissatisfaction,1,0.134544,Lower mid,Monitor,MEE audio M6 X1 Wired In-Ear Sports Headphones...,250,sounds tin iphone hear kind awkward secure ear...


In [69]:
# Saving intermediate NLP outputs for reuse

checkpoint_dir = Path("notebook_checkpoints")
checkpoint_dir.mkdir(exist_ok = True)

checkpoint_objects = {
    "pure_product_cause_topic_assignment_dataframe": pure_product_cause_topic_assignment_dataframe,
    "review_boundary_token": review_boundary_token,
    "remedy_or_transaction_tokens": remedy_or_transaction_tokens,
    "delivery_or_fulfillment_tokens": delivery_or_fulfillment_tokens,
    "evaluative_outcome_tokens": evaluative_outcome_tokens,
}

with open(checkpoint_dir / "review_analysis_checkpoint.pkl", "wb") as file:
    pickle.dump(checkpoint_objects, file)

print("Checkpoint saved :", checkpoint_dir / "review_analysis_checkpoint.pkl")

Checkpoint saved : notebook_checkpoints/review_analysis_checkpoint.pkl


In [70]:
# Reloading saved NLP outputs for continued analysis

checkpoint_dir = Path("notebook_checkpoints")

with open(checkpoint_dir / "review_analysis_checkpoint.pkl", "rb") as file:
    checkpoint_objects = pickle.load(file)

pure_product_cause_topic_assignment_dataframe = checkpoint_objects["pure_product_cause_topic_assignment_dataframe"]
review_boundary_token = checkpoint_objects["review_boundary_token"]
remedy_or_transaction_tokens = checkpoint_objects["remedy_or_transaction_tokens"]
delivery_or_fulfillment_tokens = checkpoint_objects["delivery_or_fulfillment_tokens"]
evaluative_outcome_tokens = checkpoint_objects["evaluative_outcome_tokens"]

print("Checkpoint loaded.")
print(pure_product_cause_topic_assignment_dataframe.shape)

Checkpoint loaded.
(35213, 53)


In [71]:
# Defining text normalization and issue-bucket matching rules
def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).lower()).strip()

def split_document_reviews(value, boundary_token = review_boundary_token):
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split(boundary_token) if part.strip()]

excluded_rule_tokens = (
    remedy_or_transaction_tokens
    .union(delivery_or_fulfillment_tokens)
    .union(evaluative_outcome_tokens)
)

def build_rule_text(review_text):
    tokens = [
        token
        for token in str(review_text).split()
        if token not in excluded_rule_tokens
    ]
    return " ".join(tokens).strip()

def has_pattern(text_value, patterns):
    return any(re.search(pattern, text_value) for pattern in patterns)

rule_definitions = {
    "charging_power_failure": {
        "title_terms": [
            r"\bcharger\b", r"\bcharging cable\b", r"\bcharging cord\b",
            r"\bbattery charger\b", r"\brechargeable battery\b", r"\bpower bank\b",
            r"\bbattery\b"
        ],
        "review_terms": [
            r"\bcharger\b", r"\bbattery\b", r"\bcharging\b", r"\bcharge\b",
            r"\bpower adapter\b", r"\bcharging port\b"
        ],
        "failure_terms": [
            r"\bnot_charging\b", r"\bnot_charge\b", r"\bno_power\b",
            r"\bnot_turn_on\b", r"\bwill_not_turn_on\b",
            r"\bbattery_died\b", r"\bdead_on_arrival\b"
        ],
        "exclude_terms": [
            r"\baudio cable\b", r"\bav cable\b", r"\bstereo cable\b",
            r"\bcomponent cable\b", r"\bcomposite cable\b", r"\bs[\- ]?video\b",
            r"\brca\b", r"\bcoax\b", r"\bantenna\b", r"\breception\b",
            r"\bchannels\b", r"\bstations\b", r"\bbackpack\b", r"\bbag\b"
        ],
        "title_only": False
    },
    "connectivity_pairing_failure": {
        "title_terms": [
            r"\brouter\b", r"\bmodem\b", r"\bwifi\b", r"\bbluetooth\b",
            r"\bnetwork adapter\b", r"\bwireless adapter\b",
            r"\breceiver\b", r"\bdongle\b", r"\bmesh\b", r"\bextender\b"
        ],
        "review_terms": [
            r"\brouter\b", r"\bmodem\b", r"\bwifi\b", r"\bbluetooth\b",
            r"\bnetwork\b", r"\binternet\b", r"\breceiver\b", r"\bdongle\b",
            r"\bpair\b"
        ],
        "failure_terms": [
            r"\bnot_connect\b", r"\bnot_connecting\b", r"\bkeeps_disconnecting\b",
            r"\bnot_pair\b", r"\bnot_pairing\b", r"\bloses connection\b",
            r"\bdrops connection\b", r"\bdisconnect(?:ed|ing)?\b",
            r"\bpairing failed\b", r"\bconnection drops\b"
        ],
        "exclude_terms": [
            r"\bantenna\b", r"\breception\b", r"\bchannels\b", r"\bstations\b",
            r"\btripod\b", r"\bmount\b", r"\bstrap\b",
            r"\baudio cable\b", r"\bav cable\b", r"\bstereo cable\b",
            r"\bheadphone jack\b", r"\baux\b",
            r"\bhdmi arc\b", r"\barc\b"
        ],
        "title_only": False
    },
    "storage_data_reliability": {
        "title_terms": [
            r"\bssd\b", r"\bhard drive\b", r"\bhard_drive\b", r"\bflash drive\b",
            r"\bmemory card\b", r"\bmicro_sd\b", r"\bsd_card\b",
            r"\bexternal drive\b", r"\bdisk\b"
        ],
        "review_terms": [
            r"\bssd\b", r"\bhard_drive\b", r"\bdrive\b", r"\bdisk\b",
            r"\bdata\b", r"\bfiles\b", r"\bmicro_sd\b", r"\bsd_card\b",
            r"\bmemory card\b", r"\bflash drive\b"
        ],
        "failure_terms": [
            r"\bnot recognized\b", r"\bfailed recognize\b", r"\bcorrupt(?:ed)?\b",
            r"\bdata loss\b", r"\blost all\b", r"\bfiles missing\b",
            r"\bwould not format\b", r"\bnot readable\b", r"\bcan not be read\b",
            r"\bread error\b", r"\bdrive failed\b", r"\bcard failed\b",
            r"\bdead_on_arrival\b"
        ],
        "exclude_terms": [
            r"\bapp\b", r"\bsync\b", r"\bcloud\b"
        ],
        "title_only": False
    },
    "compatibility_fit_issue": {
        "title_terms": [
            r"\bcase\b", r"\badapter\b", r"\bhdmi\b", r"\bscreen protector\b",
            r"\bscreen_protector\b", r"\busb c\b", r"\busb a\b",
            r"\bmicro sd\b", r"\bsd card\b"
        ],
        "review_terms": [
            r"\bcase\b", r"\badapter\b", r"\bhdmi\b", r"\bprotector\b",
            r"\bport\b", r"\bslot\b"
        ],
        "failure_terms": [
            r"\bnot_fit\b", r"\bnot_compatible\b", r"\bdoes not fit\b",
            r"\bwrong size\b", r"\btoo small\b", r"\btoo big\b"
        ],
        "exclude_terms": [],
        "title_only": False
    },
    "physical_build_installation_failure": {
        "title_terms": [
            r"\bmount\b", r"\bmounting\b", r"\bbracket\b", r"\bscrews\b",
            r"\bstrap\b", r"\bhinge\b", r"\bclip\b", r"\bscreen protector\b",
            r"\bscreen_protector\b"
        ],
        "review_terms": [
            r"\bmount\b", r"\bbracket\b", r"\bscrew\b", r"\bstrap\b",
            r"\bclip\b", r"\bprotector\b", r"\bcase\b"
        ],
        "failure_terms": [
            r"\bbroken\b", r"\bbroke\b", r"\bcracked\b", r"\bcracking\b",
            r"\bflimsy\b", r"\bstripped\b", r"\bbubbles\b", r"\bfell apart\b",
            r"\bpoor_quality\b"
        ],
        "exclude_terms": [
            r"\bfitbit\b", r"\bsmartwatch\b", r"\bsmart watch\b"
        ],
        "title_only": False
    },
    "audio_device_failure": {
        "title_terms": [
            r"\bearbuds?\b", r"\bheadphones?\b", r"\bspeaker\b",
            r"\bspeakers\b", r"\bmicrophone\b", r"\bmic\b"
        ],
        "review_terms": [
            r"\bearbuds?\b", r"\bheadphones?\b", r"\bspeaker\b",
            r"\bspeakers\b", r"\bmicrophone\b", r"\bmic\b",
            r"\bearpiece\b", r"\baudio\b"
        ],
        "failure_terms": [
            r"\bno_sound\b", r"\bdistorted\b", r"\bstatic\b", r"\bbuzzing\b",
            r"\bhissing\b", r"\bleft earpiece\b", r"\bright earpiece\b",
            r"\bone side\b", r"\bleft side\b", r"\bright side\b"
        ],
        "exclude_terms": [
            r"\bantenna\b", r"\bradio\b"
        ],
        "title_only": False
    },
    "video_display_failure": {
        "title_terms": [
            r"\bmonitor\b", r"\bdisplay\b", r"\bcamera\b", r"\bscreen\b",
            r"\bvideo\b", r"\bprojector\b", r"\bwebcam\b"
        ],
        "review_terms": [
            r"\bmonitor\b", r"\bdisplay\b", r"\bcamera\b", r"\bscreen\b",
            r"\bvideo\b", r"\brecording\b"
        ],
        "failure_terms": [
            r"\bflicker(?:ing)?\b", r"\bdead pixels?\b", r"\bblack screen\b",
            r"\bgreen horizontal lines\b", r"\bpixelated\b", r"\bblurry\b",
            r"\bnot recording\b", r"\bno_signal\b"
        ],
        "exclude_terms": [
            r"\bantenna\b", r"\breception\b", r"\bchannels\b", r"\bstations\b",
            r"\bradio\b"
        ],
        "title_only": False
    },
    "input_control_failure": {
        "title_terms": [
            r"\bkeyboard\b", r"\bmouse\b", r"\bremote\b", r"\bcontroller\b", r"\btrackpad\b"
        ],
        "review_terms": [
            r"\bkeyboard\b", r"\bmouse\b", r"\bremote\b", r"\bcontroller\b",
            r"\btrackpad\b", r"\bkeys\b", r"\bbuttons\b", r"\bscroll wheel\b",
            r"\bcursor\b", r"\bleft click\b", r"\bright click\b"
        ],
        "failure_terms": [
            r"\bdouble typing\b", r"\blag(?:ging)?\b", r"\bbuttons? stuck\b",
            r"\bkeys? stuck\b",
            r"\bscroll wheel\b.*\bnot_working\b",
            r"\bcursor\b.*\blag(?:ging)?\b",
            r"\bleft click\b.*\bnot_working\b",
            r"\bright click\b.*\bnot_working\b",
            r"\bkeyboard\b.*\bstopped_working\b", r"\bkeyboard\b.*\bnot_working\b",
            r"\bmouse\b.*\bstopped_working\b", r"\bmouse\b.*\bnot_working\b",
            r"\bremote\b.*\bstopped_working\b", r"\bremote\b.*\bnot_working\b",
            r"\bcontroller\b.*\bstopped_working\b", r"\bcontroller\b.*\bnot_working\b"
        ],
        "exclude_terms": [
            r"\bfitbit\b", r"\bsmartwatch\b", r"\bsmart watch\b"
        ],
        "title_only": False
    },
    "wearable_tracking_failure": {
        "title_terms": [
            r"\bfitbit\b", r"\bsmartwatch\b", r"\bsmart watch\b",
            r"\bactivity tracker\b", r"\bfitness tracker\b"
        ],
        "review_terms": [
            r"\bfitbit\b", r"\bsmartwatch\b", r"\bsmart watch\b",
            r"\btracking\b", r"\bheart rate\b", r"\bsteps\b", r"\bsleep\b"
        ],
        "failure_terms": [
            r"\binaccurate\b", r"\bnot_tracking\b", r"\bwon.t track\b",
            r"\bwrong heart rate\b", r"\bwrong step count\b",
            r"\binaccurate heart rate\b", r"\binaccurate sleep\b"
        ],
        "exclude_terms": [
            r"\bband\b", r"\bstrap\b", r"\bwrong size\b", r"\bnot_fit\b",
            r"\bmouse\b", r"\bkeyboard\b", r"\bremote\b", r"\bcable\b",
            r"\bcharger\b", r"\badapter\b", r"\bhard drive\b", r"\bssd\b", r"\bsd_card\b"
        ],
        "title_only": True
    },
    "antenna_reception_failure": {
        "title_terms": [
            r"\bantenna\b", r"\brabbit ears\b", r"\bradio antenna\b", r"\btv antenna\b"
        ],
        "review_terms": [
            r"\bantenna\b", r"\breception\b", r"\bchannels\b", r"\bstations\b"
        ],
        "failure_terms": [
            r"\bweak signal\b", r"\bpoor reception\b", r"\bchannels missing\b",
            r"\bchannels disappear\b", r"\bfuzzy\b", r"\bpixelated\b", r"\bno_signal\b"
        ],
        "exclude_terms": [
            r"\bhdmi\b", r"\badapter\b", r"\bmouse\b", r"\bkeyboard\b",
            r"\bfm transmitter\b", r"\bremote\b", r"\bdisplayport\b",
            r"\bdash cam\b", r"\bspeaker cable\b", r"\baudio cable\b"
        ],
        "title_only": True
    }
}

bucket_names = list(rule_definitions.keys())

bucket_priority = [
    "storage_data_reliability",
    "charging_power_failure",
    "connectivity_pairing_failure",
    "compatibility_fit_issue",
    "video_display_failure",
    "audio_device_failure",
    "input_control_failure",
    "physical_build_installation_failure",
    "wearable_tracking_failure",
    "antenna_reception_failure"
]

source_columns = [
    "document_id",
    "cohort_role",
    "price_band",
    "issue_priority_level",
    "product_display_name",
    "sampled_review_count",
    "modeling_text_clipped"
]

In [72]:
# Applying issue-bucket rules to Electronics review documents

review_rows = []

for record in pure_product_cause_topic_assignment_dataframe[source_columns].itertuples(index = False):
    title_text = normalize_text(record.product_display_name)
    review_list = split_document_reviews(record.modeling_text_clipped, boundary_token = review_boundary_token)
    review_total_count = len(review_list)

    for review_index, raw_review in enumerate(review_list, start = 1):
        review_text = normalize_text(build_rule_text(raw_review))
        if not review_text:
            continue

        row = {
            "document_id": record.document_id,
            "cohort_role": record.cohort_role,
            "price_band": record.price_band,
            "issue_priority_level": record.issue_priority_level,
            "product_display_name": record.product_display_name,
            "sampled_review_count": record.sampled_review_count,
            "review_total_count": review_total_count,
            "review_index": review_index,
            "review_text": review_text
        }

        for bucket_name, rule in rule_definitions.items():
            title_hit = has_pattern(title_text, rule["title_terms"])
            review_hit = has_pattern(review_text, rule["review_terms"])
            failure_hit = has_pattern(review_text, rule["failure_terms"])
            exclude_hit = has_pattern(title_text, rule["exclude_terms"]) or has_pattern(review_text, rule["exclude_terms"])
            object_gate = title_hit if rule["title_only"] else (title_hit or review_hit)

            row[bucket_name] = object_gate and failure_hit and not exclude_hit

        matched_buckets = [bucket for bucket in bucket_names if row[bucket]]
        row["matched_bucket_count"] = len(matched_buckets)
        row["review_primary_bucket"] = next((bucket for bucket in bucket_priority if row[bucket]), None)

        review_rows.append(row)

review_match_df = pd.DataFrame(review_rows)

document_base_df = (
    pure_product_cause_topic_assignment_dataframe[
        [
            "document_id",
            "cohort_role",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "sampled_review_count",
        ]
    ]
    .drop_duplicates("document_id")
    .copy()
)

review_bucket_counts_df = (
    review_match_df
    .melt(
        id_vars = [
            "document_id",
            "cohort_role",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "review_total_count",
        ],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "matched",
    )
    .loc[lambda df: df["matched"]]
    .groupby(
        [
            "document_id",
            "cohort_role",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "review_total_count",
            "bucket_name",
        ],
        dropna = False,
    )
    .size()
    .reset_index(name = "matched_review_count")
)

review_bucket_counts_df["matched_review_ratio"] = (
    review_bucket_counts_df["matched_review_count"]
    / review_bucket_counts_df["review_total_count"].replace(0, np.nan)
)

review_bucket_counts_df["passes_document_threshold"] = (
    review_bucket_counts_df["matched_review_count"] >= 2
)

document_bucket_df = (
    review_bucket_counts_df
    .loc[review_bucket_counts_df["passes_document_threshold"]]
    .copy()
)

document_bucket_wide_df = (
    document_bucket_df
    .assign(bucket_flag = True)
    .pivot_table(
        index = "document_id",
        columns = "bucket_name",
        values = "bucket_flag",
        aggfunc = "max",
        fill_value = False,
    )
    .reset_index()
)

document_match_df = document_base_df.merge(
    document_bucket_wide_df,
    on = "document_id",
    how = "left",
)

for bucket_name in bucket_names:
    if bucket_name not in document_match_df.columns:
        document_match_df[bucket_name] = False

    document_match_df[bucket_name] = (
        document_match_df[bucket_name]
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )

document_match_df["matched_bucket_count"] = document_match_df[bucket_names].sum(axis = 1)
document_match_df["has_any_bucket"] = document_match_df["matched_bucket_count"] > 0
document_match_df["has_multiple_buckets"] = document_match_df["matched_bucket_count"] >= 2

document_primary_bucket_df = (
    document_bucket_df
    .sort_values(
        [
            "document_id",
            "matched_review_count",
            "matched_review_ratio",
            "bucket_name",
        ],
        ascending = [True, False, False, True],
    )
    .groupby("document_id", group_keys = False)
    .head(1)
    .reset_index(drop = True)
)

coverage_summary_df = (
    document_match_df
    .groupby("cohort_role", dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        any_bucket_count = ("has_any_bucket", "sum"),
        multi_bucket_count = ("has_multiple_buckets", "sum"),
        unclassified_count = ("has_any_bucket", lambda values: (~values).sum())
    )
    .reset_index()
)

coverage_summary_df["any_bucket_ratio"] = (
    coverage_summary_df["any_bucket_count"] / coverage_summary_df["document_count"]
)

coverage_summary_df["multi_bucket_ratio"] = (
    coverage_summary_df["multi_bucket_count"] / coverage_summary_df["document_count"]
)

coverage_summary_df["unclassified_ratio"] = (
    coverage_summary_df["unclassified_count"] / coverage_summary_df["document_count"]
)

dissatisfaction_total = document_match_df["cohort_role"].eq("Dissatisfaction").sum()
satisfaction_total = document_match_df["cohort_role"].eq("Satisfaction").sum()

bucket_summary_df = pd.DataFrame({
    "bucket_name": bucket_names,
    "dissatisfaction_document_count": [
        document_match_df.loc[
            document_match_df["cohort_role"].eq("Dissatisfaction"), bucket
        ].sum()
        for bucket in bucket_names
    ],
    "satisfaction_document_count": [
        document_match_df.loc[
            document_match_df["cohort_role"].eq("Satisfaction"), bucket
        ].sum()
        for bucket in bucket_names
    ]
})

bucket_summary_df["dissatisfaction_ratio"] = (
    bucket_summary_df["dissatisfaction_document_count"] / dissatisfaction_total
)

bucket_summary_df["satisfaction_ratio"] = (
    bucket_summary_df["satisfaction_document_count"] / satisfaction_total
)

bucket_summary_df["lift_vs_satisfaction"] = (
    (bucket_summary_df["dissatisfaction_ratio"] + 1e-6)
    / (bucket_summary_df["satisfaction_ratio"] + 1e-6)
)

bucket_summary_df["ratio_gap"] = (
    bucket_summary_df["dissatisfaction_ratio"]
    - bucket_summary_df["satisfaction_ratio"]
)

display(coverage_summary_df)
display(bucket_summary_df.sort_values(["lift_vs_satisfaction", "ratio_gap"], ascending = [False, False]))
display(document_primary_bucket_df.head(20))

,cohort_role,document_count,any_bucket_count,multi_bucket_count,unclassified_count,any_bucket_ratio,multi_bucket_ratio,unclassified_ratio
0,Dissatisfaction,16853,12665,4975,4188,0.751498,0.295200,0.248502
1,Satisfaction,18360,4227,481,14133,0.230229,0.026198,0.769771


,bucket_name,dissatisfaction_document_count,satisfaction_document_count,dissatisfaction_ratio,satisfaction_ratio,lift_vs_satisfaction,ratio_gap
8,wearable_tracking_failure,87,5,0.005162,0.000272,18.890220,0.004890
0,charging_power_failure,3198,298,0.189758,0.016231,11.690503,0.173528
2,storage_data_reliability,961,94,0.057022,0.005120,11.135604,0.051903
6,video_display_failure,1680,324,0.099686,0.017647,5.648582,0.082038
1,connectivity_pairing_failure,2771,561,0.164422,0.030556,5.380933,0.133866
3,compatibility_fit_issue,2806,579,0.166499,0.031536,5.279507,0.134963
5,audio_device_failure,2938,991,0.174331,0.053976,3.229744,0.120355
7,input_control_failure,1420,565,0.084258,0.030773,2.737956,0.053485
4,physical_build_installation_failure,3256,1302,0.193200,0.070915,2.724363,0.122285
9,antenna_reception_failure,98,42,0.005815,0.002288,2.541307,0.003527


,document_id,cohort_role,price_band,issue_priority_level,product_display_name,review_total_count,bucket_name,matched_review_count,matched_review_ratio,passes_document_threshold
0,Dissatisfaction|true|Budget|Monitor|B00004Z5M1...,Dissatisfaction,Budget,Monitor,Belkin - F3U133b10 (F3U133b10) Hi-Speed USB A/...,64,compatibility_fit_issue,3,0.046875,True
1,Dissatisfaction|true|Budget|Monitor|B00005MDZD...,Dissatisfaction,Budget,Monitor,Gam3Gear Component AV Audio Video Cable for PS...,131,video_display_failure,10,0.076336,True
2,Dissatisfaction|true|Budget|Monitor|B00005T3DX...,Dissatisfaction,Budget,Monitor,"Duracell CR2 3V Lithium Battery, 1 Count Pack,...",46,charging_power_failure,2,0.043478,True
3,Dissatisfaction|true|Budget|Monitor|B00005T3GH...,Dissatisfaction,Budget,Monitor,RCA AH216 Stereo Headphone Adapter Plug,50,audio_device_failure,4,0.080000,True
4,Dissatisfaction|true|Budget|Monitor|B00009UHJS...,Dissatisfaction,Budget,Monitor,Scosche MDA1B Compatible with 1988-05 GM Micro...,30,compatibility_fit_issue,5,0.166667,True
5,Dissatisfaction|true|Budget|Monitor|B0000D88FU...,Dissatisfaction,Budget,Monitor,"GE RG59 Coaxial Cable 25ft. (7.6m), Black, F-T...",46,physical_build_installation_failure,3,0.065217,True
6,Dissatisfaction|true|Budget|Monitor|B00029MTMQ...,Dissatisfaction,Budget,Monitor,Zalman ZM-MIC1 High Sensitivity Headphone Micr...,120,audio_device_failure,19,0.158333,True
7,Dissatisfaction|true|Budget|Monitor|B0002A4M4I...,Dissatisfaction,Budget,Monitor,Canon USB Cable IFC-400PCU for Canon Cameras &...,26,compatibility_fit_issue,2,0.076923,True
8,Dissatisfaction|true|Budget|Monitor|B0002GX1XA...,Dissatisfaction,Budget,Monitor,"C2G 43036 TAA Compliant Cable Ties, 4 Inch Lon...",63,physical_build_installation_failure,2,0.031746,True
9,Dissatisfaction|true|Budget|Monitor|B0002KR13M...,Dissatisfaction,Budget,Monitor,DTOL MC 3.5mm Audio Cable - 3.5mm TRS Female t...,111,audio_device_failure,16,0.144144,True


In [73]:
# Auditing document-level threshold effects on issue classification
document_review_count_df = (
    review_match_df
    .groupby("document_id", dropna = False)
    .agg(
        review_total_count = ("review_index", "count"),
        matched_review_count = ("matched_bucket_count", lambda values: (values > 0).sum())
    )
    .reset_index()
)

document_threshold_audit_df = document_base_df.merge(
    document_review_count_df,
    on = "document_id",
    how = "left"
)

document_threshold_audit_df["review_total_count"] = (
    document_threshold_audit_df["review_total_count"]
    .fillna(0)
    .astype(int)
)

document_threshold_audit_df["matched_review_count"] = (
    document_threshold_audit_df["matched_review_count"]
    .fillna(0)
    .astype(int)
)

document_threshold_audit_df["has_any_review_match"] = (
    document_threshold_audit_df["matched_review_count"] > 0
)

document_threshold_audit_df["below_two_review_document"] = (
    document_threshold_audit_df["review_total_count"] < 2
)

document_threshold_audit_df["lost_short_matched_document"] = (
    document_threshold_audit_df["below_two_review_document"] &
    document_threshold_audit_df["has_any_review_match"]
)

threshold_loss_summary_df = (
    document_threshold_audit_df
    .groupby("cohort_role", dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        below_two_review_document_count = ("below_two_review_document", "sum"),
        any_review_match_count = ("has_any_review_match", "sum"),
        lost_short_matched_document_count = ("lost_short_matched_document", "sum")
    )
    .reset_index()
)

threshold_loss_summary_df["below_two_review_document_ratio"] = (
    threshold_loss_summary_df["below_two_review_document_count"] /
    threshold_loss_summary_df["document_count"]
)

threshold_loss_summary_df["lost_short_matched_document_ratio"] = (
    threshold_loss_summary_df["lost_short_matched_document_count"] /
    threshold_loss_summary_df["document_count"]
)

display(threshold_loss_summary_df)

,cohort_role,document_count,below_two_review_document_count,any_review_match_count,lost_short_matched_document_count,below_two_review_document_ratio,lost_short_matched_document_ratio
0,Dissatisfaction,16853,0,15052,0,0.0,0.0
1,Satisfaction,18360,0,9949,0,0.0,0.0


In [74]:
# Saving issue-bucket outputs for validation and dashboard preparation
checkpoint_dir = "electronics_issue_bucket_checkpoint"

import os
os.makedirs(checkpoint_dir, exist_ok = True)

review_match_df.to_parquet(f"{checkpoint_dir}/review_match_df.parquet", index = False)
document_base_df.to_parquet(f"{checkpoint_dir}/document_base_df.parquet", index = False)
review_bucket_counts_df.to_parquet(f"{checkpoint_dir}/review_bucket_counts_df.parquet", index = False)
document_bucket_df.to_parquet(f"{checkpoint_dir}/document_bucket_df.parquet", index = False)
document_match_df.to_parquet(f"{checkpoint_dir}/document_match_df.parquet", index = False)
document_primary_bucket_df.to_parquet(f"{checkpoint_dir}/document_primary_bucket_df.parquet", index = False)
coverage_summary_df.to_parquet(f"{checkpoint_dir}/coverage_summary_df.parquet", index = False)
bucket_summary_df.to_parquet(f"{checkpoint_dir}/bucket_summary_df.parquet", index = False)

print("Electronics issue bucket checkpoint saved successfully.")
print(checkpoint_dir)

Electronics issue bucket checkpoint saved successfully.
electronics_issue_bucket_checkpoint


In [75]:
# Reloading issue-bucket outputs for validation and reporting
checkpoint_dir = "electronics_issue_bucket_checkpoint"

review_match_df = pd.read_parquet(f"{checkpoint_dir}/review_match_df.parquet")
document_base_df = pd.read_parquet(f"{checkpoint_dir}/document_base_df.parquet")
review_bucket_counts_df = pd.read_parquet(f"{checkpoint_dir}/review_bucket_counts_df.parquet")
document_bucket_df = pd.read_parquet(f"{checkpoint_dir}/document_bucket_df.parquet")
document_match_df = pd.read_parquet(f"{checkpoint_dir}/document_match_df.parquet")
document_primary_bucket_df = pd.read_parquet(f"{checkpoint_dir}/document_primary_bucket_df.parquet")
coverage_summary_df = pd.read_parquet(f"{checkpoint_dir}/coverage_summary_df.parquet")
bucket_summary_df = pd.read_parquet(f"{checkpoint_dir}/bucket_summary_df.parquet")

print("Electronics issue bucket checkpoint reloaded successfully.")

Electronics issue bucket checkpoint reloaded successfully.


In [76]:
# Creating a validation sample from surviving issue-bucket matches
validation_sample_size = 50
validation_seed = 42

validated_candidate_keys_df = document_bucket_df[
    ["document_id", "bucket_name"]
].drop_duplicates()

validation_candidates_df = (
    review_match_df
    .melt(
        id_vars = [
            "document_id",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "review_text"
        ],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "matched"
    )
    .loc[lambda df: df["matched"]]
    .merge(
        validated_candidate_keys_df,
        on = ["document_id", "bucket_name"],
        how = "inner"
    )
    .copy()
)

validation_candidates_df["random_order"] = np.random.default_rng(
    validation_seed
).random(len(validation_candidates_df))

validation_sample_df = (
    validation_candidates_df
    .sort_values(["bucket_name", "random_order"])
    .groupby("bucket_name", group_keys = False)
    .head(validation_sample_size)
    .drop(columns = "random_order")
    .reset_index(drop = True)
)

def explain_match(product_name, review_text, bucket_name):
    rule = rule_definitions[bucket_name]
    title_text = normalize_text(product_name)
    text = normalize_text(review_text)

    title_hit = has_pattern(title_text, rule["title_terms"])
    review_hit = has_pattern(text, rule["review_terms"])
    failure_hit = has_pattern(text, rule["failure_terms"])
    exclude_hit = (
        has_pattern(title_text, rule["exclude_terms"]) or
        has_pattern(text, rule["exclude_terms"])
    )
    object_gate = title_hit if rule["title_only"] else (title_hit or review_hit)

    return pd.Series({
        "title_hit": title_hit,
        "review_hit": review_hit,
        "failure_hit": failure_hit,
        "exclude_hit": exclude_hit,
        "object_gate": object_gate
    })

validation_debug_df = validation_sample_df.apply(
    lambda row: explain_match(
        product_name = row["product_display_name"],
        review_text = row["review_text"],
        bucket_name = row["bucket_name"]
    ),
    axis = 1
)

validation_sample_df = pd.concat(
    [validation_sample_df, validation_debug_df],
    axis = 1
)

validation_sample_df["is_valid_match"] = pd.NA
validation_sample_df["error_type"] = ""
validation_sample_df["validation_note"] = ""

display(validation_sample_df)

,document_id,price_band,issue_priority_level,product_display_name,review_text,bucket_name,matched,title_hit,review_hit,failure_hit,exclude_hit,object_gate,is_valid_match,error_type,validation_note
0,Satisfaction|true|Premium|Monitor|B09BC2YNKF|c...,Premium,Monitor,HD7698A Long Range Outdoor HDTV Antenna - 65+ ...,this mile range antenna is replacing one adver...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
1,Dissatisfaction|true|Upper mid|Monitor|B0C46LB...,Upper mid,Monitor,2023 Amplified HD Digital TV Antenna Long 250+...,the product came with everything however it is...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
2,Dissatisfaction|true|Premium|Monitor|B0BWLZ5PM...,Premium,Monitor,[Newest 2021] Five Star Outdoor HDTV Antenna u...,this tv antenna is piece of crap have no_signa...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
3,Satisfaction|true|Upper mid|Priority review|B0...,Upper mid,Priority review,TV Antenna -Amplified HD Digital Indoor TV Ant...,does_not_work miles from stations no_signal do...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
4,Dissatisfaction|true|Lower mid|Priority review...,Lower mid,Priority review,"TV Antenna Amplified, HD Digital TV Antenna fo...",was not able to pick up any signal move it aro...,antenna_reception_failure,True,True,False,True,False,True,<NA>,,
5,Dissatisfaction|true|Upper mid|Monitor|B07HGQT...,Upper mid,Monitor,RONIN FACTORY Truck Radio Antenna Accessory fo...,as suspected this short antenna has good and p...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
6,Dissatisfaction|true|Premium|Monitor|B07465763...,Premium,Monitor,Channel Master Digital Advantage 100 Direction...,poor reception ended up purchasing different p...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
7,Satisfaction|true|Upper mid|Priority review|B0...,Upper mid,Priority review,TV Antenna -Amplified HD Digital Indoor TV Ant...,the antenna to me very quickly it was easy to ...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
8,Dissatisfaction|true|Lower mid|Priority review...,Lower mid,Priority review,CravenSpeed Stubby Antenna Compatible with Toy...,replaced the oem antenna on my toyota tundra b...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,
9,Dissatisfaction|true|Budget|Priority review|B0...,Budget,Priority review,"TV Antenna Amplifier, 25dB High Gain Signal Bo...",ended up losing channels and it pixelated to f...,antenna_reception_failure,True,True,True,True,False,True,<NA>,,


In [77]:
output_path = "validation_sample_df.csv"

validation_sample_df.to_csv(output_path, index = False)

print(f"Saved CSV : {output_path}")

Saved CSV : validation_sample_df.csv


#### Labeling the newly generated "validation_sample_df", saved as "validation_sample_df.csv". 
#### Saving the labeled version as a new file named "labeled_validation_sample_df.csv

Labeling Logic : reviewing sampled reviews and marking TRUE only when the review text explicitly and semantically reflects the assigned issue theme, otherwise FALSE.

In [78]:
# Loading externally labeled positive-match validation results
validation_sample_df = pd.read_csv("labeled_validation_sample_df.csv")

print(validation_sample_df.shape)

validation_sample_df.head()

(500, 15)


,document_id,price_band,issue_priority_level,product_display_name,review_text,bucket_name,matched,title_hit,review_hit,failure_hit,exclude_hit,object_gate,is_valid_match,error_type,validation_note
0,Satisfaction|true|Premium|Monitor|B09BC2YNKF|c...,Premium,Monitor,HD7698A Long Range Outdoor HDTV Antenna - 65+ ...,this mile range antenna is replacing one adver...,antenna_reception_failure,True,True,True,True,False,True,True,NaN,NaN
1,Dissatisfaction|true|Upper mid|Monitor|B0C46LB...,Upper mid,Monitor,2023 Amplified HD Digital TV Antenna Long 250+...,the product came with everything however it is...,antenna_reception_failure,True,True,True,True,False,True,True,NaN,NaN
2,Dissatisfaction|true|Premium|Monitor|B0BWLZ5PM...,Premium,Monitor,[Newest 2021] Five Star Outdoor HDTV Antenna u...,this tv antenna is piece of crap have no_signa...,antenna_reception_failure,True,True,True,True,False,True,True,NaN,NaN
3,Satisfaction|true|Upper mid|Priority review|B0...,Upper mid,Priority review,TV Antenna -Amplified HD Digital Indoor TV Ant...,does_not_work miles from stations no_signal do...,antenna_reception_failure,True,True,True,True,False,True,True,NaN,NaN
4,Dissatisfaction|true|Lower mid|Priority review...,Lower mid,Priority review,"TV Antenna Amplified, HD Digital TV Antenna fo...",was not able to pick up any signal move it aro...,antenna_reception_failure,True,True,False,True,False,True,True,NaN,NaN


In [79]:
# Calculating precision results for manually validated issue buckets

precision_thresholds = {
    "charging_power_failure": 0.85,
    "connectivity_pairing_failure": 0.85,
    "storage_data_reliability": 0.85,
    "compatibility_fit_issue": 0.85,
    "physical_build_installation_failure": 0.85,
    "audio_device_failure": 0.85,
    "video_display_failure": 0.85,
    "input_control_failure": 0.85,
    "wearable_tracking_failure": 0.75,
    "antenna_reception_failure": 0.75
}

validated_df = validation_sample_df.dropna(subset = ["is_valid_match"]).copy()

validated_df["is_valid_match"] = (
    validated_df["is_valid_match"]
    .astype(str)
    .str.upper()
    .map({"TRUE": True, "FALSE": False})
)

validated_df = validated_df.dropna(subset = ["is_valid_match"]).copy()

bucket_precision_df = (
    validated_df
    .groupby("bucket_name", dropna = False)
    .agg(
        reviewed_count = ("is_valid_match", "count"),
        valid_count = ("is_valid_match", "sum")
    )
    .reset_index()
)

bucket_precision_df["precision"] = (
    bucket_precision_df["valid_count"] / bucket_precision_df["reviewed_count"]
)

bucket_precision_df["target_precision"] = bucket_precision_df["bucket_name"].map(precision_thresholds)

bucket_precision_df["passes"] = (
    bucket_precision_df["precision"] >= bucket_precision_df["target_precision"]
)

bucket_precision_df["action"] = np.where(
    bucket_precision_df["passes"],
    "Keep as final",
    "Tighten rule"
)

error_breakdown_df = (
    validated_df.loc[~validated_df["is_valid_match"]]
    .groupby(["bucket_name", "error_type"], dropna = False)
    .size()
    .reset_index(name = "error_count")
    .sort_values(["bucket_name", "error_count"], ascending = [True, False])
)

display(bucket_precision_df.sort_values(["passes", "precision"]))
display(error_breakdown_df)
display(
    validated_df.loc[
        ~validated_df["is_valid_match"],
        [
            "bucket_name",
            "product_display_name",
            "review_text",
            "error_type",
            "validation_note"
        ]
    ]
)

,bucket_name,reviewed_count,valid_count,precision,target_precision,passes,action
0,antenna_reception_failure,50,50,1.0,0.75,True,Keep as final
1,audio_device_failure,50,50,1.0,0.85,True,Keep as final
2,charging_power_failure,50,50,1.0,0.85,True,Keep as final
3,compatibility_fit_issue,50,50,1.0,0.85,True,Keep as final
4,connectivity_pairing_failure,50,50,1.0,0.85,True,Keep as final
5,input_control_failure,50,50,1.0,0.85,True,Keep as final
6,physical_build_installation_failure,50,50,1.0,0.85,True,Keep as final
7,storage_data_reliability,50,50,1.0,0.85,True,Keep as final
8,video_display_failure,50,50,1.0,0.85,True,Keep as final
9,wearable_tracking_failure,50,50,1.0,0.75,True,Keep as final


,bucket_name,error_type,error_count


,bucket_name,product_display_name,review_text,error_type,validation_note


In [80]:
validation_sample_df["bucket_name"].value_counts().sort_index()

bucket_name
antenna_reception_failure              50
audio_device_failure                   50
charging_power_failure                 50
compatibility_fit_issue                50
connectivity_pairing_failure           50
input_control_failure                  50
physical_build_installation_failure    50
storage_data_reliability               50
video_display_failure                  50
wearable_tracking_failure              50
Name: count, dtype: int64

In [81]:
# Creating false-negative samples to check for missed issue matches

document_bucket_threshold_audit_df = (
    review_bucket_counts_df
    .groupby(["document_id", "bucket_name"], dropna = False)
    .agg(
        matched_review_count = ("matched_review_count", "max"),
        matched_review_ratio = ("matched_review_ratio", "max")
    )
    .reset_index()
)

threshold_sensitivity_df = (
    document_bucket_threshold_audit_df
    .assign(
        passes_threshold_1 = lambda df: df["matched_review_count"] >= 1,
        passes_threshold_2 = lambda df: df["matched_review_count"] >= 2,
        passes_threshold_3 = lambda df: df["matched_review_count"] >= 3
    )
    .groupby("bucket_name", dropna = False)
    .agg(
        document_bucket_pairs_threshold_1 = ("passes_threshold_1", "sum"),
        document_bucket_pairs_threshold_2 = ("passes_threshold_2", "sum"),
        document_bucket_pairs_threshold_3 = ("passes_threshold_3", "sum")
    )
    .reset_index()
)

threshold_sensitivity_df["lost_pairs_from_threshold_2"] = (
    threshold_sensitivity_df["document_bucket_pairs_threshold_1"]
    - threshold_sensitivity_df["document_bucket_pairs_threshold_2"]
)

threshold_sensitivity_df["lost_pair_ratio_from_threshold_2"] = (
    threshold_sensitivity_df["lost_pairs_from_threshold_2"]
    / threshold_sensitivity_df["document_bucket_pairs_threshold_1"].replace(0, np.nan)
)

display(
    threshold_sensitivity_df.sort_values(
        "lost_pair_ratio_from_threshold_2",
        ascending = False
    )
)

unclassified_or_threshold_lost_documents_df = (
    document_threshold_audit_df
    .merge(
        document_match_df[["document_id", "has_any_bucket", "matched_bucket_count"]],
        on = "document_id",
        how = "left"
    )
)

unclassified_or_threshold_lost_documents_df["has_any_bucket"] = (
    unclassified_or_threshold_lost_documents_df["has_any_bucket"]
    .fillna(False)
    .astype(bool)
)

unclassified_or_threshold_lost_documents_df["had_review_level_match_but_no_bucket"] = (
    unclassified_or_threshold_lost_documents_df["has_any_review_match"]
    & ~unclassified_or_threshold_lost_documents_df["has_any_bucket"]
)

threshold_loss_by_cohort_df = (
    unclassified_or_threshold_lost_documents_df
    .groupby("cohort_role", dropna = False)
    .agg(
        document_count = ("document_id", "count"),
        unclassified_document_count = ("has_any_bucket", lambda values: (~values).sum()),
        review_match_but_no_bucket_count = ("had_review_level_match_but_no_bucket", "sum")
    )
    .reset_index()
)

threshold_loss_by_cohort_df["unclassified_document_ratio"] = (
    threshold_loss_by_cohort_df["unclassified_document_count"]
    / threshold_loss_by_cohort_df["document_count"]
)

threshold_loss_by_cohort_df["review_match_but_no_bucket_ratio"] = (
    threshold_loss_by_cohort_df["review_match_but_no_bucket_count"]
    / threshold_loss_by_cohort_df["document_count"]
)

display(threshold_loss_by_cohort_df)

false_negative_candidate_documents_df = (
    unclassified_or_threshold_lost_documents_df
    .loc[
        ~unclassified_or_threshold_lost_documents_df["has_any_bucket"],
        ["document_id", "cohort_role", "price_band", "issue_priority_level", "product_display_name"]
    ]
    .copy()
)

false_negative_review_candidates_df = (
    review_match_df
    .loc[
        review_match_df["matched_bucket_count"].eq(0),
        [
            "document_id",
            "cohort_role",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "review_text"
        ]
    ]
    .merge(
        false_negative_candidate_documents_df[["document_id"]],
        on = "document_id",
        how = "inner"
    )
    .copy()
)

false_negative_review_candidates_df["random_order"] = np.random.default_rng(42).random(
    len(false_negative_review_candidates_df)
)

false_negative_validation_sample_df = (
    false_negative_review_candidates_df
    .sort_values(["cohort_role", "price_band", "issue_priority_level", "random_order"])
    .groupby(["cohort_role", "price_band", "issue_priority_level"], group_keys = False)
    .head(20)
    .drop(columns = "random_order")
    .reset_index(drop = True)
)

false_negative_validation_sample_df["is_missed_match"] = pd.NA
false_negative_validation_sample_df["missed_bucket_name"] = ""
false_negative_validation_sample_df["validation_note"] = ""

display(false_negative_validation_sample_df)
false_negative_validation_sample_df.to_csv("false_negative_validation_sample_df.csv", index = False)

print(f"Saved CSV : false_negative_validation_sample_df.csv")

,bucket_name,document_bucket_pairs_threshold_1,document_bucket_pairs_threshold_2,document_bucket_pairs_threshold_3,lost_pairs_from_threshold_2,lost_pair_ratio_from_threshold_2
3,compatibility_fit_issue,7327,3385,2117,3942,0.538010
8,video_display_failure,4031,2004,1283,2027,0.502853
6,physical_build_installation_failure,8939,4558,3236,4381,0.490100
7,storage_data_reliability,1967,1055,779,912,0.463650
5,input_control_failure,3669,1985,1321,1684,0.458981
4,connectivity_pairing_failure,5907,3332,2215,2575,0.435923
9,wearable_tracking_failure,160,92,66,68,0.425000
2,charging_power_failure,5796,3496,2533,2300,0.396825
0,antenna_reception_failure,221,140,83,81,0.366516
1,audio_device_failure,6149,3929,2924,2220,0.361034


,cohort_role,document_count,unclassified_document_count,review_match_but_no_bucket_count,unclassified_document_ratio,review_match_but_no_bucket_ratio
0,Dissatisfaction,16853,4188,2387,0.248502,0.141637
1,Satisfaction,18360,14133,5722,0.769771,0.311656


,document_id,cohort_role,price_band,issue_priority_level,product_display_name,review_text,is_missed_match,missed_bucket_name,validation_note
0,Dissatisfaction|true|Budget|Monitor|B011BIIO9I...,Dissatisfaction,Budget,Monitor,"Replacement Earpads, Mudder 2 Pieces Foam Ear ...",have the qs2 bose quite comfort headphones and...,<NA>,,
1,Dissatisfaction|true|Budget|Monitor|B07H3V1YQ6...,Dissatisfaction,Budget,Monitor,MOSISO Silicone Keyboard Cover Protective Skin...,this mosiso black keyboard cover does not keep...,<NA>,,
2,Dissatisfaction|true|Budget|Monitor|B093SVDY5V...,Dissatisfaction,Budget,Monitor,"Ethernet Cable,VANDESAIL CAT7 Network Cable RJ...",deployed about of these cables last year and a...,<NA>,,
3,Dissatisfaction|true|Budget|Monitor|B07ZCSL46C...,Dissatisfaction,Budget,Monitor,Tobfit Bands Compatible with Fitbit Versa 2 an...,ordered two of these one pink and one gray the...,<NA>,,
4,Dissatisfaction|true|Budget|Monitor|B079SCW1X8...,Dissatisfaction,Budget,Monitor,AK Bands Compatible for Fitbit Alta/Fitbit Alt...,missing of the turquoise band how do get the o...,<NA>,,
5,Dissatisfaction|true|Budget|Monitor|B07J4Q48X8...,Dissatisfaction,Budget,Monitor,Wepro Waterproof Bands Compatible with Fitbit ...,love the color of these bands so much they are...,<NA>,,
6,Dissatisfaction|true|Budget|Monitor|B01JPEKJYI...,Dissatisfaction,Budget,Monitor,"Rankie Micro USB Cable, Nylon Braided Extremel...",these break so easily actually had one break w...,<NA>,,
7,Dissatisfaction|true|Budget|Monitor|B083JLDPDP...,Dissatisfaction,Budget,Monitor,TotalMount Monitor Stand for Headphones and He...,this is overall great product if you are able ...,<NA>,,
8,Dissatisfaction|true|Budget|Monitor|B07TF7N15C...,Dissatisfaction,Budget,Monitor,NotoCity for Garmin Forerunner 35 Band Soft Si...,this is piece of crap spent two hours trying t...,<NA>,,
9,Dissatisfaction|true|Budget|Monitor|B08N6PZR6Y...,Dissatisfaction,Budget,Monitor,JETech Wireless FM Transmitter Radio Car Kit f...,so much static does_not_work very well,<NA>,,


Saved CSV : false_negative_validation_sample_df.csv


#### Labeling the newly generated "false_negative_validation_sample_df", saved as "false_negative_validation_sample_df.csv". 
#### Saving the labeled version as a new file named "labeled_false_negative_validation_sample_df.csv

Labeling Logic : reviewing unbucketed reviews and marking TRUE only when the review clearly contains a missed issue theme that should have been assigned to a predefined bucket.

In [82]:
# Loading externally labeled missed-issue validation results

false_negative_validation_sample_df = pd.read_csv("labeled_false_negative_validation_sample_df.csv")

false_negative_validated_df = false_negative_validation_sample_df.dropna(
    subset = ["is_missed_match"]
).copy()

false_negative_validated_df["is_missed_match"] = (
    false_negative_validated_df["is_missed_match"]
    .astype(str)
    .str.upper()
    .map({"TRUE": True, "FALSE": False})
)

false_negative_validated_df = false_negative_validated_df.dropna(
    subset = ["is_missed_match"]
).copy()

false_negative_summary_df = (
    false_negative_validated_df
    .groupby(["cohort_role"], dropna = False)
    .agg(
        reviewed_count = ("is_missed_match", "count"),
        missed_match_count = ("is_missed_match", "sum")
    )
    .reset_index()
)

false_negative_summary_df["estimated_false_negative_rate"] = (
    false_negative_summary_df["missed_match_count"]
    / false_negative_summary_df["reviewed_count"]
)

missed_bucket_breakdown_df = (
    false_negative_validated_df
    .loc[false_negative_validated_df["is_missed_match"]]
    .groupby(["missed_bucket_name"], dropna = False)
    .size()
    .reset_index(name = "missed_count")
    .sort_values("missed_count", ascending = False)
)

display(false_negative_summary_df)
display(missed_bucket_breakdown_df)
display(
    false_negative_validated_df.loc[
        false_negative_validated_df["is_missed_match"],
        [
            "cohort_role",
            "price_band",
            "issue_priority_level",
            "product_display_name",
            "review_text",
            "missed_bucket_name",
            "validation_note"
        ]
    ]
)

,cohort_role,reviewed_count,missed_match_count,estimated_false_negative_rate
0,Dissatisfaction,180,126,0.7
1,Satisfaction,180,18,0.1


,missed_bucket_name,missed_count
6,physical_build_installation_failure,25
1,audio_device_failure,23
3,compatibility_fit_issue,23
2,charging_power_failure,20
4,connectivity_pairing_failure,15
8,video_display_failure,13
5,input_control_failure,11
0,antenna_reception_failure,7
7,storage_data_reliability,4
9,wearable_tracking_failure,3


,cohort_role,price_band,issue_priority_level,product_display_name,review_text,missed_bucket_name,validation_note
0,Dissatisfaction,Budget,Monitor,"Replacement Earpads, Mudder 2 Pieces Foam Ear ...",have the qs2 bose quite comfort headphones and...,physical_build_installation_failure,Earpads become dislodged and do not stay attac...
1,Dissatisfaction,Budget,Monitor,MOSISO Silicone Keyboard Cover Protective Skin...,this mosiso black keyboard cover does not keep...,physical_build_installation_failure,Keyboard cover loses shape and blocks keyboard...
3,Dissatisfaction,Budget,Monitor,Tobfit Bands Compatible with Fitbit Versa 2 an...,ordered two of these one pink and one gray the...,compatibility_fit_issue,"Fitbit band will not attach to the device, ind..."
5,Dissatisfaction,Budget,Monitor,Wepro Waterproof Bands Compatible with Fitbit ...,love the color of these bands so much they are...,physical_build_installation_failure,"Bands scuff very easily after minimal use, ind..."
6,Dissatisfaction,Budget,Monitor,"Rankie Micro USB Cable, Nylon Braided Extremel...",these break so easily actually had one break w...,physical_build_installation_failure,Charging cables break easily and leave pieces ...
7,Dissatisfaction,Budget,Monitor,TotalMount Monitor Stand for Headphones and He...,this is overall great product if you are able ...,physical_build_installation_failure,Holder/stand falls off because it does not sti...
8,Dissatisfaction,Budget,Monitor,NotoCity for Garmin Forerunner 35 Band Soft Si...,this is piece of crap spent two hours trying t...,compatibility_fit_issue,One side fits but the other side does not alig...
9,Dissatisfaction,Budget,Monitor,JETech Wireless FM Transmitter Radio Car Kit f...,so much static does_not_work very well,audio_device_failure,Review reports heavy static and poor function ...
10,Dissatisfaction,Budget,Monitor,Macally 2.4G Wireless Mouse with USB Receiver ...,it was defective and would not stay engaged de...,input_control_failure,"Mouse is defective and will not stay engaged, ..."
15,Dissatisfaction,Budget,Monitor,Ablus 120 Pockets Photo Album for Fujifilm Ins...,do not get it you can not put the minis in her...,compatibility_fit_issue,Photo sleeves only fit sideways rather than up...


In [83]:
# Comparing issue-theme exposure between dissatisfied and satisfied documents
cohort_bucket_exposure_df = (
    document_match_df
    .groupby("cohort_role", dropna = False)[bucket_names]
    .mean()
    .reset_index()
    .melt(
        id_vars = "cohort_role",
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "bucket_exposure_ratio"
    )
)

cohort_bucket_pivot_df = (
    cohort_bucket_exposure_df
    .pivot(
        index = "bucket_name",
        columns = "cohort_role",
        values = "bucket_exposure_ratio"
    )
    .reset_index()
)

cohort_bucket_pivot_df["dissatisfaction_lift_vs_satisfaction"] = (
    (cohort_bucket_pivot_df.get("Dissatisfaction", 0) + 1e-6) /
    (cohort_bucket_pivot_df.get("Satisfaction", 0) + 1e-6)
)

cohort_bucket_pivot_df["exposure_gap"] = (
    cohort_bucket_pivot_df.get("Dissatisfaction", 0) -
    cohort_bucket_pivot_df.get("Satisfaction", 0)
)

display(
    cohort_bucket_pivot_df.sort_values(
        ["dissatisfaction_lift_vs_satisfaction", "exposure_gap"],
        ascending = [False, False]
    )
)

cohort_role,bucket_name,Dissatisfaction,Satisfaction,dissatisfaction_lift_vs_satisfaction,exposure_gap
9,wearable_tracking_failure,0.005162,0.000272,18.890220,0.004890
2,charging_power_failure,0.189758,0.016231,11.690503,0.173528
7,storage_data_reliability,0.057022,0.005120,11.135604,0.051903
8,video_display_failure,0.099686,0.017647,5.648582,0.082038
4,connectivity_pairing_failure,0.164422,0.030556,5.380933,0.133866
3,compatibility_fit_issue,0.166499,0.031536,5.279507,0.134963
1,audio_device_failure,0.174331,0.053976,3.229744,0.120355
5,input_control_failure,0.084258,0.030773,2.737956,0.053485
6,physical_build_installation_failure,0.193200,0.070915,2.724363,0.122285
0,antenna_reception_failure,0.005815,0.002288,2.541307,0.003527


In [84]:
# Reviewing issue-theme exposure across price bands
price_band_bucket_exposure_df = (
    document_match_df
    .groupby(["cohort_role", "price_band"], dropna = False)[bucket_names]
    .mean()
    .reset_index()
    .melt(
        id_vars = ["cohort_role", "price_band"],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "bucket_exposure_ratio"
    )
)

display(
    price_band_bucket_exposure_df.sort_values(
        ["cohort_role", "bucket_name", "bucket_exposure_ratio"],
        ascending = [True, True, False]
    )
)

,cohort_role,price_band,bucket_name,bucket_exposure_ratio
91,Dissatisfaction,Lower mid,antenna_reception_failure,0.007880
93,Dissatisfaction,Upper mid,antenna_reception_failure,0.007459
90,Dissatisfaction,Budget,antenna_reception_failure,0.005069
92,Dissatisfaction,Premium,antenna_reception_failure,0.003277
94,Dissatisfaction,Very premium,antenna_reception_failure,0.000000
52,Dissatisfaction,Premium,audio_device_failure,0.197979
53,Dissatisfaction,Upper mid,audio_device_failure,0.190099
51,Dissatisfaction,Lower mid,audio_device_failure,0.189116
54,Dissatisfaction,Very premium,audio_device_failure,0.161634
50,Dissatisfaction,Budget,audio_device_failure,0.123823


In [85]:
# Reviewing issue-theme exposure across product priority groups
priority_bucket_exposure_df = (
    document_match_df
    .groupby(["cohort_role", "issue_priority_level"], dropna = False)[bucket_names]
    .mean()
    .reset_index()
    .melt(
        id_vars = ["cohort_role", "issue_priority_level"],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "bucket_exposure_ratio"
    )
)

display(
    priority_bucket_exposure_df.sort_values(
        ["cohort_role", "issue_priority_level", "bucket_exposure_ratio"],
        ascending = [True, True, False]
    )
)

,cohort_role,issue_priority_level,bucket_name,bucket_exposure_ratio
16,Dissatisfaction,Monitor,physical_build_installation_failure,0.209949
12,Dissatisfaction,Monitor,compatibility_fit_issue,0.179109
0,Dissatisfaction,Monitor,charging_power_failure,0.167975
20,Dissatisfaction,Monitor,audio_device_failure,0.158122
4,Dissatisfaction,Monitor,connectivity_pairing_failure,0.137136
24,Dissatisfaction,Monitor,video_display_failure,0.095883
28,Dissatisfaction,Monitor,input_control_failure,0.074656
8,Dissatisfaction,Monitor,storage_data_reliability,0.064963
36,Dissatisfaction,Monitor,antenna_reception_failure,0.003124
32,Dissatisfaction,Monitor,wearable_tracking_failure,0.002483


In [86]:
# Calculating how often issue themes appear together
cooccurrence_rows = []

for i, bucket_a in enumerate(bucket_names):
    for bucket_b in bucket_names[i + 1:]:
        both_count = (
            document_match_df[bucket_a] &
            document_match_df[bucket_b]
        ).sum()

        bucket_a_count = document_match_df[bucket_a].sum()
        bucket_b_count = document_match_df[bucket_b].sum()

        cooccurrence_rows.append({
            "bucket_a": bucket_a,
            "bucket_b": bucket_b,
            "both_document_count": both_count,
            "bucket_a_count": bucket_a_count,
            "bucket_b_count": bucket_b_count,
            "cooccurrence_ratio_with_bucket_a": both_count / bucket_a_count if bucket_a_count else 0,
            "cooccurrence_ratio_with_bucket_b": both_count / bucket_b_count if bucket_b_count else 0
        })

bucket_cooccurrence_df = pd.DataFrame(cooccurrence_rows)

display(
    bucket_cooccurrence_df.sort_values(
        "both_document_count",
        ascending = False
    )
)

,bucket_a,bucket_b,both_document_count,bucket_a_count,bucket_b_count,cooccurrence_ratio_with_bucket_a,cooccurrence_ratio_with_bucket_b
24,compatibility_fit_issue,physical_build_installation_failure,1496,3385,4558,0.441950,0.328214
12,connectivity_pairing_failure,audio_device_failure,1138,3332,3929,0.341537,0.289641
4,charging_power_failure,audio_device_failure,896,3496,3929,0.256293,0.228048
0,charging_power_failure,connectivity_pairing_failure,885,3496,3332,0.253146,0.265606
2,charging_power_failure,compatibility_fit_issue,460,3496,3385,0.131579,0.135894
14,connectivity_pairing_failure,input_control_failure,419,3332,1985,0.125750,0.211083
30,physical_build_installation_failure,audio_device_failure,361,4558,3929,0.079201,0.091881
25,compatibility_fit_issue,audio_device_failure,326,3385,3929,0.096307,0.082973
3,charging_power_failure,physical_build_installation_failure,317,3496,4558,0.090675,0.069548
10,connectivity_pairing_failure,compatibility_fit_issue,275,3332,3385,0.082533,0.081241


In [87]:
# Identifying top Electronics products linked to each issue theme
product_bucket_risk_df = (
    document_match_df
    .melt(
        id_vars = [
            "document_id",
            "cohort_role",
            "price_band",
            "issue_priority_level",
            "product_display_name"
        ],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "has_bucket"
    )
    .loc[lambda df: df["has_bucket"]]
    .copy()
)

top_products_by_bucket_df = (
    product_bucket_risk_df
    .groupby(
        ["bucket_name", "product_display_name", "price_band", "issue_priority_level"],
        dropna = False
    )
    .size()
    .reset_index(name = "document_count")
    .sort_values(["bucket_name", "document_count"], ascending = [True, False])
    .groupby("bucket_name", group_keys = False)
    .head(20)
    .reset_index(drop = True)

)

display(top_products_by_bucket_df)

,bucket_name,product_display_name,price_band,issue_priority_level,document_count
0,antenna_reception_failure,Mediasonic Homeworx HW110AN Super Thin Indoor ...,Budget,Priority review,8
1,antenna_reception_failure,"Vansky TV Antenna Indoor, Digital Amplified In...",Lower mid,Priority review,7
2,antenna_reception_failure,"Mohu Leaf 50 Amplified Indoor TV Antenna, 60-M...",Upper mid,Priority review,5
3,antenna_reception_failure,"Winegard LNA-200 Boost XT HDTV Preamplifier, T...",Upper mid,Priority review,5
4,antenna_reception_failure,2023 Amplified HD Digital TV Antenna Long 250+...,Upper mid,Monitor,4
5,antenna_reception_failure,CravenSpeed Stubby Antenna Compatible with Toy...,Lower mid,Priority review,4
6,antenna_reception_failure,RCA ANT1650F Flat Digital Amplified Indoor TV ...,Upper mid,Priority review,3
7,antenna_reception_failure,RCA Compact Outdoor or Attic Yagi TV Antenna –...,Upper mid,Monitor,3
8,antenna_reception_failure,24db Distribution Amplifier | Digital TV Anten...,Upper mid,Priority review,2
9,antenna_reception_failure,BOSS Audio Systems MRANT12W Marine Rubber Ante...,Budget,Priority review,2


In [88]:
# Combining issue-theme metrics into a business insight summary
business_insight_summary_df = (
    cohort_bucket_pivot_df
    .merge(
        bucket_summary_df[
            [
                "bucket_name",
                "dissatisfaction_document_count",
                "satisfaction_document_count",
                "dissatisfaction_ratio",
                "satisfaction_ratio",
                "lift_vs_satisfaction",
                "ratio_gap"
            ]
        ],
        on = "bucket_name",
        how = "left"
    )
    .sort_values(["lift_vs_satisfaction", "ratio_gap"], ascending = [False, False])
)

display(business_insight_summary_df)

,bucket_name,Dissatisfaction,Satisfaction,dissatisfaction_lift_vs_satisfaction,exposure_gap,dissatisfaction_document_count,satisfaction_document_count,dissatisfaction_ratio,satisfaction_ratio,lift_vs_satisfaction,ratio_gap
9,wearable_tracking_failure,0.005162,0.000272,18.890220,0.004890,87,5,0.005162,0.000272,18.890220,0.004890
2,charging_power_failure,0.189758,0.016231,11.690503,0.173528,3198,298,0.189758,0.016231,11.690503,0.173528
7,storage_data_reliability,0.057022,0.005120,11.135604,0.051903,961,94,0.057022,0.005120,11.135604,0.051903
8,video_display_failure,0.099686,0.017647,5.648582,0.082038,1680,324,0.099686,0.017647,5.648582,0.082038
4,connectivity_pairing_failure,0.164422,0.030556,5.380933,0.133866,2771,561,0.164422,0.030556,5.380933,0.133866
3,compatibility_fit_issue,0.166499,0.031536,5.279507,0.134963,2806,579,0.166499,0.031536,5.279507,0.134963
1,audio_device_failure,0.174331,0.053976,3.229744,0.120355,2938,991,0.174331,0.053976,3.229744,0.120355
5,input_control_failure,0.084258,0.030773,2.737956,0.053485,1420,565,0.084258,0.030773,2.737956,0.053485
6,physical_build_installation_failure,0.193200,0.070915,2.724363,0.122285,3256,1302,0.193200,0.070915,2.724363,0.122285
0,antenna_reception_failure,0.005815,0.002288,2.541307,0.003527,98,42,0.005815,0.002288,2.541307,0.003527


In [89]:
# Creating final dashboard-ready Electronics NLP analytical tables

price_band_order_map = {
    "Budget": 1,
    "Lower mid": 2,
    "Upper mid": 3,
    "Premium": 4,
    "Very premium": 5
}

bucket_order_map = {
    "charging_power_failure": 1,
    "connectivity_pairing_failure": 2,
    "storage_data_reliability": 3,
    "compatibility_fit_issue": 4,
    "physical_build_installation_failure": 5,
    "audio_device_failure": 6,
    "video_display_failure": 7,
    "input_control_failure": 8,
    "wearable_tracking_failure": 9,
    "antenna_reception_failure": 10
}

bucket_label_map = {
    "charging_power_failure": "Charging / Power Failure",
    "connectivity_pairing_failure": "Connectivity / Pairing Failure",
    "storage_data_reliability": "Storage / Data Reliability",
    "compatibility_fit_issue": "Compatibility / Fit Issue",
    "physical_build_installation_failure": "Physical Build / Installation Failure",
    "audio_device_failure": "Audio Device Failure",
    "video_display_failure": "Video / Display Failure",
    "input_control_failure": "Input / Control Failure",
    "wearable_tracking_failure": "Wearable Tracking Failure",
    "antenna_reception_failure": "Antenna / Reception Failure"
}

bucket_description_map = {
    "charging_power_failure": "Charging, battery, power-on, and power-supply reliability issues",
    "connectivity_pairing_failure": "Wireless, network, Bluetooth, pairing, and disconnect issues",
    "storage_data_reliability": "Drive, memory-card, formatting, recognition, and data-loss issues",
    "compatibility_fit_issue": "Fit, model, port, size, adapter, and compatibility mismatch issues",
    "physical_build_installation_failure": "Breakage, mounting, screws, case, protector, and installation friction",
    "audio_device_failure": "Headphone, earbud, speaker, microphone, static, no-sound, and distortion issues",
    "video_display_failure": "Monitor, display, camera, screen, flicker, no-signal, and recording issues",
    "input_control_failure": "Keyboard, mouse, remote, controller, button, lag, and response issues",
    "wearable_tracking_failure": "Smartwatch, tracker, heart-rate, sleep, step, and measurement issues",
    "antenna_reception_failure": "TV/radio antenna signal, reception, channel, and no-signal issues"
}

bucket_interpretation_df = (
    pd.DataFrame({"bucket_name": bucket_names})
    .assign(
        bucket_order = lambda df: df["bucket_name"].map(bucket_order_map),
        business_label = lambda df: df["bucket_name"].map(bucket_label_map),
        dashboard_description = lambda df: df["bucket_name"].map(bucket_description_map)
    )
    .sort_values("bucket_order")
    .reset_index(drop = True)
)

document_match_final_df = document_match_df.copy()

document_id_parts = document_match_final_df["document_id"].astype(str).str.split(r"\|", regex = True)

document_match_final_df["parent_asin"] = document_id_parts.str[4]

electronics_nlp_scope_quality_df = coverage_summary_df.copy()
electronics_nlp_scope_quality_df["evidence_scope"] = (
    "Validated rule-based issue buckets applied to comparison-eligible Electronics NLP documents"
)

electronics_nlp_validation_quality_df = (
    bucket_precision_df
    .merge(bucket_interpretation_df, on = "bucket_name", how = "left")
    .sort_values("bucket_order")
    .reset_index(drop = True)
)

electronics_nlp_false_negative_quality_df = false_negative_summary_df.copy()

electronics_nlp_bucket_overview_df = (
    bucket_summary_df[
        [
            "bucket_name",
            "dissatisfaction_document_count",
            "satisfaction_document_count",
            "dissatisfaction_ratio",
            "satisfaction_ratio",
            "lift_vs_satisfaction",
            "ratio_gap"
        ]
    ]
    .merge(bucket_interpretation_df, on = "bucket_name", how = "left")
)

electronics_nlp_bucket_overview_df["business_signal_type"] = np.select(
    [
        (electronics_nlp_bucket_overview_df["dissatisfaction_document_count"] < 100),
        (electronics_nlp_bucket_overview_df["lift_vs_satisfaction"] >= 8)
        & (electronics_nlp_bucket_overview_df["dissatisfaction_ratio"] >= 0.03),
        (electronics_nlp_bucket_overview_df["lift_vs_satisfaction"] >= 4)
        & (electronics_nlp_bucket_overview_df["dissatisfaction_ratio"] >= 0.03),
        (electronics_nlp_bucket_overview_df["dissatisfaction_ratio"] >= 0.08)
    ],
    [
        "Niche signal",
        "Strong dissatisfaction separator",
        "Moderate dissatisfaction separator",
        "Broad operational risk"
    ],
    default = "Supporting signal"
)

electronics_nlp_bucket_overview_df["evidence_level"] = np.select(
    [
        electronics_nlp_bucket_overview_df["dissatisfaction_document_count"] >= 1000,
        electronics_nlp_bucket_overview_df["dissatisfaction_document_count"] >= 300,
        electronics_nlp_bucket_overview_df["dissatisfaction_document_count"] >= 100
    ],
    ["High", "Moderate", "Limited"],
    default = "Low"
)

electronics_nlp_bucket_overview_df["interpretation_caution"] = np.select(
    [
        electronics_nlp_bucket_overview_df["dissatisfaction_document_count"] < 100,
        electronics_nlp_bucket_overview_df["satisfaction_document_count"] < 50
    ],
    [
        "Low volume; use as niche signal",
        "Small comparison baseline; interpret lift cautiously"
    ],
    default = "Suitable for business interpretation"
)

electronics_nlp_bucket_overview_df = (
    electronics_nlp_bucket_overview_df
    .sort_values(["bucket_order"])
    .reset_index(drop = True)
)

electronics_nlp_price_bucket_exposure_df = (
    document_match_final_df
    .groupby(["cohort_role", "price_band"], dropna = False)[bucket_names]
    .mean()
    .reset_index()
    .melt(
        id_vars = ["cohort_role", "price_band"],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "bucket_exposure_ratio"
    )
    .pivot_table(
        index = ["price_band", "bucket_name"],
        columns = "cohort_role",
        values = "bucket_exposure_ratio",
        aggfunc = "mean",
        fill_value = 0
    )
    .reset_index()
    .rename(columns = {
        "Dissatisfaction": "dissatisfaction_exposure_ratio",
        "Satisfaction": "satisfaction_exposure_ratio"
    })
    .merge(bucket_interpretation_df, on = "bucket_name", how = "left")
)

electronics_nlp_price_bucket_exposure_df["lift_vs_satisfaction"] = (
    (electronics_nlp_price_bucket_exposure_df["dissatisfaction_exposure_ratio"] + 1e-6)
    / (electronics_nlp_price_bucket_exposure_df["satisfaction_exposure_ratio"] + 1e-6)
)

electronics_nlp_price_bucket_exposure_df["exposure_gap"] = (
    electronics_nlp_price_bucket_exposure_df["dissatisfaction_exposure_ratio"]
    - electronics_nlp_price_bucket_exposure_df["satisfaction_exposure_ratio"]
)

electronics_nlp_price_bucket_exposure_df["price_band_order"] = (
    electronics_nlp_price_bucket_exposure_df["price_band"].map(price_band_order_map)
)

electronics_nlp_price_bucket_exposure_df = (
    electronics_nlp_price_bucket_exposure_df
    .sort_values(["price_band_order", "bucket_order"])
    .reset_index(drop = True)
)

electronics_nlp_priority_bucket_exposure_df = (
    document_match_final_df
    .groupby(["cohort_role", "issue_priority_level"], dropna = False)[bucket_names]
    .mean()
    .reset_index()
    .melt(
        id_vars = ["cohort_role", "issue_priority_level"],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "bucket_exposure_ratio"
    )
    .pivot_table(
        index = ["cohort_role", "bucket_name"],
        columns = "issue_priority_level",
        values = "bucket_exposure_ratio",
        aggfunc = "mean",
        fill_value = 0
    )
    .reset_index()
    .rename(columns = {
        "Monitor": "monitor_exposure_ratio",
        "Priority review": "priority_review_exposure_ratio"
    })
    .merge(bucket_interpretation_df, on = "bucket_name", how = "left")
)

electronics_nlp_priority_bucket_exposure_df["priority_lift_vs_monitor"] = (
    (electronics_nlp_priority_bucket_exposure_df["priority_review_exposure_ratio"] + 1e-6)
    / (electronics_nlp_priority_bucket_exposure_df["monitor_exposure_ratio"] + 1e-6)
)

electronics_nlp_priority_bucket_exposure_df["priority_gap_vs_monitor"] = (
    electronics_nlp_priority_bucket_exposure_df["priority_review_exposure_ratio"]
    - electronics_nlp_priority_bucket_exposure_df["monitor_exposure_ratio"]
)

electronics_nlp_priority_bucket_exposure_df = (
    electronics_nlp_priority_bucket_exposure_df
    .sort_values(["cohort_role", "bucket_order"])
    .reset_index(drop = True)
)

cooccurrence_rows = []

for cohort in sorted(document_match_final_df["cohort_role"].dropna().unique()):
    cohort_df = document_match_final_df.loc[document_match_final_df["cohort_role"].eq(cohort)]
    total_documents = len(cohort_df)

    for i, bucket_a in enumerate(bucket_names):
        for bucket_b in bucket_names[i + 1:]:
            both_count = int((cohort_df[bucket_a] & cohort_df[bucket_b]).sum())
            bucket_a_count = int(cohort_df[bucket_a].sum())
            bucket_b_count = int(cohort_df[bucket_b].sum())

            cooccurrence_rows.append({
                "cohort_role": cohort,
                "bucket_a": bucket_a,
                "bucket_b": bucket_b,
                "both_document_count": both_count,
                "bucket_a_count": bucket_a_count,
                "bucket_b_count": bucket_b_count,
                "total_documents": total_documents,
                "cooccurrence_ratio_total": both_count / total_documents if total_documents else 0,
                "cooccurrence_ratio_with_bucket_a": both_count / bucket_a_count if bucket_a_count else 0,
                "cooccurrence_ratio_with_bucket_b": both_count / bucket_b_count if bucket_b_count else 0
            })

electronics_nlp_bucket_cooccurrence_df = (
    pd.DataFrame(cooccurrence_rows)
    .merge(
        bucket_interpretation_df[["bucket_name", "business_label", "bucket_order"]].rename(
            columns = {
                "bucket_name": "bucket_a",
                "business_label": "bucket_a_label",
                "bucket_order": "bucket_a_order"
            }
        ),
        on = "bucket_a",
        how = "left"
    )
    .merge(
        bucket_interpretation_df[["bucket_name", "business_label", "bucket_order"]].rename(
            columns = {
                "bucket_name": "bucket_b",
                "business_label": "bucket_b_label",
                "bucket_order": "bucket_b_order"
            }
        ),
        on = "bucket_b",
        how = "left"
    )
    .sort_values(["cohort_role", "both_document_count"], ascending = [True, False])
    .reset_index(drop = True)
)

electronics_nlp_multibucket_intensity_df = (
    document_match_final_df
    .groupby(["cohort_role", "matched_bucket_count"], dropna = False)
    .size()
    .reset_index(name = "document_count")
)

electronics_nlp_multibucket_intensity_df = electronics_nlp_multibucket_intensity_df.merge(
    document_match_final_df
    .groupby("cohort_role", dropna = False)
    .size()
    .reset_index(name = "cohort_document_count"),
    on = "cohort_role",
    how = "left"
)

electronics_nlp_multibucket_intensity_df["document_ratio"] = (
    electronics_nlp_multibucket_intensity_df["document_count"]
    / electronics_nlp_multibucket_intensity_df["cohort_document_count"]
)

product_bucket_risk_df = (
    document_match_final_df
    .loc[document_match_final_df["cohort_role"].eq("Dissatisfaction")]
    .melt(
        id_vars = [
            "document_id",
            "parent_asin",
            "price_band",
            "issue_priority_level",
            "product_display_name"
        ],
        value_vars = bucket_names,
        var_name = "bucket_name",
        value_name = "has_bucket"
    )
    .loc[lambda df: df["has_bucket"]]
    .copy()
)

electronics_nlp_top_products_by_bucket_df = (
    product_bucket_risk_df
    .groupby(
        ["bucket_name", "parent_asin", "product_display_name", "price_band", "issue_priority_level"],
        dropna = False
    )
    .agg(dissatisfaction_document_count = ("document_id", "count"))
    .reset_index()
    .merge(bucket_interpretation_df, on = "bucket_name", how = "left")
    .sort_values(["bucket_order", "dissatisfaction_document_count"], ascending = [True, False])
)

electronics_nlp_top_products_by_bucket_df["bucket_product_rank"] = (
    electronics_nlp_top_products_by_bucket_df
    .groupby("bucket_name")["dissatisfaction_document_count"]
    .rank(method = "first", ascending = False)
    .astype(int)
)

electronics_nlp_top_products_by_bucket_df = (
    electronics_nlp_top_products_by_bucket_df
    .loc[lambda df: df["bucket_product_rank"] <= 10]
    .sort_values(["bucket_order", "bucket_product_rank"])
    .reset_index(drop = True)
)

print("Final analytical dataframes created.")

Final analytical dataframes created.


In [90]:
# Uploading final Electronics NLP dashboard tables to BigQuery
final_dashboard_tables = {
    "electronics_nlp_bucket_dictionary": bucket_interpretation_df,
    "electronics_nlp_scope_quality": electronics_nlp_scope_quality_df,
    "electronics_nlp_validation_quality": electronics_nlp_validation_quality_df,
    "electronics_nlp_false_negative_quality": electronics_nlp_false_negative_quality_df,
    "electronics_nlp_bucket_overview": electronics_nlp_bucket_overview_df,
    "electronics_nlp_price_bucket_exposure": electronics_nlp_price_bucket_exposure_df,
    "electronics_nlp_priority_bucket_exposure": electronics_nlp_priority_bucket_exposure_df,
    "electronics_nlp_bucket_cooccurrence": electronics_nlp_bucket_cooccurrence_df,
    "electronics_nlp_multibucket_intensity": electronics_nlp_multibucket_intensity_df,
    "electronics_nlp_top_products_by_bucket": electronics_nlp_top_products_by_bucket_df
}

load_job_config = bigquery.LoadJobConfig(
    write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect = True
)

for table_name, dataframe in final_dashboard_tables.items():
    destination_table_id = (
        f"{configuration['gcp_project_id']}."
        f"{configuration['bigquery_core_dataset_id']}."
        f"{table_name}"
    )

    dataframe_to_upload = dataframe.copy()

    load_job = bigquery_client.load_table_from_dataframe(
        dataframe_to_upload,
        destination_table_id,
        job_config = load_job_config,
        location = configuration["bigquery_location"]
    )

    load_job.result()

    print(f"Uploaded {table_name} : {len(dataframe_to_upload):,} rows")

Uploaded electronics_nlp_bucket_dictionary : 10 rows
Uploaded electronics_nlp_scope_quality : 2 rows
Uploaded electronics_nlp_validation_quality : 10 rows
Uploaded electronics_nlp_false_negative_quality : 2 rows
Uploaded electronics_nlp_bucket_overview : 10 rows
Uploaded electronics_nlp_price_bucket_exposure : 50 rows
Uploaded electronics_nlp_priority_bucket_exposure : 20 rows
Uploaded electronics_nlp_bucket_cooccurrence : 90 rows
Uploaded electronics_nlp_multibucket_intensity : 12 rows
Uploaded electronics_nlp_top_products_by_bucket : 100 rows
